In [4]:
import numpy as np 

### TODO List



### Tableau implementation Class

In [5]:
# class that implements tableau and necessary operations
# Reference -Improved Simulation of Stabilizer Circuits by Scott Aaronson & Daniel Gottesman
class Cirq_Tableau:
    
    def __init__(
        self,
        pauli_word: list[str] = None  
    ):
        if pauli_word is None or len(pauli_word) == 0:
            self._column_num = None
            self._row_num = None

            self._ss, self._xs, self._zs = None, None, None
        else:
            self._column_num = len(pauli_word[0]) 
            self._row_num = len(pauli_word)

            self._ss = self.create_sign(pauli_word)
            self._xs, self._zs = self.create_tab(pauli_word)

    # setters & getters 
    @property
    def ss(self) -> np.ndarray:
        return self._ss

    @ss.setter 
    def ss(self, new_ss: np.ndarray):
        self._ss = new_ss
        
    @property
    def xs(self) -> np.ndarray:
        return self._xs

    @xs.setter 
    def xs(self, new_xs: np.ndarray):
        self._xs = new_xs
        
    @property
    def zs(self) -> np.ndarray:
        return self._zs

    @zs.setter 
    def zs(self, new_zs: np.ndarray):
        self._zs = new_zs
        
    @property
    def column_num(self) -> int:
        return self._column_num

    @column_num.setter 
    def column_num(self, new_num: int):
        self._column_num = new_num
    
    @property
    def row_num(self) -> int:
        return self._row_num

    @row_num.setter 
    def row_num(self, new_num: int):
        self._row_num = new_num

    # functions that create parts of tableau
    def create_sign(self, pauli_word: list[str]):
        temp_ss = np.zeros((self.row_num), dtype=int)
        for pauli_string in pauli_word:
            if "-" in pauli_string:
                index = pauli_word.index(pauli_string)
                pauli_word[index] = pauli_string.replace("-", "")
                temp_ss[index] = 1
        return temp_ss
        
    def create_tab(self, pauli_word: list[str]):
        temp_x = np.zeros((self.row_num, self.column_num), dtype=int)
        temp_z = np.zeros((self.row_num, self.column_num), dtype=int)
        for i, pauli_string in enumerate(pauli_word):
            for j, pauli in enumerate(pauli_string):
                if pauli == "X" or pauli == "Y":
                    temp_x[i][j] = 1
                if pauli == "Z" or pauli == "Y":
                    temp_z[i][j] = 1
        return temp_x, temp_z

    # Clifford operations on tableau. 
    def apply_H(self,column: int):
        self.ss ^= self.xs[:, column] & self.zs[:, column]
        self.xs[:, column], self.zs[:, column] = self.zs[:, column].copy(), self.xs[:, column].copy()

    def apply_S(self, column: int):
        self.ss ^= self.xs[:, column] & self.zs[:, column]
        self.zs[:, column] = self.xs[:, column] ^ self.zs[:, column]

    def apply_CX(self, control: int, target: int):
        self.ss ^= (
            (self.xs[:, control] & self.zs[:, target])
            &(~(self.xs[:, target] ^ self.zs[:, control]))
        )
        self.xs[:, target] ^= self.xs[:, control]
        self.zs[:, control] ^= self.zs[:, target]

    # class operations necessary for comparisons and equating
    def copy(self):
        new_tab = Cirq_Tableau()
        new_tab.column_num = self.column_num
        new_tab.row_num = self.row_num
        new_tab.ss = self.ss.copy()
        new_tab.zs = self.zs.copy()
        new_tab.xs = self.xs.copy()
        return new_tab
    
    def __eq__(self, other):
        if not isinstance(other, type(self)):
            return NotImplemented  
        return (
            self.column_num == other.column_num
            and self.row_num == other.row_num
            and np.array_equal(self.ss, other.ss)
            and np.array_equal(self.xs, other.xs)
            and np.array_equal(self.zs, other.zs)
        )
    
    def __copy__(self):
        return self.copy()
        

    def __str__(self):
        ss = np.expand_dims(self.ss, axis = 1)
        xz = np.concatenate((self.xs, self.zs, ss), axis=1)
        return str(xz)


### Functions

In [6]:
def commute(p_string_1: str, p_string_2: str):
    '''
    Function to test if two Pauli strings commute  - shamelessly copied from Ji's QuCLEAR code 

    :param p_string_1: First Pauli string
    :param p_string_2: Second Pauli string
    '''

    do_commute = False # assume they do not commute until proven otherwise

    anticommute_count = 0 # count of anticommuting Paulis
    
    for i in range(len(p_string_1)):
        
        if p_string_1[i] != p_string_2[i] and p_string_1[i] != "I" and p_string_2[i] != "I":
            anticommute_count += 1

    if anticommute_count % 2 == 0:
        do_commute = True

    return do_commute

In [7]:
def convert_commute_set(pauli_word: list[str]) -> list[list[str]]:
    '''
    Function to convert Pauli word into smaller Pauli word in which all Pauli strings commute - shamelessly copied from Ji's QuCLEAR code

    :param pauli_word: A list of strings to be converted into a list of lists of strings
    '''

    current_set = []
    commute_sets = []

    for pauli in pauli_word:

        # all 'I' Pauli string which will be removed in prune function 
        if all( p == "I" for p in pauli):
            continue 

        if not current_set:
            current_set.append(pauli) # current set is empty 

        else:

            add = True

            # check if each pauli string in current set commutes with new pauli string
            for elmt in current_set:
                if not commute(elmt, pauli):
                    add = False
                    break
            if add:
                current_set.append(pauli)
            else:
                commute_sets.append(current_set)
                current_set = [pauli]

    return commute_sets

In [8]:
def op_count(path):
    '''
    Function to count the number of CNOTs and single qubit operations present in whole set

    :param path: List of lists of action tuples that describes all operations to implement circuit
    '''

    # dictionary that keeps track of counts
    count_dict = dict({
        "CNOT": 0,
        "Single_q": 0
    })

    # loop over all operations 
    for op in path:

        for elmt in op:

            # match case for different operation types
            match elmt[0]:
                case "CX":
                    count_dict["CNOT"] += 1
                case "Y" |"-Y":
                    count_dict["Single_q"] += 5 #  -Z = HSYS*H | Z = HS-YS*H
                case "X" | "-X":
                    count_dict["Single_q"] += 3# Z = HXH | -Z = H-XH
                case "S" | "H" | "Z" | "-Z":
                    count_dict["Single_q"] += 1

    return count_dict

In [57]:
def edges(num_qubits: int):
    '''
    Function to create list of action tuples for each state 
    action tuples -> (operation: str, qubits acted on: int/s, weight of action: int)
    
    :param num_qubits: Integer that represents number of qubits 
    '''
    container = [] # list to hold all elements

    # TODO: make creation of possible actions less crude than listing out every combination - not good but works for now
    # loop to create action tuples for every possible combination
    for i in range(num_qubits):
        if i != (num_qubits-1):
            for j in range((i+1),num_qubits):
                container.append([("CX", i, j, 2)])
                container.append([("S", i, 1),("CX", i, j, 2)])
                container.append([("H", i, 1),("CX", i, j, 2)])
                container.append([("S", j, 1),("CX", i, j, 2)])
                container.append([("H", j, 1),("CX", i, j, 2)])
                container.append([("S", j, 1),("S", i, 1),("CX", i, j, 2)])
                container.append([("H", j, 1),("S", i, 1),("CX", i, j, 2)])
                container.append([("S", j, 1),("H", i, 1),("CX", i, j, 2)])
                container.append([("H", j, 1),("H", i, 1),("CX", i, j, 2)])
                container.append([("CX", j, i, 2)])
                container.append([("S", i, 1),("CX", j, i, 2)])
                container.append([("H", i, 1),("CX", j, i, 2)])
                container.append([("S", j, 1),("CX", j, i, 2)])
                container.append([("H", j, 1),("CX", j, i, 2)])
                container.append([("S", j, 1),("S", i, 1),("CX", j, i, 2)])
                container.append([("H", j, 1),("S", i, 1),("CX", j, i, 2)])
                container.append([("S", j, 1),("H", i, 1),("CX", j, i, 2)])
                container.append([("H", j, 1),("H", i, 1),("CX", j, i, 2)])
                
    return container

In [10]:
def heuristic(tab: Cirq_Tableau):
    '''
    Function to approximate "distance" to state with only single qubit Pauli strings

    :param tab: Cirq_Tableau used to calculate value
    '''

    # bitwise ORs x and z arrays in tableau to create array where nonidentity operation indices| have 1 in them
    compress = tab.xs | tab.zs

    # returns sum of the sum of each row less one
    return sum([int(sum(line))-1 for line in compress])

In [11]:
def g_func(path, current_action):
    '''
    Function to calculate "distance" travelled 

    :param path: list of action tuples that have been taken to get to current state
    '''
    lst = [] # list to hold all action tuples in each chunk in path

    # loop to extract all action tuples 
    for chunk in path:
        lst += chunk

    lst += current_action
    # returns sum over whole lst of weights of operations in action tuples
    return sum([x[-1] for x in lst]) if lst else 0

In [12]:
def prune(tableau: Cirq_Tableau, path):
    '''
    Function to remove rows in a tableau with only a single nonidentity operation
    
    :param tableau: Cirq_Tableau to act on
    :param path: list of action tuples to add to
    '''
    prn_ndxs = [] # list to hold all row indices to be pruned
    
    tab = tableau.copy() # copy of tableau to work on

    # extracts x, z and sign arrays
    x, z, s = tab.xs, tab.zs, tab.ss

    # bitwise ORs x and z arrays in tableau to create array where nonidentity operation indices have 1 in them
    weight_array = x | z

    # loop that checks through each row to find out if it has a Pauli weight of 1
    for r_ndx in range(len(weight_array)):
        if sum(weight_array[r_ndx])  == 1:

            # appends index to be pruned if Pauli weight == 1
            prn_ndxs.append(r_ndx)

            # checks what type of Pauli is on that index X, Y or Z and appends to path with action weight of zero 
            if sum(z[r_ndx]) == 0:
                if s[r_ndx] == 0:
                    path.append([("X", int(np.argmax(x[r_ndx] == 1)), 0)])
                else:
                    path.append([("-X", int(np.argmax(x[r_ndx] == 1)), 0)])
            elif sum(x[r_ndx]) == 0:
                if s[r_ndx] == 0:
                    path.append([("Z", int(np.argmax(z[r_ndx] == 1)), 0)])
                else:
                    path.append([("-Z", int(np.argmax(z[r_ndx] == 1)), 0)])
            else:
                if s[r_ndx] == 0:
                    path.append([("Y", int(np.argmax(x[r_ndx] == 1)), 0)])
                else:
                    path.append([("-Y", int(np.argmax(x[r_ndx] == 1)), 0)])
                    
        elif sum(weight_array[r_ndx])  == 0: # elif removes fully I lines 
            x, z, s = np.delete(x, r_ndx, axis=0), np.delete(z, r_ndx, axis=0), np.delete(s, r_ndx)
            
    # prunes rows 
    x, z, s = np.delete(x, prn_ndxs, axis=0), np.delete(z, prn_ndxs, axis=0), np.delete(s, prn_ndxs)
    column_num = len(x[0]) if x.size != 0 else 0 
    row_num = len(x)

    # updates tableau and returns it 
    tab.xs, tab.zs, tab.ss, tab.column_num, tab.row_num = x, z, s, column_num,row_num
    return tab
    

In [55]:
def search_prune(start: "Cirq_Tableau"):
    '''
    Function to find shortest number of 
    Note that this is holding the assumption that all the Pauli Strings Commute. Will need to implement commuting set function. 
    '''
    visited = [] # keeps track of all visited states to avoid looping back and forth between states 
    path_list = [] # list to hold path taken so far - used to calculate g
    start = prune(start, path_list) # prune any possible single qubit pauli strings & add them to path
    
    state = None # current state being explored
    action = None # current operation taken to get to current state
    
    while start.row_num != 0:

        f = float('inf') # initial value for heuristic 
        current_edges = edges(start.column_num) # edges from the current state ~ 18n^2 where n is the number of qubits

        # loop to go over all the edges (chunks of Clifford Operations) 
        for chunk in current_edges:
            tab = start.copy()
            for op in chunk:
                match op[0]:
                    case "CX":
                        tab.apply_CX(op[1], op[2])
                    case "H":
                        tab.apply_H(op[1])
                    case "S":
                        tab.apply_S(op[1])

            current_f = g_func(path_list, chunk) + heuristic(tab) #f value for current state 

            # TODO find more elegant way to check for reduction without introducing an extra ~k operations here
            tab_list = [sum(line) for line in (tab.xs | tab.zs)] # list to see if any lines exist with only one nonidentity Pauli

            # checks if current_f is less than f
            # if it is not it also checks if it is equal but reduces the rows
            # lastly it requires that tab is not in visited to avoid revisiting states 
            if (current_f < f or (current_f == f and 1 in tab_list)) and tab not in visited: 
                
                state = tab
                f = current_f
                action = chunk
                
        # append specifc chunk of action tuples to path and prune if necessary 
        path_list.append(action)
        visited.append(state)
        start = prune(state, path_list)

    # return full path
    return path_list
        

### Testing

*Initial Test on simple dumby pauli to see if process produces correct result*

In [69]:
# dumby Pauli Word to test how search prune works - created by Mulundano  
word = ["XXII", "IIZZ"]

In [70]:
tableau = Cirq_Tableau(word)
print(tableau)

[[1 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 1 0]]


In [71]:
path = search_prune(tableau)
path

[[('CX', 3, 2, 2)], [('Z', 2, 0)], [('CX', 1, 0, 2)], [('X', 1, 0)]]

In [72]:
op_count(path)

{'CNOT': 2, 'Single_q': 4}

*Testing first on small commuting blocks from Ji's paper*

*First tests are on ucc blocks*

In [46]:
ucc_2_4 = [
    [
        "YZXI",
        "XZXI",
        "YZYI",
        "XZYI"
    ],
    [
        "IYZX",
        "IXZX",
        "IYZY",
        "IXZY"
    ],
    [
        "YXYX",
        "XXYX",
        "YYYX",
        "XYYX",
        "YXYY",
        "XXYY",
        "YYYY",
        "XYYY",
        "YXXX",
        "XXXX",
        "YYXX",
        "XYXX",
        "YXXY",
        "XXXY",
        "YYXY",
        "XYXY"
    ]
]

In [53]:
path = []

for pauli_word in ucc_2_4:
    path += search_prune(Cirq_Tableau(pauli_word))
    
op_count(path) # need to run circuit through quantum optimizer for qiskit before I can say anything

{'CNOT': 18, 'Single_q': 103}

In [58]:
ucc_2_6 = [
    [
        "YZXIII",
        "XZXIII",
        "YZYIII",
        "XZYIII"
    ],
    [
        "YZZZXI",
        "XZZZXI",
        "YZZZYI",
        "XZZZYI"
    ],
    [
        "IYZXII",
        "IXZXII",
        "IYZYII",
        "IXZYII"
    ],
    [
        "IYZZZX",
        "IXZZZX",
        "IYZZZY",
        "IXZZZY"
    ],
    [
        "YXYXII",
        "XXYXII",
        "YYYXII",
        "XYYXII",
        "YXYYII",
        "XXYYII",
        "YYYYII",
        "XYYYII",
        "YXXXII",
        "XXXXII",
        "YYXXII",
        "XYXXII",
        "YXXYII",
        "XXXYII",
        "YYXYII",
        "XYXYII"
    ],
    [
        "YXYZZX",
        "XXYZZX",
        "YYYZZX",
        "XYYZZX",
        "YXYZZY",
        "XXYZZY",
        "YYYZZY",
        "XYYZZY",
        "YXXZZX",
        "XXXZZX",
        "YYXZZX",
        "XYXZZX",
        "YXXZZY",
        "XXXZZY",
        "YYXZZY",
        "XYXZZY"
    ],
    [
        "YXIYXI",
        "XXIYXI",
        "YYIYXI",
        "XYIYXI",
        "YXIYYI",
        "XXIYYI",
        "YYIYYI",
        "XYIYYI",
        "YXIXXI",
        "XXIXXI",
        "YYIXXI",
        "XYIXXI",
        "YXIXYI",
        "XXIXYI",
        "YYIXYI",
        "XYIXYI"
    ],
    [
        "YXIIYX",
        "XXIIYX",
        "YYIIYX",
        "XYIIYX",
        "YXIIYY",
        "XXIIYY",
        "YYIIYY",
        "XYIIYY",
        "YXIIXX",
        "XXIIXX",
        "YYIIXX",
        "XYIIXX",
        "YXIIXY",
        "XXIIXY",
        "YYIIXY",
        "XYIIXY"
    ]
]

In [59]:
path = []

for pauli_word in ucc_2_6:
    path += search_prune(Cirq_Tableau(pauli_word))
    
op_count(path)

{'CNOT': 66, 'Single_q': 341}

In [61]:
ucc_4_8 = [
    [
        "YZZZXIII",
        "XZZZXIII",
        "YZZZYIII",
        "XZZZYIII"
    ],
    [
        "YZZZZZXI",
        "XZZZZZXI",
        "YZZZZZYI",
        "XZZZZZYI"
    ],
    [
        "IYZZZXII",
        "IXZZZXII",
        "IYZZZYII",
        "IXZZZYII"
    ],
    [
        "IYZZZZZX",
        "IXZZZZZX",
        "IYZZZZZY",
        "IXZZZZZY"
    ],
    [
        "IIYZXIII",
        "IIXZXIII",
        "IIYZYIII",
        "IIXZYIII"
    ],
    [
        "IIYZZZXI",
        "IIXZZZXI",
        "IIYZZZYI",
        "IIXZZZYI"
    ],
    [
        "IIIYZXII",
        "IIIXZXII",
        "IIIYZYII",
        "IIIXZYII"
    ],
    [
        "IIIYZZZX",
        "IIIXZZZX",
        "IIIYZZZY",
        "IIIXZZZY"
    ],
    [
        "YXIIYXII",
        "XXIIYXII",
        "YYIIYXII",
        "XYIIYXII",
        "YXIIYYII",
        "XXIIYYII",
        "YYIIYYII",
        "XYIIYYII",
        "YXIIXXII",
        "XXIIXXII",
        "YYIIXXII",
        "XYIIXXII",
        "YXIIXYII",
        "XXIIXYII",
        "YYIIXYII",
        "XYIIXYII"
    ],
    [
        "YXIIYZZX",
        "XXIIYZZX",
        "YYIIYZZX",
        "XYIIYZZX",
        "YXIIYZZY",
        "XXIIYZZY",
        "YYIIYZZY",
        "XYIIYZZY",
        "YXIIXZZX",
        "XXIIXZZX",
        "YYIIXZZX",
        "XYIIXZZX",
        "YXIIXZZY",
        "XXIIXZZY",
        "YYIIXZZY",
        "XYIIXZZY"
    ],
    [
        "YXIIIYXI",
        "XXIIIYXI",
        "YYIIIYXI",
        "XYIIIYXI",
        "YXIIIYYI",
        "XXIIIYYI",
        "YYIIIYYI",
        "XYIIIYYI",
        "YXIIIXXI",
        "XXIIIXXI",
        "YYIIIXXI",
        "XYIIIXXI",
        "YXIIIXYI",
        "XXIIIXYI",
        "YYIIIXYI",
        "XYIIIXYI"
    ],
    [
        "YXIIIIYX",
        "XXIIIIYX",
        "YYIIIIYX",
        "XYIIIIYX",
        "YXIIIIYY",
        "XXIIIIYY",
        "YYIIIIYY",
        "XYIIIIYY",
        "YXIIIIXX",
        "XXIIIIXX",
        "YYIIIIXX",
        "XYIIIIXX",
        "YXIIIIXY",
        "XXIIIIXY",
        "YYIIIIXY",
        "XYIIIIXY"
    ],
    [
        "YZXIYZXI",
        "XZXIYZXI",
        "YZYIYZXI",
        "XZYIYZXI",
        "YZXIYZYI",
        "XZXIYZYI",
        "YZYIYZYI",
        "XZYIYZYI",
        "YZXIXZXI",
        "XZXIXZXI",
        "YZYIXZXI",
        "XZYIXZXI",
        "YZXIXZYI",
        "XZXIXZYI",
        "YZYIXZYI",
        "XZYIXZYI"
    ],
    [
        "YZZXYXII",
        "XZZXYXII",
        "YZZYYXII",
        "XZZYYXII",
        "YZZXYYII",
        "XZZXYYII",
        "YZZYYYII",
        "XZZYYYII",
        "YZZXXXII",
        "XZZXXXII",
        "YZZYXXII",
        "XZZYXXII",
        "YZZXXYII",
        "XZZXXYII",
        "YZZYXYII",
        "XZZYXYII"
    ],
    [
        "YZZXYZZX",
        "XZZXYZZX",
        "YZZYYZZX",
        "XZZYYZZX",
        "YZZXYZZY",
        "XZZXYZZY",
        "YZZYYZZY",
        "XZZYYZZY",
        "YZZXXZZX",
        "XZZXXZZX",
        "YZZYXZZX",
        "XZZYXZZX",
        "YZZXXZZY",
        "XZZXXZZY",
        "YZZYXZZY",
        "XZZYXZZY"
    ],
    [
        "YZZXIYXI",
        "XZZXIYXI",
        "YZZYIYXI",
        "XZZYIYXI",
        "YZZXIYYI",
        "XZZXIYYI",
        "YZZYIYYI",
        "XZZYIYYI",
        "YZZXIXXI",
        "XZZXIXXI",
        "YZZYIXXI",
        "XZZYIXXI",
        "YZZXIXYI",
        "XZZXIXYI",
        "YZZYIXYI",
        "XZZYIXYI"
    ],
    [
        "YZZXIIYX",
        "XZZXIIYX",
        "YZZYIIYX",
        "XZZYIIYX",
        "YZZXIIYY",
        "XZZXIIYY",
        "YZZYIIYY",
        "XZZYIIYY",
        "YZZXIIXX",
        "XZZXIIXX",
        "YZZYIIXX",
        "XZZYIIXX",
        "YZZXIIXY",
        "XZZXIIXY",
        "YZZYIIXY",
        "XZZYIIXY"
    ],
    [
        "IYXIYXII",
        "IXXIYXII",
        "IYYIYXII",
        "IXYIYXII",
        "IYXIYYII",
        "IXXIYYII",
        "IYYIYYII",
        "IXYIYYII",
        "IYXIXXII",
        "IXXIXXII",
        "IYYIXXII",
        "IXYIXXII",
        "IYXIXYII",
        "IXXIXYII",
        "IYYIXYII",
        "IXYIXYII"
    ],
    [
        "IYXIYZZX",
        "IXXIYZZX",
        "IYYIYZZX",
        "IXYIYZZX",
        "IYXIYZZY",
        "IXXIYZZY",
        "IYYIYZZY",
        "IXYIYZZY",
        "IYXIXZZX",
        "IXXIXZZX",
        "IYYIXZZX",
        "IXYIXZZX",
        "IYXIXZZY",
        "IXXIXZZY",
        "IYYIXZZY",
        "IXYIXZZY"
    ],
    [
        "IYXIIYXI",
        "IXXIIYXI",
        "IYYIIYXI",
        "IXYIIYXI",
        "IYXIIYYI",
        "IXXIIYYI",
        "IYYIIYYI",
        "IXYIIYYI",
        "IYXIIXXI",
        "IXXIIXXI",
        "IYYIIXXI",
        "IXYIIXXI",
        "IYXIIXYI",
        "IXXIIXYI",
        "IYYIIXYI",
        "IXYIIXYI"
    ],
    [
        "IYXIIIYX",
        "IXXIIIYX",
        "IYYIIIYX",
        "IXYIIIYX",
        "IYXIIIYY",
        "IXXIIIYY",
        "IYYIIIYY",
        "IXYIIIYY",
        "IYXIIIXX",
        "IXXIIIXX",
        "IYYIIIXX",
        "IXYIIIXX",
        "IYXIIIXY",
        "IXXIIIXY",
        "IYYIIIXY",
        "IXYIIIXY"
    ],
    [
        "IYZXIYZX",
        "IXZXIYZX",
        "IYZYIYZX",
        "IXZYIYZX",
        "IYZXIYZY",
        "IXZXIYZY",
        "IYZYIYZY",
        "IXZYIYZY",
        "IYZXIXZX",
        "IXZXIXZX",
        "IYZYIXZX",
        "IXZYIXZX",
        "IYZXIXZY",
        "IXZXIXZY",
        "IYZYIXZY",
        "IXZYIXZY"
    ],
    [
        "IIYXYXII",
        "IIXXYXII",
        "IIYYYXII",
        "IIXYYXII",
        "IIYXYYII",
        "IIXXYYII",
        "IIYYYYII",
        "IIXYYYII",
        "IIYXXXII",
        "IIXXXXII",
        "IIYYXXII",
        "IIXYXXII",
        "IIYXXYII",
        "IIXXXYII",
        "IIYYXYII",
        "IIXYXYII"
    ],
    [
        "IIYXYZZX",
        "IIXXYZZX",
        "IIYYYZZX",
        "IIXYYZZX",
        "IIYXYZZY",
        "IIXXYZZY",
        "IIYYYZZY",
        "IIXYYZZY",
        "IIYXXZZX",
        "IIXXXZZX",
        "IIYYXZZX",
        "IIXYXZZX",
        "IIYXXZZY",
        "IIXXXZZY",
        "IIYYXZZY",
        "IIXYXZZY"
    ],
    [
        "IIYXIYXI",
        "IIXXIYXI",
        "IIYYIYXI",
        "IIXYIYXI",
        "IIYXIYYI",
        "IIXXIYYI",
        "IIYYIYYI",
        "IIXYIYYI",
        "IIYXIXXI",
        "IIXXIXXI",
        "IIYYIXXI",
        "IIXYIXXI",
        "IIYXIXYI",
        "IIXXIXYI",
        "IIYYIXYI",
        "IIXYIXYI"
    ],
    [
        "IIYXIIYX",
        "IIXXIIYX",
        "IIYYIIYX",
        "IIXYIIYX",
        "IIYXIIYY",
        "IIXXIIYY",
        "IIYYIIYY",
        "IIXYIIYY",
        "IIYXIIXX",
        "IIXXIIXX",
        "IIYYIIXX",
        "IIXYIIXX",
        "IIYXIIXY",
        "IIXXIIXY",
        "IIYYIIXY",
        "IIXYIIXY"
    ]
]

In [62]:
path = []

for pauli_word in ucc_4_8:
    path += search_prune(Cirq_Tableau(pauli_word))
    
op_count(path)

{'CNOT': 276, 'Single_q': 1359}

In [64]:
ucc_6_12 = [
    [
        "YZZZZZXIIIII",
        "XZZZZZXIIIII",
        "YZZZZZYIIIII",
        "XZZZZZYIIIII"
    ],
    [
        "YZZZZZZZXIII",
        "XZZZZZZZXIII",
        "YZZZZZZZYIII",
        "XZZZZZZZYIII"
    ],
    [
        "YZZZZZZZZZXI",
        "XZZZZZZZZZXI",
        "YZZZZZZZZZYI",
        "XZZZZZZZZZYI"
    ],
    [
        "IYZZZZZXIIII",
        "IXZZZZZXIIII",
        "IYZZZZZYIIII",
        "IXZZZZZYIIII"
    ],
    [
        "IYZZZZZZZXII",
        "IXZZZZZZZXII",
        "IYZZZZZZZYII",
        "IXZZZZZZZYII"
    ],
    [
        "IYZZZZZZZZZX",
        "IXZZZZZZZZZX",
        "IYZZZZZZZZZY",
        "IXZZZZZZZZZY"
    ],
    [
        "IIYZZZXIIIII",
        "IIXZZZXIIIII",
        "IIYZZZYIIIII",
        "IIXZZZYIIIII"
    ],
    [
        "IIYZZZZZXIII",
        "IIXZZZZZXIII",
        "IIYZZZZZYIII",
        "IIXZZZZZYIII"
    ],
    [
        "IIYZZZZZZZXI",
        "IIXZZZZZZZXI",
        "IIYZZZZZZZYI",
        "IIXZZZZZZZYI"
    ],
    [
        "IIIYZZZXIIII",
        "IIIXZZZXIIII",
        "IIIYZZZYIIII",
        "IIIXZZZYIIII"
    ],
    [
        "IIIYZZZZZXII",
        "IIIXZZZZZXII",
        "IIIYZZZZZYII",
        "IIIXZZZZZYII"
    ],
    [
        "IIIYZZZZZZZX",
        "IIIXZZZZZZZX",
        "IIIYZZZZZZZY",
        "IIIXZZZZZZZY"
    ],
    [
        "IIIIYZXIIIII",
        "IIIIXZXIIIII",
        "IIIIYZYIIIII",
        "IIIIXZYIIIII"
    ],
    [
        "IIIIYZZZXIII",
        "IIIIXZZZXIII",
        "IIIIYZZZYIII",
        "IIIIXZZZYIII"
    ],
    [
        "IIIIYZZZZZXI",
        "IIIIXZZZZZXI",
        "IIIIYZZZZZYI",
        "IIIIXZZZZZYI"
    ],
    [
        "IIIIIYZXIIII",
        "IIIIIXZXIIII",
        "IIIIIYZYIIII",
        "IIIIIXZYIIII"
    ],
    [
        "IIIIIYZZZXII",
        "IIIIIXZZZXII",
        "IIIIIYZZZYII",
        "IIIIIXZZZYII"
    ],
    [
        "IIIIIYZZZZZX",
        "IIIIIXZZZZZX",
        "IIIIIYZZZZZY",
        "IIIIIXZZZZZY"
    ],
    [
        "YXIIIIYXIIII",
        "XXIIIIYXIIII",
        "YYIIIIYXIIII",
        "XYIIIIYXIIII",
        "YXIIIIYYIIII",
        "XXIIIIYYIIII",
        "YYIIIIYYIIII",
        "XYIIIIYYIIII",
        "YXIIIIXXIIII",
        "XXIIIIXXIIII",
        "YYIIIIXXIIII",
        "XYIIIIXXIIII",
        "YXIIIIXYIIII",
        "XXIIIIXYIIII",
        "YYIIIIXYIIII",
        "XYIIIIXYIIII"
    ],
    [
        "YXIIIIYZZXII",
        "XXIIIIYZZXII",
        "YYIIIIYZZXII",
        "XYIIIIYZZXII",
        "YXIIIIYZZYII",
        "XXIIIIYZZYII",
        "YYIIIIYZZYII",
        "XYIIIIYZZYII",
        "YXIIIIXZZXII",
        "XXIIIIXZZXII",
        "YYIIIIXZZXII",
        "XYIIIIXZZXII",
        "YXIIIIXZZYII",
        "XXIIIIXZZYII",
        "YYIIIIXZZYII",
        "XYIIIIXZZYII"
    ],
    [
        "YXIIIIYZZZZX",
        "XXIIIIYZZZZX",
        "YYIIIIYZZZZX",
        "XYIIIIYZZZZX",
        "YXIIIIYZZZZY",
        "XXIIIIYZZZZY",
        "YYIIIIYZZZZY",
        "XYIIIIYZZZZY",
        "YXIIIIXZZZZX",
        "XXIIIIXZZZZX",
        "YYIIIIXZZZZX",
        "XYIIIIXZZZZX",
        "YXIIIIXZZZZY",
        "XXIIIIXZZZZY",
        "YYIIIIXZZZZY",
        "XYIIIIXZZZZY"
    ],
    [
        "YXIIIIIYXIII",
        "XXIIIIIYXIII",
        "YYIIIIIYXIII",
        "XYIIIIIYXIII",
        "YXIIIIIYYIII",
        "XXIIIIIYYIII",
        "YYIIIIIYYIII",
        "XYIIIIIYYIII",
        "YXIIIIIXXIII",
        "XXIIIIIXXIII",
        "YYIIIIIXXIII",
        "XYIIIIIXXIII",
        "YXIIIIIXYIII",
        "XXIIIIIXYIII",
        "YYIIIIIXYIII",
        "XYIIIIIXYIII"
    ],
    [
        "YXIIIIIYZZXI",
        "XXIIIIIYZZXI",
        "YYIIIIIYZZXI",
        "XYIIIIIYZZXI",
        "YXIIIIIYZZYI",
        "XXIIIIIYZZYI",
        "YYIIIIIYZZYI",
        "XYIIIIIYZZYI",
        "YXIIIIIXZZXI",
        "XXIIIIIXZZXI",
        "YYIIIIIXZZXI",
        "XYIIIIIXZZXI",
        "YXIIIIIXZZYI",
        "XXIIIIIXZZYI",
        "YYIIIIIXZZYI",
        "XYIIIIIXZZYI"
    ],
    [
        "YXIIIIIIYXII",
        "XXIIIIIIYXII",
        "YYIIIIIIYXII",
        "XYIIIIIIYXII",
        "YXIIIIIIYYII",
        "XXIIIIIIYYII",
        "YYIIIIIIYYII",
        "XYIIIIIIYYII",
        "YXIIIIIIXXII",
        "XXIIIIIIXXII",
        "YYIIIIIIXXII",
        "XYIIIIIIXXII",
        "YXIIIIIIXYII",
        "XXIIIIIIXYII",
        "YYIIIIIIXYII",
        "XYIIIIIIXYII"
    ],
    [
        "YXIIIIIIYZZX",
        "XXIIIIIIYZZX",
        "YYIIIIIIYZZX",
        "XYIIIIIIYZZX",
        "YXIIIIIIYZZY",
        "XXIIIIIIYZZY",
        "YYIIIIIIYZZY",
        "XYIIIIIIYZZY",
        "YXIIIIIIXZZX",
        "XXIIIIIIXZZX",
        "YYIIIIIIXZZX",
        "XYIIIIIIXZZX",
        "YXIIIIIIXZZY",
        "XXIIIIIIXZZY",
        "YYIIIIIIXZZY",
        "XYIIIIIIXZZY"
    ],
    [
        "YXIIIIIIIYXI",
        "XXIIIIIIIYXI",
        "YYIIIIIIIYXI",
        "XYIIIIIIIYXI",
        "YXIIIIIIIYYI",
        "XXIIIIIIIYYI",
        "YYIIIIIIIYYI",
        "XYIIIIIIIYYI",
        "YXIIIIIIIXXI",
        "XXIIIIIIIXXI",
        "YYIIIIIIIXXI",
        "XYIIIIIIIXXI",
        "YXIIIIIIIXYI",
        "XXIIIIIIIXYI",
        "YYIIIIIIIXYI",
        "XYIIIIIIIXYI"
    ],
    [
        "YXIIIIIIIIYX",
        "XXIIIIIIIIYX",
        "YYIIIIIIIIYX",
        "XYIIIIIIIIYX",
        "YXIIIIIIIIYY",
        "XXIIIIIIIIYY",
        "YYIIIIIIIIYY",
        "XYIIIIIIIIYY",
        "YXIIIIIIIIXX",
        "XXIIIIIIIIXX",
        "YYIIIIIIIIXX",
        "XYIIIIIIIIXX",
        "YXIIIIIIIIXY",
        "XXIIIIIIIIXY",
        "YYIIIIIIIIXY",
        "XYIIIIIIIIXY"
    ],
    [
        "YZXIIIYZXIII",
        "XZXIIIYZXIII",
        "YZYIIIYZXIII",
        "XZYIIIYZXIII",
        "YZXIIIYZYIII",
        "XZXIIIYZYIII",
        "YZYIIIYZYIII",
        "XZYIIIYZYIII",
        "YZXIIIXZXIII",
        "XZXIIIXZXIII",
        "YZYIIIXZXIII",
        "XZYIIIXZXIII",
        "YZXIIIXZYIII",
        "XZXIIIXZYIII",
        "YZYIIIXZYIII",
        "XZYIIIXZYIII"
    ],
    [
        "YZXIIIYZZZXI",
        "XZXIIIYZZZXI",
        "YZYIIIYZZZXI",
        "XZYIIIYZZZXI",
        "YZXIIIYZZZYI",
        "XZXIIIYZZZYI",
        "YZYIIIYZZZYI",
        "XZYIIIYZZZYI",
        "YZXIIIXZZZXI",
        "XZXIIIXZZZXI",
        "YZYIIIXZZZXI",
        "XZYIIIXZZZXI",
        "YZXIIIXZZZYI",
        "XZXIIIXZZZYI",
        "YZYIIIXZZZYI",
        "XZYIIIXZZZYI"
    ],
    [
        "YZXIIIIIYZXI",
        "XZXIIIIIYZXI",
        "YZYIIIIIYZXI",
        "XZYIIIIIYZXI",
        "YZXIIIIIYZYI",
        "XZXIIIIIYZYI",
        "YZYIIIIIYZYI",
        "XZYIIIIIYZYI",
        "YZXIIIIIXZXI",
        "XZXIIIIIXZXI",
        "YZYIIIIIXZXI",
        "XZYIIIIIXZXI",
        "YZXIIIIIXZYI",
        "XZXIIIIIXZYI",
        "YZYIIIIIXZYI",
        "XZYIIIIIXZYI"
    ],
    [
        "YZZXIIYXIIII",
        "XZZXIIYXIIII",
        "YZZYIIYXIIII",
        "XZZYIIYXIIII",
        "YZZXIIYYIIII",
        "XZZXIIYYIIII",
        "YZZYIIYYIIII",
        "XZZYIIYYIIII",
        "YZZXIIXXIIII",
        "XZZXIIXXIIII",
        "YZZYIIXXIIII",
        "XZZYIIXXIIII",
        "YZZXIIXYIIII",
        "XZZXIIXYIIII",
        "YZZYIIXYIIII",
        "XZZYIIXYIIII"
    ],
    [
        "YZZXIIYZZXII",
        "XZZXIIYZZXII",
        "YZZYIIYZZXII",
        "XZZYIIYZZXII",
        "YZZXIIYZZYII",
        "XZZXIIYZZYII",
        "YZZYIIYZZYII",
        "XZZYIIYZZYII",
        "YZZXIIXZZXII",
        "XZZXIIXZZXII",
        "YZZYIIXZZXII",
        "XZZYIIXZZXII",
        "YZZXIIXZZYII",
        "XZZXIIXZZYII",
        "YZZYIIXZZYII",
        "XZZYIIXZZYII"
    ],
    [
        "YZZXIIYZZZZX",
        "XZZXIIYZZZZX",
        "YZZYIIYZZZZX",
        "XZZYIIYZZZZX",
        "YZZXIIYZZZZY",
        "XZZXIIYZZZZY",
        "YZZYIIYZZZZY",
        "XZZYIIYZZZZY",
        "YZZXIIXZZZZX",
        "XZZXIIXZZZZX",
        "YZZYIIXZZZZX",
        "XZZYIIXZZZZX",
        "YZZXIIXZZZZY",
        "XZZXIIXZZZZY",
        "YZZYIIXZZZZY",
        "XZZYIIXZZZZY"
    ],
    [
        "YZZXIIIYXIII",
        "XZZXIIIYXIII",
        "YZZYIIIYXIII",
        "XZZYIIIYXIII",
        "YZZXIIIYYIII",
        "XZZXIIIYYIII",
        "YZZYIIIYYIII",
        "XZZYIIIYYIII",
        "YZZXIIIXXIII",
        "XZZXIIIXXIII",
        "YZZYIIIXXIII",
        "XZZYIIIXXIII",
        "YZZXIIIXYIII",
        "XZZXIIIXYIII",
        "YZZYIIIXYIII",
        "XZZYIIIXYIII"
    ],
    [
        "YZZXIIIYZZXI",
        "XZZXIIIYZZXI",
        "YZZYIIIYZZXI",
        "XZZYIIIYZZXI",
        "YZZXIIIYZZYI",
        "XZZXIIIYZZYI",
        "YZZYIIIYZZYI",
        "XZZYIIIYZZYI",
        "YZZXIIIXZZXI",
        "XZZXIIIXZZXI",
        "YZZYIIIXZZXI",
        "XZZYIIIXZZXI",
        "YZZXIIIXZZYI",
        "XZZXIIIXZZYI",
        "YZZYIIIXZZYI",
        "XZZYIIIXZZYI"
    ],
    [
        "YZZXIIIIYXII",
        "XZZXIIIIYXII",
        "YZZYIIIIYXII",
        "XZZYIIIIYXII",
        "YZZXIIIIYYII",
        "XZZXIIIIYYII",
        "YZZYIIIIYYII",
        "XZZYIIIIYYII",
        "YZZXIIIIXXII",
        "XZZXIIIIXXII",
        "YZZYIIIIXXII",
        "XZZYIIIIXXII",
        "YZZXIIIIXYII",
        "XZZXIIIIXYII",
        "YZZYIIIIXYII",
        "XZZYIIIIXYII"
    ],
    [
        "YZZXIIIIYZZX",
        "XZZXIIIIYZZX",
        "YZZYIIIIYZZX",
        "XZZYIIIIYZZX",
        "YZZXIIIIYZZY",
        "XZZXIIIIYZZY",
        "YZZYIIIIYZZY",
        "XZZYIIIIYZZY",
        "YZZXIIIIXZZX",
        "XZZXIIIIXZZX",
        "YZZYIIIIXZZX",
        "XZZYIIIIXZZX",
        "YZZXIIIIXZZY",
        "XZZXIIIIXZZY",
        "YZZYIIIIXZZY",
        "XZZYIIIIXZZY"
    ],
    [
        "YZZXIIIIIYXI",
        "XZZXIIIIIYXI",
        "YZZYIIIIIYXI",
        "XZZYIIIIIYXI",
        "YZZXIIIIIYYI",
        "XZZXIIIIIYYI",
        "YZZYIIIIIYYI",
        "XZZYIIIIIYYI",
        "YZZXIIIIIXXI",
        "XZZXIIIIIXXI",
        "YZZYIIIIIXXI",
        "XZZYIIIIIXXI",
        "YZZXIIIIIXYI",
        "XZZXIIIIIXYI",
        "YZZYIIIIIXYI",
        "XZZYIIIIIXYI"
    ],
    [
        "YZZXIIIIIIYX",
        "XZZXIIIIIIYX",
        "YZZYIIIIIIYX",
        "XZZYIIIIIIYX",
        "YZZXIIIIIIYY",
        "XZZXIIIIIIYY",
        "YZZYIIIIIIYY",
        "XZZYIIIIIIYY",
        "YZZXIIIIIIXX",
        "XZZXIIIIIIXX",
        "YZZYIIIIIIXX",
        "XZZYIIIIIIXX",
        "YZZXIIIIIIXY",
        "XZZXIIIIIIXY",
        "YZZYIIIIIIXY",
        "XZZYIIIIIIXY"
    ],
    [
        "YZZZXIYZXIII",
        "XZZZXIYZXIII",
        "YZZZYIYZXIII",
        "XZZZYIYZXIII",
        "YZZZXIYZYIII",
        "XZZZXIYZYIII",
        "YZZZYIYZYIII",
        "XZZZYIYZYIII",
        "YZZZXIXZXIII",
        "XZZZXIXZXIII",
        "YZZZYIXZXIII",
        "XZZZYIXZXIII",
        "YZZZXIXZYIII",
        "XZZZXIXZYIII",
        "YZZZYIXZYIII",
        "XZZZYIXZYIII"
    ],
    [
        "YZZZXIYZZZXI",
        "XZZZXIYZZZXI",
        "YZZZYIYZZZXI",
        "XZZZYIYZZZXI",
        "YZZZXIYZZZYI",
        "XZZZXIYZZZYI",
        "YZZZYIYZZZYI",
        "XZZZYIYZZZYI",
        "YZZZXIXZZZXI",
        "XZZZXIXZZZXI",
        "YZZZYIXZZZXI",
        "XZZZYIXZZZXI",
        "YZZZXIXZZZYI",
        "XZZZXIXZZZYI",
        "YZZZYIXZZZYI",
        "XZZZYIXZZZYI"
    ],
    [
        "YZZZXIIIYZXI",
        "XZZZXIIIYZXI",
        "YZZZYIIIYZXI",
        "XZZZYIIIYZXI",
        "YZZZXIIIYZYI",
        "XZZZXIIIYZYI",
        "YZZZYIIIYZYI",
        "XZZZYIIIYZYI",
        "YZZZXIIIXZXI",
        "XZZZXIIIXZXI",
        "YZZZYIIIXZXI",
        "XZZZYIIIXZXI",
        "YZZZXIIIXZYI",
        "XZZZXIIIXZYI",
        "YZZZYIIIXZYI",
        "XZZZYIIIXZYI"
    ],
    [
        "YZZZZXYXIIII",
        "XZZZZXYXIIII",
        "YZZZZYYXIIII",
        "XZZZZYYXIIII",
        "YZZZZXYYIIII",
        "XZZZZXYYIIII",
        "YZZZZYYYIIII",
        "XZZZZYYYIIII",
        "YZZZZXXXIIII",
        "XZZZZXXXIIII",
        "YZZZZYXXIIII",
        "XZZZZYXXIIII",
        "YZZZZXXYIIII",
        "XZZZZXXYIIII",
        "YZZZZYXYIIII",
        "XZZZZYXYIIII"
    ],
    [
        "YZZZZXYZZXII",
        "XZZZZXYZZXII",
        "YZZZZYYZZXII",
        "XZZZZYYZZXII",
        "YZZZZXYZZYII",
        "XZZZZXYZZYII",
        "YZZZZYYZZYII",
        "XZZZZYYZZYII",
        "YZZZZXXZZXII",
        "XZZZZXXZZXII",
        "YZZZZYXZZXII",
        "XZZZZYXZZXII",
        "YZZZZXXZZYII",
        "XZZZZXXZZYII",
        "YZZZZYXZZYII",
        "XZZZZYXZZYII"
    ],
    [
        "YZZZZXYZZZZX",
        "XZZZZXYZZZZX",
        "YZZZZYYZZZZX",
        "XZZZZYYZZZZX",
        "YZZZZXYZZZZY",
        "XZZZZXYZZZZY",
        "YZZZZYYZZZZY",
        "XZZZZYYZZZZY",
        "YZZZZXXZZZZX",
        "XZZZZXXZZZZX",
        "YZZZZYXZZZZX",
        "XZZZZYXZZZZX",
        "YZZZZXXZZZZY",
        "XZZZZXXZZZZY",
        "YZZZZYXZZZZY",
        "XZZZZYXZZZZY"
    ],
    [
        "YZZZZXIYXIII",
        "XZZZZXIYXIII",
        "YZZZZYIYXIII",
        "XZZZZYIYXIII",
        "YZZZZXIYYIII",
        "XZZZZXIYYIII",
        "YZZZZYIYYIII",
        "XZZZZYIYYIII",
        "YZZZZXIXXIII",
        "XZZZZXIXXIII",
        "YZZZZYIXXIII",
        "XZZZZYIXXIII",
        "YZZZZXIXYIII",
        "XZZZZXIXYIII",
        "YZZZZYIXYIII",
        "XZZZZYIXYIII"
    ],
    [
        "YZZZZXIYZZXI",
        "XZZZZXIYZZXI",
        "YZZZZYIYZZXI",
        "XZZZZYIYZZXI",
        "YZZZZXIYZZYI",
        "XZZZZXIYZZYI",
        "YZZZZYIYZZYI",
        "XZZZZYIYZZYI",
        "YZZZZXIXZZXI",
        "XZZZZXIXZZXI",
        "YZZZZYIXZZXI",
        "XZZZZYIXZZXI",
        "YZZZZXIXZZYI",
        "XZZZZXIXZZYI",
        "YZZZZYIXZZYI",
        "XZZZZYIXZZYI"
    ],
    [
        "YZZZZXIIYXII",
        "XZZZZXIIYXII",
        "YZZZZYIIYXII",
        "XZZZZYIIYXII",
        "YZZZZXIIYYII",
        "XZZZZXIIYYII",
        "YZZZZYIIYYII",
        "XZZZZYIIYYII",
        "YZZZZXIIXXII",
        "XZZZZXIIXXII",
        "YZZZZYIIXXII",
        "XZZZZYIIXXII",
        "YZZZZXIIXYII",
        "XZZZZXIIXYII",
        "YZZZZYIIXYII",
        "XZZZZYIIXYII"
    ],
    [
        "YZZZZXIIYZZX",
        "XZZZZXIIYZZX",
        "YZZZZYIIYZZX",
        "XZZZZYIIYZZX",
        "YZZZZXIIYZZY",
        "XZZZZXIIYZZY",
        "YZZZZYIIYZZY",
        "XZZZZYIIYZZY",
        "YZZZZXIIXZZX",
        "XZZZZXIIXZZX",
        "YZZZZYIIXZZX",
        "XZZZZYIIXZZX",
        "YZZZZXIIXZZY",
        "XZZZZXIIXZZY",
        "YZZZZYIIXZZY",
        "XZZZZYIIXZZY"
    ],
    [
        "YZZZZXIIIYXI",
        "XZZZZXIIIYXI",
        "YZZZZYIIIYXI",
        "XZZZZYIIIYXI",
        "YZZZZXIIIYYI",
        "XZZZZXIIIYYI",
        "YZZZZYIIIYYI",
        "XZZZZYIIIYYI",
        "YZZZZXIIIXXI",
        "XZZZZXIIIXXI",
        "YZZZZYIIIXXI",
        "XZZZZYIIIXXI",
        "YZZZZXIIIXYI",
        "XZZZZXIIIXYI",
        "YZZZZYIIIXYI",
        "XZZZZYIIIXYI"
    ],
    [
        "YZZZZXIIIIYX",
        "XZZZZXIIIIYX",
        "YZZZZYIIIIYX",
        "XZZZZYIIIIYX",
        "YZZZZXIIIIYY",
        "XZZZZXIIIIYY",
        "YZZZZYIIIIYY",
        "XZZZZYIIIIYY",
        "YZZZZXIIIIXX",
        "XZZZZXIIIIXX",
        "YZZZZYIIIIXX",
        "XZZZZYIIIIXX",
        "YZZZZXIIIIXY",
        "XZZZZXIIIIXY",
        "YZZZZYIIIIXY",
        "XZZZZYIIIIXY"
    ],
    [
        "IYXIIIYXIIII",
        "IXXIIIYXIIII",
        "IYYIIIYXIIII",
        "IXYIIIYXIIII",
        "IYXIIIYYIIII",
        "IXXIIIYYIIII",
        "IYYIIIYYIIII",
        "IXYIIIYYIIII",
        "IYXIIIXXIIII",
        "IXXIIIXXIIII",
        "IYYIIIXXIIII",
        "IXYIIIXXIIII",
        "IYXIIIXYIIII",
        "IXXIIIXYIIII",
        "IYYIIIXYIIII",
        "IXYIIIXYIIII"
    ],
    [
        "IYXIIIYZZXII",
        "IXXIIIYZZXII",
        "IYYIIIYZZXII",
        "IXYIIIYZZXII",
        "IYXIIIYZZYII",
        "IXXIIIYZZYII",
        "IYYIIIYZZYII",
        "IXYIIIYZZYII",
        "IYXIIIXZZXII",
        "IXXIIIXZZXII",
        "IYYIIIXZZXII",
        "IXYIIIXZZXII",
        "IYXIIIXZZYII",
        "IXXIIIXZZYII",
        "IYYIIIXZZYII",
        "IXYIIIXZZYII"
    ],
    [
        "IYXIIIYZZZZX",
        "IXXIIIYZZZZX",
        "IYYIIIYZZZZX",
        "IXYIIIYZZZZX",
        "IYXIIIYZZZZY",
        "IXXIIIYZZZZY",
        "IYYIIIYZZZZY",
        "IXYIIIYZZZZY",
        "IYXIIIXZZZZX",
        "IXXIIIXZZZZX",
        "IYYIIIXZZZZX",
        "IXYIIIXZZZZX",
        "IYXIIIXZZZZY",
        "IXXIIIXZZZZY",
        "IYYIIIXZZZZY",
        "IXYIIIXZZZZY"
    ],
    [
        "IYXIIIIYXIII",
        "IXXIIIIYXIII",
        "IYYIIIIYXIII",
        "IXYIIIIYXIII",
        "IYXIIIIYYIII",
        "IXXIIIIYYIII",
        "IYYIIIIYYIII",
        "IXYIIIIYYIII",
        "IYXIIIIXXIII",
        "IXXIIIIXXIII",
        "IYYIIIIXXIII",
        "IXYIIIIXXIII",
        "IYXIIIIXYIII",
        "IXXIIIIXYIII",
        "IYYIIIIXYIII",
        "IXYIIIIXYIII"
    ],
    [
        "IYXIIIIYZZXI",
        "IXXIIIIYZZXI",
        "IYYIIIIYZZXI",
        "IXYIIIIYZZXI",
        "IYXIIIIYZZYI",
        "IXXIIIIYZZYI",
        "IYYIIIIYZZYI",
        "IXYIIIIYZZYI",
        "IYXIIIIXZZXI",
        "IXXIIIIXZZXI",
        "IYYIIIIXZZXI",
        "IXYIIIIXZZXI",
        "IYXIIIIXZZYI",
        "IXXIIIIXZZYI",
        "IYYIIIIXZZYI",
        "IXYIIIIXZZYI"
    ],
    [
        "IYXIIIIIYXII",
        "IXXIIIIIYXII",
        "IYYIIIIIYXII",
        "IXYIIIIIYXII",
        "IYXIIIIIYYII",
        "IXXIIIIIYYII",
        "IYYIIIIIYYII",
        "IXYIIIIIYYII",
        "IYXIIIIIXXII",
        "IXXIIIIIXXII",
        "IYYIIIIIXXII",
        "IXYIIIIIXXII",
        "IYXIIIIIXYII",
        "IXXIIIIIXYII",
        "IYYIIIIIXYII",
        "IXYIIIIIXYII"
    ],
    [
        "IYXIIIIIYZZX",
        "IXXIIIIIYZZX",
        "IYYIIIIIYZZX",
        "IXYIIIIIYZZX",
        "IYXIIIIIYZZY",
        "IXXIIIIIYZZY",
        "IYYIIIIIYZZY",
        "IXYIIIIIYZZY",
        "IYXIIIIIXZZX",
        "IXXIIIIIXZZX",
        "IYYIIIIIXZZX",
        "IXYIIIIIXZZX",
        "IYXIIIIIXZZY",
        "IXXIIIIIXZZY",
        "IYYIIIIIXZZY",
        "IXYIIIIIXZZY"
    ],
    [
        "IYXIIIIIIYXI",
        "IXXIIIIIIYXI",
        "IYYIIIIIIYXI",
        "IXYIIIIIIYXI",
        "IYXIIIIIIYYI",
        "IXXIIIIIIYYI",
        "IYYIIIIIIYYI",
        "IXYIIIIIIYYI",
        "IYXIIIIIIXXI",
        "IXXIIIIIIXXI",
        "IYYIIIIIIXXI",
        "IXYIIIIIIXXI",
        "IYXIIIIIIXYI",
        "IXXIIIIIIXYI",
        "IYYIIIIIIXYI",
        "IXYIIIIIIXYI"
    ],
    [
        "IYXIIIIIIIYX",
        "IXXIIIIIIIYX",
        "IYYIIIIIIIYX",
        "IXYIIIIIIIYX",
        "IYXIIIIIIIYY",
        "IXXIIIIIIIYY",
        "IYYIIIIIIIYY",
        "IXYIIIIIIIYY",
        "IYXIIIIIIIXX",
        "IXXIIIIIIIXX",
        "IYYIIIIIIIXX",
        "IXYIIIIIIIXX",
        "IYXIIIIIIIXY",
        "IXXIIIIIIIXY",
        "IYYIIIIIIIXY",
        "IXYIIIIIIIXY"
    ],
    [
        "IYZXIIIYZXII",
        "IXZXIIIYZXII",
        "IYZYIIIYZXII",
        "IXZYIIIYZXII",
        "IYZXIIIYZYII",
        "IXZXIIIYZYII",
        "IYZYIIIYZYII",
        "IXZYIIIYZYII",
        "IYZXIIIXZXII",
        "IXZXIIIXZXII",
        "IYZYIIIXZXII",
        "IXZYIIIXZXII",
        "IYZXIIIXZYII",
        "IXZXIIIXZYII",
        "IYZYIIIXZYII",
        "IXZYIIIXZYII"
    ],
    [
        "IYZXIIIYZZZX",
        "IXZXIIIYZZZX",
        "IYZYIIIYZZZX",
        "IXZYIIIYZZZX",
        "IYZXIIIYZZZY",
        "IXZXIIIYZZZY",
        "IYZYIIIYZZZY",
        "IXZYIIIYZZZY",
        "IYZXIIIXZZZX",
        "IXZXIIIXZZZX",
        "IYZYIIIXZZZX",
        "IXZYIIIXZZZX",
        "IYZXIIIXZZZY",
        "IXZXIIIXZZZY",
        "IYZYIIIXZZZY",
        "IXZYIIIXZZZY"
    ],
    [
        "IYZXIIIIIYZX",
        "IXZXIIIIIYZX",
        "IYZYIIIIIYZX",
        "IXZYIIIIIYZX",
        "IYZXIIIIIYZY",
        "IXZXIIIIIYZY",
        "IYZYIIIIIYZY",
        "IXZYIIIIIYZY",
        "IYZXIIIIIXZX",
        "IXZXIIIIIXZX",
        "IYZYIIIIIXZX",
        "IXZYIIIIIXZX",
        "IYZXIIIIIXZY",
        "IXZXIIIIIXZY",
        "IYZYIIIIIXZY",
        "IXZYIIIIIXZY"
    ],
    [
        "IYZZXIYXIIII",
        "IXZZXIYXIIII",
        "IYZZYIYXIIII",
        "IXZZYIYXIIII",
        "IYZZXIYYIIII",
        "IXZZXIYYIIII",
        "IYZZYIYYIIII",
        "IXZZYIYYIIII",
        "IYZZXIXXIIII",
        "IXZZXIXXIIII",
        "IYZZYIXXIIII",
        "IXZZYIXXIIII",
        "IYZZXIXYIIII",
        "IXZZXIXYIIII",
        "IYZZYIXYIIII",
        "IXZZYIXYIIII"
    ],
    [
        "IYZZXIYZZXII",
        "IXZZXIYZZXII",
        "IYZZYIYZZXII",
        "IXZZYIYZZXII",
        "IYZZXIYZZYII",
        "IXZZXIYZZYII",
        "IYZZYIYZZYII",
        "IXZZYIYZZYII",
        "IYZZXIXZZXII",
        "IXZZXIXZZXII",
        "IYZZYIXZZXII",
        "IXZZYIXZZXII",
        "IYZZXIXZZYII",
        "IXZZXIXZZYII",
        "IYZZYIXZZYII",
        "IXZZYIXZZYII"
    ],
    [
        "IYZZXIYZZZZX",
        "IXZZXIYZZZZX",
        "IYZZYIYZZZZX",
        "IXZZYIYZZZZX",
        "IYZZXIYZZZZY",
        "IXZZXIYZZZZY",
        "IYZZYIYZZZZY",
        "IXZZYIYZZZZY",
        "IYZZXIXZZZZX",
        "IXZZXIXZZZZX",
        "IYZZYIXZZZZX",
        "IXZZYIXZZZZX",
        "IYZZXIXZZZZY",
        "IXZZXIXZZZZY",
        "IYZZYIXZZZZY",
        "IXZZYIXZZZZY"
    ],
    [
        "IYZZXIIYXIII",
        "IXZZXIIYXIII",
        "IYZZYIIYXIII",
        "IXZZYIIYXIII",
        "IYZZXIIYYIII",
        "IXZZXIIYYIII",
        "IYZZYIIYYIII",
        "IXZZYIIYYIII",
        "IYZZXIIXXIII",
        "IXZZXIIXXIII",
        "IYZZYIIXXIII",
        "IXZZYIIXXIII",
        "IYZZXIIXYIII",
        "IXZZXIIXYIII",
        "IYZZYIIXYIII",
        "IXZZYIIXYIII"
    ],
    [
        "IYZZXIIYZZXI",
        "IXZZXIIYZZXI",
        "IYZZYIIYZZXI",
        "IXZZYIIYZZXI",
        "IYZZXIIYZZYI",
        "IXZZXIIYZZYI",
        "IYZZYIIYZZYI",
        "IXZZYIIYZZYI",
        "IYZZXIIXZZXI",
        "IXZZXIIXZZXI",
        "IYZZYIIXZZXI",
        "IXZZYIIXZZXI",
        "IYZZXIIXZZYI",
        "IXZZXIIXZZYI",
        "IYZZYIIXZZYI",
        "IXZZYIIXZZYI"
    ],
    [
        "IYZZXIIIYXII",
        "IXZZXIIIYXII",
        "IYZZYIIIYXII",
        "IXZZYIIIYXII",
        "IYZZXIIIYYII",
        "IXZZXIIIYYII",
        "IYZZYIIIYYII",
        "IXZZYIIIYYII",
        "IYZZXIIIXXII",
        "IXZZXIIIXXII",
        "IYZZYIIIXXII",
        "IXZZYIIIXXII",
        "IYZZXIIIXYII",
        "IXZZXIIIXYII",
        "IYZZYIIIXYII",
        "IXZZYIIIXYII"
    ],
    [
        "IYZZXIIIYZZX",
        "IXZZXIIIYZZX",
        "IYZZYIIIYZZX",
        "IXZZYIIIYZZX",
        "IYZZXIIIYZZY",
        "IXZZXIIIYZZY",
        "IYZZYIIIYZZY",
        "IXZZYIIIYZZY",
        "IYZZXIIIXZZX",
        "IXZZXIIIXZZX",
        "IYZZYIIIXZZX",
        "IXZZYIIIXZZX",
        "IYZZXIIIXZZY",
        "IXZZXIIIXZZY",
        "IYZZYIIIXZZY",
        "IXZZYIIIXZZY"
    ],
    [
        "IYZZXIIIIYXI",
        "IXZZXIIIIYXI",
        "IYZZYIIIIYXI",
        "IXZZYIIIIYXI",
        "IYZZXIIIIYYI",
        "IXZZXIIIIYYI",
        "IYZZYIIIIYYI",
        "IXZZYIIIIYYI",
        "IYZZXIIIIXXI",
        "IXZZXIIIIXXI",
        "IYZZYIIIIXXI",
        "IXZZYIIIIXXI",
        "IYZZXIIIIXYI",
        "IXZZXIIIIXYI",
        "IYZZYIIIIXYI",
        "IXZZYIIIIXYI"
    ],
    [
        "IYZZXIIIIIYX",
        "IXZZXIIIIIYX",
        "IYZZYIIIIIYX",
        "IXZZYIIIIIYX",
        "IYZZXIIIIIYY",
        "IXZZXIIIIIYY",
        "IYZZYIIIIIYY",
        "IXZZYIIIIIYY",
        "IYZZXIIIIIXX",
        "IXZZXIIIIIXX",
        "IYZZYIIIIIXX",
        "IXZZYIIIIIXX",
        "IYZZXIIIIIXY",
        "IXZZXIIIIIXY",
        "IYZZYIIIIIXY",
        "IXZZYIIIIIXY"
    ],
    [
        "IYZZZXIYZXII",
        "IXZZZXIYZXII",
        "IYZZZYIYZXII",
        "IXZZZYIYZXII",
        "IYZZZXIYZYII",
        "IXZZZXIYZYII",
        "IYZZZYIYZYII",
        "IXZZZYIYZYII",
        "IYZZZXIXZXII",
        "IXZZZXIXZXII",
        "IYZZZYIXZXII",
        "IXZZZYIXZXII",
        "IYZZZXIXZYII",
        "IXZZZXIXZYII",
        "IYZZZYIXZYII",
        "IXZZZYIXZYII"
    ],
    [
        "IYZZZXIYZZZX",
        "IXZZZXIYZZZX",
        "IYZZZYIYZZZX",
        "IXZZZYIYZZZX",
        "IYZZZXIYZZZY",
        "IXZZZXIYZZZY",
        "IYZZZYIYZZZY",
        "IXZZZYIYZZZY",
        "IYZZZXIXZZZX",
        "IXZZZXIXZZZX",
        "IYZZZYIXZZZX",
        "IXZZZYIXZZZX",
        "IYZZZXIXZZZY",
        "IXZZZXIXZZZY",
        "IYZZZYIXZZZY",
        "IXZZZYIXZZZY"
    ],
    [
        "IYZZZXIIIYZX",
        "IXZZZXIIIYZX",
        "IYZZZYIIIYZX",
        "IXZZZYIIIYZX",
        "IYZZZXIIIYZY",
        "IXZZZXIIIYZY",
        "IYZZZYIIIYZY",
        "IXZZZYIIIYZY",
        "IYZZZXIIIXZX",
        "IXZZZXIIIXZX",
        "IYZZZYIIIXZX",
        "IXZZZYIIIXZX",
        "IYZZZXIIIXZY",
        "IXZZZXIIIXZY",
        "IYZZZYIIIXZY",
        "IXZZZYIIIXZY"
    ],
    [
        "IIYXIIYXIIII",
        "IIXXIIYXIIII",
        "IIYYIIYXIIII",
        "IIXYIIYXIIII",
        "IIYXIIYYIIII",
        "IIXXIIYYIIII",
        "IIYYIIYYIIII",
        "IIXYIIYYIIII",
        "IIYXIIXXIIII",
        "IIXXIIXXIIII",
        "IIYYIIXXIIII",
        "IIXYIIXXIIII",
        "IIYXIIXYIIII",
        "IIXXIIXYIIII",
        "IIYYIIXYIIII",
        "IIXYIIXYIIII"
    ],
    [
        "IIYXIIYZZXII",
        "IIXXIIYZZXII",
        "IIYYIIYZZXII",
        "IIXYIIYZZXII",
        "IIYXIIYZZYII",
        "IIXXIIYZZYII",
        "IIYYIIYZZYII",
        "IIXYIIYZZYII",
        "IIYXIIXZZXII",
        "IIXXIIXZZXII",
        "IIYYIIXZZXII",
        "IIXYIIXZZXII",
        "IIYXIIXZZYII",
        "IIXXIIXZZYII",
        "IIYYIIXZZYII",
        "IIXYIIXZZYII"
    ],
    [
        "IIYXIIYZZZZX",
        "IIXXIIYZZZZX",
        "IIYYIIYZZZZX",
        "IIXYIIYZZZZX",
        "IIYXIIYZZZZY",
        "IIXXIIYZZZZY",
        "IIYYIIYZZZZY",
        "IIXYIIYZZZZY",
        "IIYXIIXZZZZX",
        "IIXXIIXZZZZX",
        "IIYYIIXZZZZX",
        "IIXYIIXZZZZX",
        "IIYXIIXZZZZY",
        "IIXXIIXZZZZY",
        "IIYYIIXZZZZY",
        "IIXYIIXZZZZY"
    ],
    [
        "IIYXIIIYXIII",
        "IIXXIIIYXIII",
        "IIYYIIIYXIII",
        "IIXYIIIYXIII",
        "IIYXIIIYYIII",
        "IIXXIIIYYIII",
        "IIYYIIIYYIII",
        "IIXYIIIYYIII",
        "IIYXIIIXXIII",
        "IIXXIIIXXIII",
        "IIYYIIIXXIII",
        "IIXYIIIXXIII",
        "IIYXIIIXYIII",
        "IIXXIIIXYIII",
        "IIYYIIIXYIII",
        "IIXYIIIXYIII"
    ],
    [
        "IIYXIIIYZZXI",
        "IIXXIIIYZZXI",
        "IIYYIIIYZZXI",
        "IIXYIIIYZZXI",
        "IIYXIIIYZZYI",
        "IIXXIIIYZZYI",
        "IIYYIIIYZZYI",
        "IIXYIIIYZZYI",
        "IIYXIIIXZZXI",
        "IIXXIIIXZZXI",
        "IIYYIIIXZZXI",
        "IIXYIIIXZZXI",
        "IIYXIIIXZZYI",
        "IIXXIIIXZZYI",
        "IIYYIIIXZZYI",
        "IIXYIIIXZZYI"
    ],
    [
        "IIYXIIIIYXII",
        "IIXXIIIIYXII",
        "IIYYIIIIYXII",
        "IIXYIIIIYXII",
        "IIYXIIIIYYII",
        "IIXXIIIIYYII",
        "IIYYIIIIYYII",
        "IIXYIIIIYYII",
        "IIYXIIIIXXII",
        "IIXXIIIIXXII",
        "IIYYIIIIXXII",
        "IIXYIIIIXXII",
        "IIYXIIIIXYII",
        "IIXXIIIIXYII",
        "IIYYIIIIXYII",
        "IIXYIIIIXYII"
    ],
    [
        "IIYXIIIIYZZX",
        "IIXXIIIIYZZX",
        "IIYYIIIIYZZX",
        "IIXYIIIIYZZX",
        "IIYXIIIIYZZY",
        "IIXXIIIIYZZY",
        "IIYYIIIIYZZY",
        "IIXYIIIIYZZY",
        "IIYXIIIIXZZX",
        "IIXXIIIIXZZX",
        "IIYYIIIIXZZX",
        "IIXYIIIIXZZX",
        "IIYXIIIIXZZY",
        "IIXXIIIIXZZY",
        "IIYYIIIIXZZY",
        "IIXYIIIIXZZY"
    ],
    [
        "IIYXIIIIIYXI",
        "IIXXIIIIIYXI",
        "IIYYIIIIIYXI",
        "IIXYIIIIIYXI",
        "IIYXIIIIIYYI",
        "IIXXIIIIIYYI",
        "IIYYIIIIIYYI",
        "IIXYIIIIIYYI",
        "IIYXIIIIIXXI",
        "IIXXIIIIIXXI",
        "IIYYIIIIIXXI",
        "IIXYIIIIIXXI",
        "IIYXIIIIIXYI",
        "IIXXIIIIIXYI",
        "IIYYIIIIIXYI",
        "IIXYIIIIIXYI"
    ],
    [
        "IIYXIIIIIIYX",
        "IIXXIIIIIIYX",
        "IIYYIIIIIIYX",
        "IIXYIIIIIIYX",
        "IIYXIIIIIIYY",
        "IIXXIIIIIIYY",
        "IIYYIIIIIIYY",
        "IIXYIIIIIIYY",
        "IIYXIIIIIIXX",
        "IIXXIIIIIIXX",
        "IIYYIIIIIIXX",
        "IIXYIIIIIIXX",
        "IIYXIIIIIIXY",
        "IIXXIIIIIIXY",
        "IIYYIIIIIIXY",
        "IIXYIIIIIIXY"
    ],
    [
        "IIYZXIYZXIII",
        "IIXZXIYZXIII",
        "IIYZYIYZXIII",
        "IIXZYIYZXIII",
        "IIYZXIYZYIII",
        "IIXZXIYZYIII",
        "IIYZYIYZYIII",
        "IIXZYIYZYIII",
        "IIYZXIXZXIII",
        "IIXZXIXZXIII",
        "IIYZYIXZXIII",
        "IIXZYIXZXIII",
        "IIYZXIXZYIII",
        "IIXZXIXZYIII",
        "IIYZYIXZYIII",
        "IIXZYIXZYIII"
    ],
    [
        "IIYZXIYZZZXI",
        "IIXZXIYZZZXI",
        "IIYZYIYZZZXI",
        "IIXZYIYZZZXI",
        "IIYZXIYZZZYI",
        "IIXZXIYZZZYI",
        "IIYZYIYZZZYI",
        "IIXZYIYZZZYI",
        "IIYZXIXZZZXI",
        "IIXZXIXZZZXI",
        "IIYZYIXZZZXI",
        "IIXZYIXZZZXI",
        "IIYZXIXZZZYI",
        "IIXZXIXZZZYI",
        "IIYZYIXZZZYI",
        "IIXZYIXZZZYI"
    ],
    [
        "IIYZXIIIYZXI",
        "IIXZXIIIYZXI",
        "IIYZYIIIYZXI",
        "IIXZYIIIYZXI",
        "IIYZXIIIYZYI",
        "IIXZXIIIYZYI",
        "IIYZYIIIYZYI",
        "IIXZYIIIYZYI",
        "IIYZXIIIXZXI",
        "IIXZXIIIXZXI",
        "IIYZYIIIXZXI",
        "IIXZYIIIXZXI",
        "IIYZXIIIXZYI",
        "IIXZXIIIXZYI",
        "IIYZYIIIXZYI",
        "IIXZYIIIXZYI"
    ],
    [
        "IIYZZXYXIIII",
        "IIXZZXYXIIII",
        "IIYZZYYXIIII",
        "IIXZZYYXIIII",
        "IIYZZXYYIIII",
        "IIXZZXYYIIII",
        "IIYZZYYYIIII",
        "IIXZZYYYIIII",
        "IIYZZXXXIIII",
        "IIXZZXXXIIII",
        "IIYZZYXXIIII",
        "IIXZZYXXIIII",
        "IIYZZXXYIIII",
        "IIXZZXXYIIII",
        "IIYZZYXYIIII",
        "IIXZZYXYIIII"
    ],
    [
        "IIYZZXYZZXII",
        "IIXZZXYZZXII",
        "IIYZZYYZZXII",
        "IIXZZYYZZXII",
        "IIYZZXYZZYII",
        "IIXZZXYZZYII",
        "IIYZZYYZZYII",
        "IIXZZYYZZYII",
        "IIYZZXXZZXII",
        "IIXZZXXZZXII",
        "IIYZZYXZZXII",
        "IIXZZYXZZXII",
        "IIYZZXXZZYII",
        "IIXZZXXZZYII",
        "IIYZZYXZZYII",
        "IIXZZYXZZYII"
    ],
    [
        "IIYZZXYZZZZX",
        "IIXZZXYZZZZX",
        "IIYZZYYZZZZX",
        "IIXZZYYZZZZX",
        "IIYZZXYZZZZY",
        "IIXZZXYZZZZY",
        "IIYZZYYZZZZY",
        "IIXZZYYZZZZY",
        "IIYZZXXZZZZX",
        "IIXZZXXZZZZX",
        "IIYZZYXZZZZX",
        "IIXZZYXZZZZX",
        "IIYZZXXZZZZY",
        "IIXZZXXZZZZY",
        "IIYZZYXZZZZY",
        "IIXZZYXZZZZY"
    ],
    [
        "IIYZZXIYXIII",
        "IIXZZXIYXIII",
        "IIYZZYIYXIII",
        "IIXZZYIYXIII",
        "IIYZZXIYYIII",
        "IIXZZXIYYIII",
        "IIYZZYIYYIII",
        "IIXZZYIYYIII",
        "IIYZZXIXXIII",
        "IIXZZXIXXIII",
        "IIYZZYIXXIII",
        "IIXZZYIXXIII",
        "IIYZZXIXYIII",
        "IIXZZXIXYIII",
        "IIYZZYIXYIII",
        "IIXZZYIXYIII"
    ],
    [
        "IIYZZXIYZZXI",
        "IIXZZXIYZZXI",
        "IIYZZYIYZZXI",
        "IIXZZYIYZZXI",
        "IIYZZXIYZZYI",
        "IIXZZXIYZZYI",
        "IIYZZYIYZZYI",
        "IIXZZYIYZZYI",
        "IIYZZXIXZZXI",
        "IIXZZXIXZZXI",
        "IIYZZYIXZZXI",
        "IIXZZYIXZZXI",
        "IIYZZXIXZZYI",
        "IIXZZXIXZZYI",
        "IIYZZYIXZZYI",
        "IIXZZYIXZZYI"
    ],
    [
        "IIYZZXIIYXII",
        "IIXZZXIIYXII",
        "IIYZZYIIYXII",
        "IIXZZYIIYXII",
        "IIYZZXIIYYII",
        "IIXZZXIIYYII",
        "IIYZZYIIYYII",
        "IIXZZYIIYYII",
        "IIYZZXIIXXII",
        "IIXZZXIIXXII",
        "IIYZZYIIXXII",
        "IIXZZYIIXXII",
        "IIYZZXIIXYII",
        "IIXZZXIIXYII",
        "IIYZZYIIXYII",
        "IIXZZYIIXYII"
    ],
    [
        "IIYZZXIIYZZX",
        "IIXZZXIIYZZX",
        "IIYZZYIIYZZX",
        "IIXZZYIIYZZX",
        "IIYZZXIIYZZY",
        "IIXZZXIIYZZY",
        "IIYZZYIIYZZY",
        "IIXZZYIIYZZY",
        "IIYZZXIIXZZX",
        "IIXZZXIIXZZX",
        "IIYZZYIIXZZX",
        "IIXZZYIIXZZX",
        "IIYZZXIIXZZY",
        "IIXZZXIIXZZY",
        "IIYZZYIIXZZY",
        "IIXZZYIIXZZY"
    ],
    [
        "IIYZZXIIIYXI",
        "IIXZZXIIIYXI",
        "IIYZZYIIIYXI",
        "IIXZZYIIIYXI",
        "IIYZZXIIIYYI",
        "IIXZZXIIIYYI",
        "IIYZZYIIIYYI",
        "IIXZZYIIIYYI",
        "IIYZZXIIIXXI",
        "IIXZZXIIIXXI",
        "IIYZZYIIIXXI",
        "IIXZZYIIIXXI",
        "IIYZZXIIIXYI",
        "IIXZZXIIIXYI",
        "IIYZZYIIIXYI",
        "IIXZZYIIIXYI"
    ],
    [
        "IIYZZXIIIIYX",
        "IIXZZXIIIIYX",
        "IIYZZYIIIIYX",
        "IIXZZYIIIIYX",
        "IIYZZXIIIIYY",
        "IIXZZXIIIIYY",
        "IIYZZYIIIIYY",
        "IIXZZYIIIIYY",
        "IIYZZXIIIIXX",
        "IIXZZXIIIIXX",
        "IIYZZYIIIIXX",
        "IIXZZYIIIIXX",
        "IIYZZXIIIIXY",
        "IIXZZXIIIIXY",
        "IIYZZYIIIIXY",
        "IIXZZYIIIIXY"
    ],
    [
        "IIIYXIYXIIII",
        "IIIXXIYXIIII",
        "IIIYYIYXIIII",
        "IIIXYIYXIIII",
        "IIIYXIYYIIII",
        "IIIXXIYYIIII",
        "IIIYYIYYIIII",
        "IIIXYIYYIIII",
        "IIIYXIXXIIII",
        "IIIXXIXXIIII",
        "IIIYYIXXIIII",
        "IIIXYIXXIIII",
        "IIIYXIXYIIII",
        "IIIXXIXYIIII",
        "IIIYYIXYIIII",
        "IIIXYIXYIIII"
    ],
    [
        "IIIYXIYZZXII",
        "IIIXXIYZZXII",
        "IIIYYIYZZXII",
        "IIIXYIYZZXII",
        "IIIYXIYZZYII",
        "IIIXXIYZZYII",
        "IIIYYIYZZYII",
        "IIIXYIYZZYII",
        "IIIYXIXZZXII",
        "IIIXXIXZZXII",
        "IIIYYIXZZXII",
        "IIIXYIXZZXII",
        "IIIYXIXZZYII",
        "IIIXXIXZZYII",
        "IIIYYIXZZYII",
        "IIIXYIXZZYII"
    ],
    [
        "IIIYXIYZZZZX",
        "IIIXXIYZZZZX",
        "IIIYYIYZZZZX",
        "IIIXYIYZZZZX",
        "IIIYXIYZZZZY",
        "IIIXXIYZZZZY",
        "IIIYYIYZZZZY",
        "IIIXYIYZZZZY",
        "IIIYXIXZZZZX",
        "IIIXXIXZZZZX",
        "IIIYYIXZZZZX",
        "IIIXYIXZZZZX",
        "IIIYXIXZZZZY",
        "IIIXXIXZZZZY",
        "IIIYYIXZZZZY",
        "IIIXYIXZZZZY"
    ],
    [
        "IIIYXIIYXIII",
        "IIIXXIIYXIII",
        "IIIYYIIYXIII",
        "IIIXYIIYXIII",
        "IIIYXIIYYIII",
        "IIIXXIIYYIII",
        "IIIYYIIYYIII",
        "IIIXYIIYYIII",
        "IIIYXIIXXIII",
        "IIIXXIIXXIII",
        "IIIYYIIXXIII",
        "IIIXYIIXXIII",
        "IIIYXIIXYIII",
        "IIIXXIIXYIII",
        "IIIYYIIXYIII",
        "IIIXYIIXYIII"
    ],
    [
        "IIIYXIIYZZXI",
        "IIIXXIIYZZXI",
        "IIIYYIIYZZXI",
        "IIIXYIIYZZXI",
        "IIIYXIIYZZYI",
        "IIIXXIIYZZYI",
        "IIIYYIIYZZYI",
        "IIIXYIIYZZYI",
        "IIIYXIIXZZXI",
        "IIIXXIIXZZXI",
        "IIIYYIIXZZXI",
        "IIIXYIIXZZXI",
        "IIIYXIIXZZYI",
        "IIIXXIIXZZYI",
        "IIIYYIIXZZYI",
        "IIIXYIIXZZYI"
    ],
    [
        "IIIYXIIIYXII",
        "IIIXXIIIYXII",
        "IIIYYIIIYXII",
        "IIIXYIIIYXII",
        "IIIYXIIIYYII",
        "IIIXXIIIYYII",
        "IIIYYIIIYYII",
        "IIIXYIIIYYII",
        "IIIYXIIIXXII",
        "IIIXXIIIXXII",
        "IIIYYIIIXXII",
        "IIIXYIIIXXII",
        "IIIYXIIIXYII",
        "IIIXXIIIXYII",
        "IIIYYIIIXYII",
        "IIIXYIIIXYII"
    ],
    [
        "IIIYXIIIYZZX",
        "IIIXXIIIYZZX",
        "IIIYYIIIYZZX",
        "IIIXYIIIYZZX",
        "IIIYXIIIYZZY",
        "IIIXXIIIYZZY",
        "IIIYYIIIYZZY",
        "IIIXYIIIYZZY",
        "IIIYXIIIXZZX",
        "IIIXXIIIXZZX",
        "IIIYYIIIXZZX",
        "IIIXYIIIXZZX",
        "IIIYXIIIXZZY",
        "IIIXXIIIXZZY",
        "IIIYYIIIXZZY",
        "IIIXYIIIXZZY"
    ],
    [
        "IIIYXIIIIYXI",
        "IIIXXIIIIYXI",
        "IIIYYIIIIYXI",
        "IIIXYIIIIYXI",
        "IIIYXIIIIYYI",
        "IIIXXIIIIYYI",
        "IIIYYIIIIYYI",
        "IIIXYIIIIYYI",
        "IIIYXIIIIXXI",
        "IIIXXIIIIXXI",
        "IIIYYIIIIXXI",
        "IIIXYIIIIXXI",
        "IIIYXIIIIXYI",
        "IIIXXIIIIXYI",
        "IIIYYIIIIXYI",
        "IIIXYIIIIXYI"
    ],
    [
        "IIIYXIIIIIYX",
        "IIIXXIIIIIYX",
        "IIIYYIIIIIYX",
        "IIIXYIIIIIYX",
        "IIIYXIIIIIYY",
        "IIIXXIIIIIYY",
        "IIIYYIIIIIYY",
        "IIIXYIIIIIYY",
        "IIIYXIIIIIXX",
        "IIIXXIIIIIXX",
        "IIIYYIIIIIXX",
        "IIIXYIIIIIXX",
        "IIIYXIIIIIXY",
        "IIIXXIIIIIXY",
        "IIIYYIIIIIXY",
        "IIIXYIIIIIXY"
    ],
    [
        "IIIYZXIYZXII",
        "IIIXZXIYZXII",
        "IIIYZYIYZXII",
        "IIIXZYIYZXII",
        "IIIYZXIYZYII",
        "IIIXZXIYZYII",
        "IIIYZYIYZYII",
        "IIIXZYIYZYII",
        "IIIYZXIXZXII",
        "IIIXZXIXZXII",
        "IIIYZYIXZXII",
        "IIIXZYIXZXII",
        "IIIYZXIXZYII",
        "IIIXZXIXZYII",
        "IIIYZYIXZYII",
        "IIIXZYIXZYII"
    ],
    [
        "IIIYZXIYZZZX",
        "IIIXZXIYZZZX",
        "IIIYZYIYZZZX",
        "IIIXZYIYZZZX",
        "IIIYZXIYZZZY",
        "IIIXZXIYZZZY",
        "IIIYZYIYZZZY",
        "IIIXZYIYZZZY",
        "IIIYZXIXZZZX",
        "IIIXZXIXZZZX",
        "IIIYZYIXZZZX",
        "IIIXZYIXZZZX",
        "IIIYZXIXZZZY",
        "IIIXZXIXZZZY",
        "IIIYZYIXZZZY",
        "IIIXZYIXZZZY"
    ],
    [
        "IIIYZXIIIYZX",
        "IIIXZXIIIYZX",
        "IIIYZYIIIYZX",
        "IIIXZYIIIYZX",
        "IIIYZXIIIYZY",
        "IIIXZXIIIYZY",
        "IIIYZYIIIYZY",
        "IIIXZYIIIYZY",
        "IIIYZXIIIXZX",
        "IIIXZXIIIXZX",
        "IIIYZYIIIXZX",
        "IIIXZYIIIXZX",
        "IIIYZXIIIXZY",
        "IIIXZXIIIXZY",
        "IIIYZYIIIXZY",
        "IIIXZYIIIXZY"
    ],
    [
        "IIIIYXYXIIII",
        "IIIIXXYXIIII",
        "IIIIYYYXIIII",
        "IIIIXYYXIIII",
        "IIIIYXYYIIII",
        "IIIIXXYYIIII",
        "IIIIYYYYIIII",
        "IIIIXYYYIIII",
        "IIIIYXXXIIII",
        "IIIIXXXXIIII",
        "IIIIYYXXIIII",
        "IIIIXYXXIIII",
        "IIIIYXXYIIII",
        "IIIIXXXYIIII",
        "IIIIYYXYIIII",
        "IIIIXYXYIIII"
    ],
    [
        "IIIIYXYZZXII",
        "IIIIXXYZZXII",
        "IIIIYYYZZXII",
        "IIIIXYYZZXII",
        "IIIIYXYZZYII",
        "IIIIXXYZZYII",
        "IIIIYYYZZYII",
        "IIIIXYYZZYII",
        "IIIIYXXZZXII",
        "IIIIXXXZZXII",
        "IIIIYYXZZXII",
        "IIIIXYXZZXII",
        "IIIIYXXZZYII",
        "IIIIXXXZZYII",
        "IIIIYYXZZYII",
        "IIIIXYXZZYII"
    ],
    [
        "IIIIYXYZZZZX",
        "IIIIXXYZZZZX",
        "IIIIYYYZZZZX",
        "IIIIXYYZZZZX",
        "IIIIYXYZZZZY",
        "IIIIXXYZZZZY",
        "IIIIYYYZZZZY",
        "IIIIXYYZZZZY",
        "IIIIYXXZZZZX",
        "IIIIXXXZZZZX",
        "IIIIYYXZZZZX",
        "IIIIXYXZZZZX",
        "IIIIYXXZZZZY",
        "IIIIXXXZZZZY",
        "IIIIYYXZZZZY",
        "IIIIXYXZZZZY"
    ],
    [
        "IIIIYXIYXIII",
        "IIIIXXIYXIII",
        "IIIIYYIYXIII",
        "IIIIXYIYXIII",
        "IIIIYXIYYIII",
        "IIIIXXIYYIII",
        "IIIIYYIYYIII",
        "IIIIXYIYYIII",
        "IIIIYXIXXIII",
        "IIIIXXIXXIII",
        "IIIIYYIXXIII",
        "IIIIXYIXXIII",
        "IIIIYXIXYIII",
        "IIIIXXIXYIII",
        "IIIIYYIXYIII",
        "IIIIXYIXYIII"
    ],
    [
        "IIIIYXIYZZXI",
        "IIIIXXIYZZXI",
        "IIIIYYIYZZXI",
        "IIIIXYIYZZXI",
        "IIIIYXIYZZYI",
        "IIIIXXIYZZYI",
        "IIIIYYIYZZYI",
        "IIIIXYIYZZYI",
        "IIIIYXIXZZXI",
        "IIIIXXIXZZXI",
        "IIIIYYIXZZXI",
        "IIIIXYIXZZXI",
        "IIIIYXIXZZYI",
        "IIIIXXIXZZYI",
        "IIIIYYIXZZYI",
        "IIIIXYIXZZYI"
    ],
    [
        "IIIIYXIIYXII",
        "IIIIXXIIYXII",
        "IIIIYYIIYXII",
        "IIIIXYIIYXII",
        "IIIIYXIIYYII",
        "IIIIXXIIYYII",
        "IIIIYYIIYYII",
        "IIIIXYIIYYII",
        "IIIIYXIIXXII",
        "IIIIXXIIXXII",
        "IIIIYYIIXXII",
        "IIIIXYIIXXII",
        "IIIIYXIIXYII",
        "IIIIXXIIXYII",
        "IIIIYYIIXYII",
        "IIIIXYIIXYII"
    ],
    [
        "IIIIYXIIYZZX",
        "IIIIXXIIYZZX",
        "IIIIYYIIYZZX",
        "IIIIXYIIYZZX",
        "IIIIYXIIYZZY",
        "IIIIXXIIYZZY",
        "IIIIYYIIYZZY",
        "IIIIXYIIYZZY",
        "IIIIYXIIXZZX",
        "IIIIXXIIXZZX",
        "IIIIYYIIXZZX",
        "IIIIXYIIXZZX",
        "IIIIYXIIXZZY",
        "IIIIXXIIXZZY",
        "IIIIYYIIXZZY",
        "IIIIXYIIXZZY"
    ],
    [
        "IIIIYXIIIYXI",
        "IIIIXXIIIYXI",
        "IIIIYYIIIYXI",
        "IIIIXYIIIYXI",
        "IIIIYXIIIYYI",
        "IIIIXXIIIYYI",
        "IIIIYYIIIYYI",
        "IIIIXYIIIYYI",
        "IIIIYXIIIXXI",
        "IIIIXXIIIXXI",
        "IIIIYYIIIXXI",
        "IIIIXYIIIXXI",
        "IIIIYXIIIXYI",
        "IIIIXXIIIXYI",
        "IIIIYYIIIXYI",
        "IIIIXYIIIXYI"
    ],
    [
        "IIIIYXIIIIYX",
        "IIIIXXIIIIYX",
        "IIIIYYIIIIYX",
        "IIIIXYIIIIYX",
        "IIIIYXIIIIYY",
        "IIIIXXIIIIYY",
        "IIIIYYIIIIYY",
        "IIIIXYIIIIYY",
        "IIIIYXIIIIXX",
        "IIIIXXIIIIXX",
        "IIIIYYIIIIXX",
        "IIIIXYIIIIXX",
        "IIIIYXIIIIXY",
        "IIIIXXIIIIXY",
        "IIIIYYIIIIXY",
        "IIIIXYIIIIXY"
    ]
]

In [66]:
path = []

for pauli_word in ucc_6_12:
    path += search_prune(Cirq_Tableau(pauli_word))
    
op_count(path)

{'CNOT': 1554, 'Single_q': 7031}

In [68]:
ucc_8_16 = [
    [
        "YZZZZZZZXIIIIIII",
        "XZZZZZZZXIIIIIII",
        "YZZZZZZZYIIIIIII",
        "XZZZZZZZYIIIIIII"
    ],
    [
        "YZZZZZZZZZXIIIII",
        "XZZZZZZZZZXIIIII",
        "YZZZZZZZZZYIIIII",
        "XZZZZZZZZZYIIIII"
    ],
    [
        "YZZZZZZZZZZZXIII",
        "XZZZZZZZZZZZXIII",
        "YZZZZZZZZZZZYIII",
        "XZZZZZZZZZZZYIII"
    ],
    [
        "YZZZZZZZZZZZZZXI",
        "XZZZZZZZZZZZZZXI",
        "YZZZZZZZZZZZZZYI",
        "XZZZZZZZZZZZZZYI"
    ],
    [
        "IYZZZZZZZXIIIIII",
        "IXZZZZZZZXIIIIII",
        "IYZZZZZZZYIIIIII",
        "IXZZZZZZZYIIIIII"
    ],
    [
        "IYZZZZZZZZZXIIII",
        "IXZZZZZZZZZXIIII",
        "IYZZZZZZZZZYIIII",
        "IXZZZZZZZZZYIIII"
    ],
    [
        "IYZZZZZZZZZZZXII",
        "IXZZZZZZZZZZZXII",
        "IYZZZZZZZZZZZYII",
        "IXZZZZZZZZZZZYII"
    ],
    [
        "IYZZZZZZZZZZZZZX",
        "IXZZZZZZZZZZZZZX",
        "IYZZZZZZZZZZZZZY",
        "IXZZZZZZZZZZZZZY"
    ],
    [
        "IIYZZZZZXIIIIIII",
        "IIXZZZZZXIIIIIII",
        "IIYZZZZZYIIIIIII",
        "IIXZZZZZYIIIIIII"
    ],
    [
        "IIYZZZZZZZXIIIII",
        "IIXZZZZZZZXIIIII",
        "IIYZZZZZZZYIIIII",
        "IIXZZZZZZZYIIIII"
    ],
    [
        "IIYZZZZZZZZZXIII",
        "IIXZZZZZZZZZXIII",
        "IIYZZZZZZZZZYIII",
        "IIXZZZZZZZZZYIII"
    ],
    [
        "IIYZZZZZZZZZZZXI",
        "IIXZZZZZZZZZZZXI",
        "IIYZZZZZZZZZZZYI",
        "IIXZZZZZZZZZZZYI"
    ],
    [
        "IIIYZZZZZXIIIIII",
        "IIIXZZZZZXIIIIII",
        "IIIYZZZZZYIIIIII",
        "IIIXZZZZZYIIIIII"
    ],
    [
        "IIIYZZZZZZZXIIII",
        "IIIXZZZZZZZXIIII",
        "IIIYZZZZZZZYIIII",
        "IIIXZZZZZZZYIIII"
    ],
    [
        "IIIYZZZZZZZZZXII",
        "IIIXZZZZZZZZZXII",
        "IIIYZZZZZZZZZYII",
        "IIIXZZZZZZZZZYII"
    ],
    [
        "IIIYZZZZZZZZZZZX",
        "IIIXZZZZZZZZZZZX",
        "IIIYZZZZZZZZZZZY",
        "IIIXZZZZZZZZZZZY"
    ],
    [
        "IIIIYZZZXIIIIIII",
        "IIIIXZZZXIIIIIII",
        "IIIIYZZZYIIIIIII",
        "IIIIXZZZYIIIIIII"
    ],
    [
        "IIIIYZZZZZXIIIII",
        "IIIIXZZZZZXIIIII",
        "IIIIYZZZZZYIIIII",
        "IIIIXZZZZZYIIIII"
    ],
    [
        "IIIIYZZZZZZZXIII",
        "IIIIXZZZZZZZXIII",
        "IIIIYZZZZZZZYIII",
        "IIIIXZZZZZZZYIII"
    ],
    [
        "IIIIYZZZZZZZZZXI",
        "IIIIXZZZZZZZZZXI",
        "IIIIYZZZZZZZZZYI",
        "IIIIXZZZZZZZZZYI"
    ],
    [
        "IIIIIYZZZXIIIIII",
        "IIIIIXZZZXIIIIII",
        "IIIIIYZZZYIIIIII",
        "IIIIIXZZZYIIIIII"
    ],
    [
        "IIIIIYZZZZZXIIII",
        "IIIIIXZZZZZXIIII",
        "IIIIIYZZZZZYIIII",
        "IIIIIXZZZZZYIIII"
    ],
    [
        "IIIIIYZZZZZZZXII",
        "IIIIIXZZZZZZZXII",
        "IIIIIYZZZZZZZYII",
        "IIIIIXZZZZZZZYII"
    ],
    [
        "IIIIIYZZZZZZZZZX",
        "IIIIIXZZZZZZZZZX",
        "IIIIIYZZZZZZZZZY",
        "IIIIIXZZZZZZZZZY"
    ],
    [
        "IIIIIIYZXIIIIIII",
        "IIIIIIXZXIIIIIII",
        "IIIIIIYZYIIIIIII",
        "IIIIIIXZYIIIIIII"
    ],
    [
        "IIIIIIYZZZXIIIII",
        "IIIIIIXZZZXIIIII",
        "IIIIIIYZZZYIIIII",
        "IIIIIIXZZZYIIIII"
    ],
    [
        "IIIIIIYZZZZZXIII",
        "IIIIIIXZZZZZXIII",
        "IIIIIIYZZZZZYIII",
        "IIIIIIXZZZZZYIII"
    ],
    [
        "IIIIIIYZZZZZZZXI",
        "IIIIIIXZZZZZZZXI",
        "IIIIIIYZZZZZZZYI",
        "IIIIIIXZZZZZZZYI"
    ],
    [
        "IIIIIIIYZXIIIIII",
        "IIIIIIIXZXIIIIII",
        "IIIIIIIYZYIIIIII",
        "IIIIIIIXZYIIIIII"
    ],
    [
        "IIIIIIIYZZZXIIII",
        "IIIIIIIXZZZXIIII",
        "IIIIIIIYZZZYIIII",
        "IIIIIIIXZZZYIIII"
    ],
    [
        "IIIIIIIYZZZZZXII",
        "IIIIIIIXZZZZZXII",
        "IIIIIIIYZZZZZYII",
        "IIIIIIIXZZZZZYII"
    ],
    [
        "IIIIIIIYZZZZZZZX",
        "IIIIIIIXZZZZZZZX",
        "IIIIIIIYZZZZZZZY",
        "IIIIIIIXZZZZZZZY"
    ],
    [
        "YXIIIIIIYXIIIIII",
        "XXIIIIIIYXIIIIII",
        "YYIIIIIIYXIIIIII",
        "XYIIIIIIYXIIIIII",
        "YXIIIIIIYYIIIIII",
        "XXIIIIIIYYIIIIII",
        "YYIIIIIIYYIIIIII",
        "XYIIIIIIYYIIIIII",
        "YXIIIIIIXXIIIIII",
        "XXIIIIIIXXIIIIII",
        "YYIIIIIIXXIIIIII",
        "XYIIIIIIXXIIIIII",
        "YXIIIIIIXYIIIIII",
        "XXIIIIIIXYIIIIII",
        "YYIIIIIIXYIIIIII",
        "XYIIIIIIXYIIIIII"
    ],
    [
        "YXIIIIIIYZZXIIII",
        "XXIIIIIIYZZXIIII",
        "YYIIIIIIYZZXIIII",
        "XYIIIIIIYZZXIIII",
        "YXIIIIIIYZZYIIII",
        "XXIIIIIIYZZYIIII",
        "YYIIIIIIYZZYIIII",
        "XYIIIIIIYZZYIIII",
        "YXIIIIIIXZZXIIII",
        "XXIIIIIIXZZXIIII",
        "YYIIIIIIXZZXIIII",
        "XYIIIIIIXZZXIIII",
        "YXIIIIIIXZZYIIII",
        "XXIIIIIIXZZYIIII",
        "YYIIIIIIXZZYIIII",
        "XYIIIIIIXZZYIIII"
    ],
    [
        "YXIIIIIIYZZZZXII",
        "XXIIIIIIYZZZZXII",
        "YYIIIIIIYZZZZXII",
        "XYIIIIIIYZZZZXII",
        "YXIIIIIIYZZZZYII",
        "XXIIIIIIYZZZZYII",
        "YYIIIIIIYZZZZYII",
        "XYIIIIIIYZZZZYII",
        "YXIIIIIIXZZZZXII",
        "XXIIIIIIXZZZZXII",
        "YYIIIIIIXZZZZXII",
        "XYIIIIIIXZZZZXII",
        "YXIIIIIIXZZZZYII",
        "XXIIIIIIXZZZZYII",
        "YYIIIIIIXZZZZYII",
        "XYIIIIIIXZZZZYII"
    ],
    [
        "YXIIIIIIYZZZZZZX",
        "XXIIIIIIYZZZZZZX",
        "YYIIIIIIYZZZZZZX",
        "XYIIIIIIYZZZZZZX",
        "YXIIIIIIYZZZZZZY",
        "XXIIIIIIYZZZZZZY",
        "YYIIIIIIYZZZZZZY",
        "XYIIIIIIYZZZZZZY",
        "YXIIIIIIXZZZZZZX",
        "XXIIIIIIXZZZZZZX",
        "YYIIIIIIXZZZZZZX",
        "XYIIIIIIXZZZZZZX",
        "YXIIIIIIXZZZZZZY",
        "XXIIIIIIXZZZZZZY",
        "YYIIIIIIXZZZZZZY",
        "XYIIIIIIXZZZZZZY"
    ],
    [
        "YXIIIIIIIYXIIIII",
        "XXIIIIIIIYXIIIII",
        "YYIIIIIIIYXIIIII",
        "XYIIIIIIIYXIIIII",
        "YXIIIIIIIYYIIIII",
        "XXIIIIIIIYYIIIII",
        "YYIIIIIIIYYIIIII",
        "XYIIIIIIIYYIIIII",
        "YXIIIIIIIXXIIIII",
        "XXIIIIIIIXXIIIII",
        "YYIIIIIIIXXIIIII",
        "XYIIIIIIIXXIIIII",
        "YXIIIIIIIXYIIIII",
        "XXIIIIIIIXYIIIII",
        "YYIIIIIIIXYIIIII",
        "XYIIIIIIIXYIIIII"
    ],
    [
        "YXIIIIIIIYZZXIII",
        "XXIIIIIIIYZZXIII",
        "YYIIIIIIIYZZXIII",
        "XYIIIIIIIYZZXIII",
        "YXIIIIIIIYZZYIII",
        "XXIIIIIIIYZZYIII",
        "YYIIIIIIIYZZYIII",
        "XYIIIIIIIYZZYIII",
        "YXIIIIIIIXZZXIII",
        "XXIIIIIIIXZZXIII",
        "YYIIIIIIIXZZXIII",
        "XYIIIIIIIXZZXIII",
        "YXIIIIIIIXZZYIII",
        "XXIIIIIIIXZZYIII",
        "YYIIIIIIIXZZYIII",
        "XYIIIIIIIXZZYIII"
    ],
    [
        "YXIIIIIIIYZZZZXI",
        "XXIIIIIIIYZZZZXI",
        "YYIIIIIIIYZZZZXI",
        "XYIIIIIIIYZZZZXI",
        "YXIIIIIIIYZZZZYI",
        "XXIIIIIIIYZZZZYI",
        "YYIIIIIIIYZZZZYI",
        "XYIIIIIIIYZZZZYI",
        "YXIIIIIIIXZZZZXI",
        "XXIIIIIIIXZZZZXI",
        "YYIIIIIIIXZZZZXI",
        "XYIIIIIIIXZZZZXI",
        "YXIIIIIIIXZZZZYI",
        "XXIIIIIIIXZZZZYI",
        "YYIIIIIIIXZZZZYI",
        "XYIIIIIIIXZZZZYI"
    ],
    [
        "YXIIIIIIIIYXIIII",
        "XXIIIIIIIIYXIIII",
        "YYIIIIIIIIYXIIII",
        "XYIIIIIIIIYXIIII",
        "YXIIIIIIIIYYIIII",
        "XXIIIIIIIIYYIIII",
        "YYIIIIIIIIYYIIII",
        "XYIIIIIIIIYYIIII",
        "YXIIIIIIIIXXIIII",
        "XXIIIIIIIIXXIIII",
        "YYIIIIIIIIXXIIII",
        "XYIIIIIIIIXXIIII",
        "YXIIIIIIIIXYIIII",
        "XXIIIIIIIIXYIIII",
        "YYIIIIIIIIXYIIII",
        "XYIIIIIIIIXYIIII"
    ],
    [
        "YXIIIIIIIIYZZXII",
        "XXIIIIIIIIYZZXII",
        "YYIIIIIIIIYZZXII",
        "XYIIIIIIIIYZZXII",
        "YXIIIIIIIIYZZYII",
        "XXIIIIIIIIYZZYII",
        "YYIIIIIIIIYZZYII",
        "XYIIIIIIIIYZZYII",
        "YXIIIIIIIIXZZXII",
        "XXIIIIIIIIXZZXII",
        "YYIIIIIIIIXZZXII",
        "XYIIIIIIIIXZZXII",
        "YXIIIIIIIIXZZYII",
        "XXIIIIIIIIXZZYII",
        "YYIIIIIIIIXZZYII",
        "XYIIIIIIIIXZZYII"
    ],
    [
        "YXIIIIIIIIYZZZZX",
        "XXIIIIIIIIYZZZZX",
        "YYIIIIIIIIYZZZZX",
        "XYIIIIIIIIYZZZZX",
        "YXIIIIIIIIYZZZZY",
        "XXIIIIIIIIYZZZZY",
        "YYIIIIIIIIYZZZZY",
        "XYIIIIIIIIYZZZZY",
        "YXIIIIIIIIXZZZZX",
        "XXIIIIIIIIXZZZZX",
        "YYIIIIIIIIXZZZZX",
        "XYIIIIIIIIXZZZZX",
        "YXIIIIIIIIXZZZZY",
        "XXIIIIIIIIXZZZZY",
        "YYIIIIIIIIXZZZZY",
        "XYIIIIIIIIXZZZZY"
    ],
    [
        "YXIIIIIIIIIYXIII",
        "XXIIIIIIIIIYXIII",
        "YYIIIIIIIIIYXIII",
        "XYIIIIIIIIIYXIII",
        "YXIIIIIIIIIYYIII",
        "XXIIIIIIIIIYYIII",
        "YYIIIIIIIIIYYIII",
        "XYIIIIIIIIIYYIII",
        "YXIIIIIIIIIXXIII",
        "XXIIIIIIIIIXXIII",
        "YYIIIIIIIIIXXIII",
        "XYIIIIIIIIIXXIII",
        "YXIIIIIIIIIXYIII",
        "XXIIIIIIIIIXYIII",
        "YYIIIIIIIIIXYIII",
        "XYIIIIIIIIIXYIII"
    ],
    [
        "YXIIIIIIIIIYZZXI",
        "XXIIIIIIIIIYZZXI",
        "YYIIIIIIIIIYZZXI",
        "XYIIIIIIIIIYZZXI",
        "YXIIIIIIIIIYZZYI",
        "XXIIIIIIIIIYZZYI",
        "YYIIIIIIIIIYZZYI",
        "XYIIIIIIIIIYZZYI",
        "YXIIIIIIIIIXZZXI",
        "XXIIIIIIIIIXZZXI",
        "YYIIIIIIIIIXZZXI",
        "XYIIIIIIIIIXZZXI",
        "YXIIIIIIIIIXZZYI",
        "XXIIIIIIIIIXZZYI",
        "YYIIIIIIIIIXZZYI",
        "XYIIIIIIIIIXZZYI"
    ],
    [
        "YXIIIIIIIIIIYXII",
        "XXIIIIIIIIIIYXII",
        "YYIIIIIIIIIIYXII",
        "XYIIIIIIIIIIYXII",
        "YXIIIIIIIIIIYYII",
        "XXIIIIIIIIIIYYII",
        "YYIIIIIIIIIIYYII",
        "XYIIIIIIIIIIYYII",
        "YXIIIIIIIIIIXXII",
        "XXIIIIIIIIIIXXII",
        "YYIIIIIIIIIIXXII",
        "XYIIIIIIIIIIXXII",
        "YXIIIIIIIIIIXYII",
        "XXIIIIIIIIIIXYII",
        "YYIIIIIIIIIIXYII",
        "XYIIIIIIIIIIXYII"
    ],
    [
        "YXIIIIIIIIIIYZZX",
        "XXIIIIIIIIIIYZZX",
        "YYIIIIIIIIIIYZZX",
        "XYIIIIIIIIIIYZZX",
        "YXIIIIIIIIIIYZZY",
        "XXIIIIIIIIIIYZZY",
        "YYIIIIIIIIIIYZZY",
        "XYIIIIIIIIIIYZZY",
        "YXIIIIIIIIIIXZZX",
        "XXIIIIIIIIIIXZZX",
        "YYIIIIIIIIIIXZZX",
        "XYIIIIIIIIIIXZZX",
        "YXIIIIIIIIIIXZZY",
        "XXIIIIIIIIIIXZZY",
        "YYIIIIIIIIIIXZZY",
        "XYIIIIIIIIIIXZZY"
    ],
    [
        "YXIIIIIIIIIIIYXI",
        "XXIIIIIIIIIIIYXI",
        "YYIIIIIIIIIIIYXI",
        "XYIIIIIIIIIIIYXI",
        "YXIIIIIIIIIIIYYI",
        "XXIIIIIIIIIIIYYI",
        "YYIIIIIIIIIIIYYI",
        "XYIIIIIIIIIIIYYI",
        "YXIIIIIIIIIIIXXI",
        "XXIIIIIIIIIIIXXI",
        "YYIIIIIIIIIIIXXI",
        "XYIIIIIIIIIIIXXI",
        "YXIIIIIIIIIIIXYI",
        "XXIIIIIIIIIIIXYI",
        "YYIIIIIIIIIIIXYI",
        "XYIIIIIIIIIIIXYI"
    ],
    [
        "YXIIIIIIIIIIIIYX",
        "XXIIIIIIIIIIIIYX",
        "YYIIIIIIIIIIIIYX",
        "XYIIIIIIIIIIIIYX",
        "YXIIIIIIIIIIIIYY",
        "XXIIIIIIIIIIIIYY",
        "YYIIIIIIIIIIIIYY",
        "XYIIIIIIIIIIIIYY",
        "YXIIIIIIIIIIIIXX",
        "XXIIIIIIIIIIIIXX",
        "YYIIIIIIIIIIIIXX",
        "XYIIIIIIIIIIIIXX",
        "YXIIIIIIIIIIIIXY",
        "XXIIIIIIIIIIIIXY",
        "YYIIIIIIIIIIIIXY",
        "XYIIIIIIIIIIIIXY"
    ],
    [
        "YZXIIIIIYZXIIIII",
        "XZXIIIIIYZXIIIII",
        "YZYIIIIIYZXIIIII",
        "XZYIIIIIYZXIIIII",
        "YZXIIIIIYZYIIIII",
        "XZXIIIIIYZYIIIII",
        "YZYIIIIIYZYIIIII",
        "XZYIIIIIYZYIIIII",
        "YZXIIIIIXZXIIIII",
        "XZXIIIIIXZXIIIII",
        "YZYIIIIIXZXIIIII",
        "XZYIIIIIXZXIIIII",
        "YZXIIIIIXZYIIIII",
        "XZXIIIIIXZYIIIII",
        "YZYIIIIIXZYIIIII",
        "XZYIIIIIXZYIIIII"
    ],
    [
        "YZXIIIIIYZZZXIII",
        "XZXIIIIIYZZZXIII",
        "YZYIIIIIYZZZXIII",
        "XZYIIIIIYZZZXIII",
        "YZXIIIIIYZZZYIII",
        "XZXIIIIIYZZZYIII",
        "YZYIIIIIYZZZYIII",
        "XZYIIIIIYZZZYIII",
        "YZXIIIIIXZZZXIII",
        "XZXIIIIIXZZZXIII",
        "YZYIIIIIXZZZXIII",
        "XZYIIIIIXZZZXIII",
        "YZXIIIIIXZZZYIII",
        "XZXIIIIIXZZZYIII",
        "YZYIIIIIXZZZYIII",
        "XZYIIIIIXZZZYIII"
    ],
    [
        "YZXIIIIIYZZZZZXI",
        "XZXIIIIIYZZZZZXI",
        "YZYIIIIIYZZZZZXI",
        "XZYIIIIIYZZZZZXI",
        "YZXIIIIIYZZZZZYI",
        "XZXIIIIIYZZZZZYI",
        "YZYIIIIIYZZZZZYI",
        "XZYIIIIIYZZZZZYI",
        "YZXIIIIIXZZZZZXI",
        "XZXIIIIIXZZZZZXI",
        "YZYIIIIIXZZZZZXI",
        "XZYIIIIIXZZZZZXI",
        "YZXIIIIIXZZZZZYI",
        "XZXIIIIIXZZZZZYI",
        "YZYIIIIIXZZZZZYI",
        "XZYIIIIIXZZZZZYI"
    ],
    [
        "YZXIIIIIIIYZXIII",
        "XZXIIIIIIIYZXIII",
        "YZYIIIIIIIYZXIII",
        "XZYIIIIIIIYZXIII",
        "YZXIIIIIIIYZYIII",
        "XZXIIIIIIIYZYIII",
        "YZYIIIIIIIYZYIII",
        "XZYIIIIIIIYZYIII",
        "YZXIIIIIIIXZXIII",
        "XZXIIIIIIIXZXIII",
        "YZYIIIIIIIXZXIII",
        "XZYIIIIIIIXZXIII",
        "YZXIIIIIIIXZYIII",
        "XZXIIIIIIIXZYIII",
        "YZYIIIIIIIXZYIII",
        "XZYIIIIIIIXZYIII"
    ],
    [
        "YZXIIIIIIIYZZZXI",
        "XZXIIIIIIIYZZZXI",
        "YZYIIIIIIIYZZZXI",
        "XZYIIIIIIIYZZZXI",
        "YZXIIIIIIIYZZZYI",
        "XZXIIIIIIIYZZZYI",
        "YZYIIIIIIIYZZZYI",
        "XZYIIIIIIIYZZZYI",
        "YZXIIIIIIIXZZZXI",
        "XZXIIIIIIIXZZZXI",
        "YZYIIIIIIIXZZZXI",
        "XZYIIIIIIIXZZZXI",
        "YZXIIIIIIIXZZZYI",
        "XZXIIIIIIIXZZZYI",
        "YZYIIIIIIIXZZZYI",
        "XZYIIIIIIIXZZZYI"
    ],
    [
        "YZXIIIIIIIIIYZXI",
        "XZXIIIIIIIIIYZXI",
        "YZYIIIIIIIIIYZXI",
        "XZYIIIIIIIIIYZXI",
        "YZXIIIIIIIIIYZYI",
        "XZXIIIIIIIIIYZYI",
        "YZYIIIIIIIIIYZYI",
        "XZYIIIIIIIIIYZYI",
        "YZXIIIIIIIIIXZXI",
        "XZXIIIIIIIIIXZXI",
        "YZYIIIIIIIIIXZXI",
        "XZYIIIIIIIIIXZXI",
        "YZXIIIIIIIIIXZYI",
        "XZXIIIIIIIIIXZYI",
        "YZYIIIIIIIIIXZYI",
        "XZYIIIIIIIIIXZYI"
    ],
    [
        "YZZXIIIIYXIIIIII",
        "XZZXIIIIYXIIIIII",
        "YZZYIIIIYXIIIIII",
        "XZZYIIIIYXIIIIII",
        "YZZXIIIIYYIIIIII",
        "XZZXIIIIYYIIIIII",
        "YZZYIIIIYYIIIIII",
        "XZZYIIIIYYIIIIII",
        "YZZXIIIIXXIIIIII",
        "XZZXIIIIXXIIIIII",
        "YZZYIIIIXXIIIIII",
        "XZZYIIIIXXIIIIII",
        "YZZXIIIIXYIIIIII",
        "XZZXIIIIXYIIIIII",
        "YZZYIIIIXYIIIIII",
        "XZZYIIIIXYIIIIII"
    ],
    [
        "YZZXIIIIYZZXIIII",
        "XZZXIIIIYZZXIIII",
        "YZZYIIIIYZZXIIII",
        "XZZYIIIIYZZXIIII",
        "YZZXIIIIYZZYIIII",
        "XZZXIIIIYZZYIIII",
        "YZZYIIIIYZZYIIII",
        "XZZYIIIIYZZYIIII",
        "YZZXIIIIXZZXIIII",
        "XZZXIIIIXZZXIIII",
        "YZZYIIIIXZZXIIII",
        "XZZYIIIIXZZXIIII",
        "YZZXIIIIXZZYIIII",
        "XZZXIIIIXZZYIIII",
        "YZZYIIIIXZZYIIII",
        "XZZYIIIIXZZYIIII"
    ],
    [
        "YZZXIIIIYZZZZXII",
        "XZZXIIIIYZZZZXII",
        "YZZYIIIIYZZZZXII",
        "XZZYIIIIYZZZZXII",
        "YZZXIIIIYZZZZYII",
        "XZZXIIIIYZZZZYII",
        "YZZYIIIIYZZZZYII",
        "XZZYIIIIYZZZZYII",
        "YZZXIIIIXZZZZXII",
        "XZZXIIIIXZZZZXII",
        "YZZYIIIIXZZZZXII",
        "XZZYIIIIXZZZZXII",
        "YZZXIIIIXZZZZYII",
        "XZZXIIIIXZZZZYII",
        "YZZYIIIIXZZZZYII",
        "XZZYIIIIXZZZZYII"
    ],
    [
        "YZZXIIIIYZZZZZZX",
        "XZZXIIIIYZZZZZZX",
        "YZZYIIIIYZZZZZZX",
        "XZZYIIIIYZZZZZZX",
        "YZZXIIIIYZZZZZZY",
        "XZZXIIIIYZZZZZZY",
        "YZZYIIIIYZZZZZZY",
        "XZZYIIIIYZZZZZZY",
        "YZZXIIIIXZZZZZZX",
        "XZZXIIIIXZZZZZZX",
        "YZZYIIIIXZZZZZZX",
        "XZZYIIIIXZZZZZZX",
        "YZZXIIIIXZZZZZZY",
        "XZZXIIIIXZZZZZZY",
        "YZZYIIIIXZZZZZZY",
        "XZZYIIIIXZZZZZZY"
    ],
    [
        "YZZXIIIIIYXIIIII",
        "XZZXIIIIIYXIIIII",
        "YZZYIIIIIYXIIIII",
        "XZZYIIIIIYXIIIII",
        "YZZXIIIIIYYIIIII",
        "XZZXIIIIIYYIIIII",
        "YZZYIIIIIYYIIIII",
        "XZZYIIIIIYYIIIII",
        "YZZXIIIIIXXIIIII",
        "XZZXIIIIIXXIIIII",
        "YZZYIIIIIXXIIIII",
        "XZZYIIIIIXXIIIII",
        "YZZXIIIIIXYIIIII",
        "XZZXIIIIIXYIIIII",
        "YZZYIIIIIXYIIIII",
        "XZZYIIIIIXYIIIII"
    ],
    [
        "YZZXIIIIIYZZXIII",
        "XZZXIIIIIYZZXIII",
        "YZZYIIIIIYZZXIII",
        "XZZYIIIIIYZZXIII",
        "YZZXIIIIIYZZYIII",
        "XZZXIIIIIYZZYIII",
        "YZZYIIIIIYZZYIII",
        "XZZYIIIIIYZZYIII",
        "YZZXIIIIIXZZXIII",
        "XZZXIIIIIXZZXIII",
        "YZZYIIIIIXZZXIII",
        "XZZYIIIIIXZZXIII",
        "YZZXIIIIIXZZYIII",
        "XZZXIIIIIXZZYIII",
        "YZZYIIIIIXZZYIII",
        "XZZYIIIIIXZZYIII"
    ],
    [
        "YZZXIIIIIYZZZZXI",
        "XZZXIIIIIYZZZZXI",
        "YZZYIIIIIYZZZZXI",
        "XZZYIIIIIYZZZZXI",
        "YZZXIIIIIYZZZZYI",
        "XZZXIIIIIYZZZZYI",
        "YZZYIIIIIYZZZZYI",
        "XZZYIIIIIYZZZZYI",
        "YZZXIIIIIXZZZZXI",
        "XZZXIIIIIXZZZZXI",
        "YZZYIIIIIXZZZZXI",
        "XZZYIIIIIXZZZZXI",
        "YZZXIIIIIXZZZZYI",
        "XZZXIIIIIXZZZZYI",
        "YZZYIIIIIXZZZZYI",
        "XZZYIIIIIXZZZZYI"
    ],
    [
        "YZZXIIIIIIYXIIII",
        "XZZXIIIIIIYXIIII",
        "YZZYIIIIIIYXIIII",
        "XZZYIIIIIIYXIIII",
        "YZZXIIIIIIYYIIII",
        "XZZXIIIIIIYYIIII",
        "YZZYIIIIIIYYIIII",
        "XZZYIIIIIIYYIIII",
        "YZZXIIIIIIXXIIII",
        "XZZXIIIIIIXXIIII",
        "YZZYIIIIIIXXIIII",
        "XZZYIIIIIIXXIIII",
        "YZZXIIIIIIXYIIII",
        "XZZXIIIIIIXYIIII",
        "YZZYIIIIIIXYIIII",
        "XZZYIIIIIIXYIIII"
    ],
    [
        "YZZXIIIIIIYZZXII",
        "XZZXIIIIIIYZZXII",
        "YZZYIIIIIIYZZXII",
        "XZZYIIIIIIYZZXII",
        "YZZXIIIIIIYZZYII",
        "XZZXIIIIIIYZZYII",
        "YZZYIIIIIIYZZYII",
        "XZZYIIIIIIYZZYII",
        "YZZXIIIIIIXZZXII",
        "XZZXIIIIIIXZZXII",
        "YZZYIIIIIIXZZXII",
        "XZZYIIIIIIXZZXII",
        "YZZXIIIIIIXZZYII",
        "XZZXIIIIIIXZZYII",
        "YZZYIIIIIIXZZYII",
        "XZZYIIIIIIXZZYII"
    ],
    [
        "YZZXIIIIIIYZZZZX",
        "XZZXIIIIIIYZZZZX",
        "YZZYIIIIIIYZZZZX",
        "XZZYIIIIIIYZZZZX",
        "YZZXIIIIIIYZZZZY",
        "XZZXIIIIIIYZZZZY",
        "YZZYIIIIIIYZZZZY",
        "XZZYIIIIIIYZZZZY",
        "YZZXIIIIIIXZZZZX",
        "XZZXIIIIIIXZZZZX",
        "YZZYIIIIIIXZZZZX",
        "XZZYIIIIIIXZZZZX",
        "YZZXIIIIIIXZZZZY",
        "XZZXIIIIIIXZZZZY",
        "YZZYIIIIIIXZZZZY",
        "XZZYIIIIIIXZZZZY"
    ],
    [
        "YZZXIIIIIIIYXIII",
        "XZZXIIIIIIIYXIII",
        "YZZYIIIIIIIYXIII",
        "XZZYIIIIIIIYXIII",
        "YZZXIIIIIIIYYIII",
        "XZZXIIIIIIIYYIII",
        "YZZYIIIIIIIYYIII",
        "XZZYIIIIIIIYYIII",
        "YZZXIIIIIIIXXIII",
        "XZZXIIIIIIIXXIII",
        "YZZYIIIIIIIXXIII",
        "XZZYIIIIIIIXXIII",
        "YZZXIIIIIIIXYIII",
        "XZZXIIIIIIIXYIII",
        "YZZYIIIIIIIXYIII",
        "XZZYIIIIIIIXYIII"
    ],
    [
        "YZZXIIIIIIIYZZXI",
        "XZZXIIIIIIIYZZXI",
        "YZZYIIIIIIIYZZXI",
        "XZZYIIIIIIIYZZXI",
        "YZZXIIIIIIIYZZYI",
        "XZZXIIIIIIIYZZYI",
        "YZZYIIIIIIIYZZYI",
        "XZZYIIIIIIIYZZYI",
        "YZZXIIIIIIIXZZXI",
        "XZZXIIIIIIIXZZXI",
        "YZZYIIIIIIIXZZXI",
        "XZZYIIIIIIIXZZXI",
        "YZZXIIIIIIIXZZYI",
        "XZZXIIIIIIIXZZYI",
        "YZZYIIIIIIIXZZYI",
        "XZZYIIIIIIIXZZYI"
    ],
    [
        "YZZXIIIIIIIIYXII",
        "XZZXIIIIIIIIYXII",
        "YZZYIIIIIIIIYXII",
        "XZZYIIIIIIIIYXII",
        "YZZXIIIIIIIIYYII",
        "XZZXIIIIIIIIYYII",
        "YZZYIIIIIIIIYYII",
        "XZZYIIIIIIIIYYII",
        "YZZXIIIIIIIIXXII",
        "XZZXIIIIIIIIXXII",
        "YZZYIIIIIIIIXXII",
        "XZZYIIIIIIIIXXII",
        "YZZXIIIIIIIIXYII",
        "XZZXIIIIIIIIXYII",
        "YZZYIIIIIIIIXYII",
        "XZZYIIIIIIIIXYII"
    ],
    [
        "YZZXIIIIIIIIYZZX",
        "XZZXIIIIIIIIYZZX",
        "YZZYIIIIIIIIYZZX",
        "XZZYIIIIIIIIYZZX",
        "YZZXIIIIIIIIYZZY",
        "XZZXIIIIIIIIYZZY",
        "YZZYIIIIIIIIYZZY",
        "XZZYIIIIIIIIYZZY",
        "YZZXIIIIIIIIXZZX",
        "XZZXIIIIIIIIXZZX",
        "YZZYIIIIIIIIXZZX",
        "XZZYIIIIIIIIXZZX",
        "YZZXIIIIIIIIXZZY",
        "XZZXIIIIIIIIXZZY",
        "YZZYIIIIIIIIXZZY",
        "XZZYIIIIIIIIXZZY"
    ],
    [
        "YZZXIIIIIIIIIYXI",
        "XZZXIIIIIIIIIYXI",
        "YZZYIIIIIIIIIYXI",
        "XZZYIIIIIIIIIYXI",
        "YZZXIIIIIIIIIYYI",
        "XZZXIIIIIIIIIYYI",
        "YZZYIIIIIIIIIYYI",
        "XZZYIIIIIIIIIYYI",
        "YZZXIIIIIIIIIXXI",
        "XZZXIIIIIIIIIXXI",
        "YZZYIIIIIIIIIXXI",
        "XZZYIIIIIIIIIXXI",
        "YZZXIIIIIIIIIXYI",
        "XZZXIIIIIIIIIXYI",
        "YZZYIIIIIIIIIXYI",
        "XZZYIIIIIIIIIXYI"
    ],
    [
        "YZZXIIIIIIIIIIYX",
        "XZZXIIIIIIIIIIYX",
        "YZZYIIIIIIIIIIYX",
        "XZZYIIIIIIIIIIYX",
        "YZZXIIIIIIIIIIYY",
        "XZZXIIIIIIIIIIYY",
        "YZZYIIIIIIIIIIYY",
        "XZZYIIIIIIIIIIYY",
        "YZZXIIIIIIIIIIXX",
        "XZZXIIIIIIIIIIXX",
        "YZZYIIIIIIIIIIXX",
        "XZZYIIIIIIIIIIXX",
        "YZZXIIIIIIIIIIXY",
        "XZZXIIIIIIIIIIXY",
        "YZZYIIIIIIIIIIXY",
        "XZZYIIIIIIIIIIXY"
    ],
    [
        "YZZZXIIIYZXIIIII",
        "XZZZXIIIYZXIIIII",
        "YZZZYIIIYZXIIIII",
        "XZZZYIIIYZXIIIII",
        "YZZZXIIIYZYIIIII",
        "XZZZXIIIYZYIIIII",
        "YZZZYIIIYZYIIIII",
        "XZZZYIIIYZYIIIII",
        "YZZZXIIIXZXIIIII",
        "XZZZXIIIXZXIIIII",
        "YZZZYIIIXZXIIIII",
        "XZZZYIIIXZXIIIII",
        "YZZZXIIIXZYIIIII",
        "XZZZXIIIXZYIIIII",
        "YZZZYIIIXZYIIIII",
        "XZZZYIIIXZYIIIII"
    ],
    [
        "YZZZXIIIYZZZXIII",
        "XZZZXIIIYZZZXIII",
        "YZZZYIIIYZZZXIII",
        "XZZZYIIIYZZZXIII",
        "YZZZXIIIYZZZYIII",
        "XZZZXIIIYZZZYIII",
        "YZZZYIIIYZZZYIII",
        "XZZZYIIIYZZZYIII",
        "YZZZXIIIXZZZXIII",
        "XZZZXIIIXZZZXIII",
        "YZZZYIIIXZZZXIII",
        "XZZZYIIIXZZZXIII",
        "YZZZXIIIXZZZYIII",
        "XZZZXIIIXZZZYIII",
        "YZZZYIIIXZZZYIII",
        "XZZZYIIIXZZZYIII"
    ],
    [
        "YZZZXIIIYZZZZZXI",
        "XZZZXIIIYZZZZZXI",
        "YZZZYIIIYZZZZZXI",
        "XZZZYIIIYZZZZZXI",
        "YZZZXIIIYZZZZZYI",
        "XZZZXIIIYZZZZZYI",
        "YZZZYIIIYZZZZZYI",
        "XZZZYIIIYZZZZZYI",
        "YZZZXIIIXZZZZZXI",
        "XZZZXIIIXZZZZZXI",
        "YZZZYIIIXZZZZZXI",
        "XZZZYIIIXZZZZZXI",
        "YZZZXIIIXZZZZZYI",
        "XZZZXIIIXZZZZZYI",
        "YZZZYIIIXZZZZZYI",
        "XZZZYIIIXZZZZZYI"
    ],
    [
        "YZZZXIIIIIYZXIII",
        "XZZZXIIIIIYZXIII",
        "YZZZYIIIIIYZXIII",
        "XZZZYIIIIIYZXIII",
        "YZZZXIIIIIYZYIII",
        "XZZZXIIIIIYZYIII",
        "YZZZYIIIIIYZYIII",
        "XZZZYIIIIIYZYIII",
        "YZZZXIIIIIXZXIII",
        "XZZZXIIIIIXZXIII",
        "YZZZYIIIIIXZXIII",
        "XZZZYIIIIIXZXIII",
        "YZZZXIIIIIXZYIII",
        "XZZZXIIIIIXZYIII",
        "YZZZYIIIIIXZYIII",
        "XZZZYIIIIIXZYIII"
    ],
    [
        "YZZZXIIIIIYZZZXI",
        "XZZZXIIIIIYZZZXI",
        "YZZZYIIIIIYZZZXI",
        "XZZZYIIIIIYZZZXI",
        "YZZZXIIIIIYZZZYI",
        "XZZZXIIIIIYZZZYI",
        "YZZZYIIIIIYZZZYI",
        "XZZZYIIIIIYZZZYI",
        "YZZZXIIIIIXZZZXI",
        "XZZZXIIIIIXZZZXI",
        "YZZZYIIIIIXZZZXI",
        "XZZZYIIIIIXZZZXI",
        "YZZZXIIIIIXZZZYI",
        "XZZZXIIIIIXZZZYI",
        "YZZZYIIIIIXZZZYI",
        "XZZZYIIIIIXZZZYI"
    ],
    [
        "YZZZXIIIIIIIYZXI",
        "XZZZXIIIIIIIYZXI",
        "YZZZYIIIIIIIYZXI",
        "XZZZYIIIIIIIYZXI",
        "YZZZXIIIIIIIYZYI",
        "XZZZXIIIIIIIYZYI",
        "YZZZYIIIIIIIYZYI",
        "XZZZYIIIIIIIYZYI",
        "YZZZXIIIIIIIXZXI",
        "XZZZXIIIIIIIXZXI",
        "YZZZYIIIIIIIXZXI",
        "XZZZYIIIIIIIXZXI",
        "YZZZXIIIIIIIXZYI",
        "XZZZXIIIIIIIXZYI",
        "YZZZYIIIIIIIXZYI",
        "XZZZYIIIIIIIXZYI"
    ],
    [
        "YZZZZXIIYXIIIIII",
        "XZZZZXIIYXIIIIII",
        "YZZZZYIIYXIIIIII",
        "XZZZZYIIYXIIIIII",
        "YZZZZXIIYYIIIIII",
        "XZZZZXIIYYIIIIII",
        "YZZZZYIIYYIIIIII",
        "XZZZZYIIYYIIIIII",
        "YZZZZXIIXXIIIIII",
        "XZZZZXIIXXIIIIII",
        "YZZZZYIIXXIIIIII",
        "XZZZZYIIXXIIIIII",
        "YZZZZXIIXYIIIIII",
        "XZZZZXIIXYIIIIII",
        "YZZZZYIIXYIIIIII",
        "XZZZZYIIXYIIIIII"
    ],
    [
        "YZZZZXIIYZZXIIII",
        "XZZZZXIIYZZXIIII",
        "YZZZZYIIYZZXIIII",
        "XZZZZYIIYZZXIIII",
        "YZZZZXIIYZZYIIII",
        "XZZZZXIIYZZYIIII",
        "YZZZZYIIYZZYIIII",
        "XZZZZYIIYZZYIIII",
        "YZZZZXIIXZZXIIII",
        "XZZZZXIIXZZXIIII",
        "YZZZZYIIXZZXIIII",
        "XZZZZYIIXZZXIIII",
        "YZZZZXIIXZZYIIII",
        "XZZZZXIIXZZYIIII",
        "YZZZZYIIXZZYIIII",
        "XZZZZYIIXZZYIIII"
    ],
    [
        "YZZZZXIIYZZZZXII",
        "XZZZZXIIYZZZZXII",
        "YZZZZYIIYZZZZXII",
        "XZZZZYIIYZZZZXII",
        "YZZZZXIIYZZZZYII",
        "XZZZZXIIYZZZZYII",
        "YZZZZYIIYZZZZYII",
        "XZZZZYIIYZZZZYII",
        "YZZZZXIIXZZZZXII",
        "XZZZZXIIXZZZZXII",
        "YZZZZYIIXZZZZXII",
        "XZZZZYIIXZZZZXII",
        "YZZZZXIIXZZZZYII",
        "XZZZZXIIXZZZZYII",
        "YZZZZYIIXZZZZYII",
        "XZZZZYIIXZZZZYII"
    ],
    [
        "YZZZZXIIYZZZZZZX",
        "XZZZZXIIYZZZZZZX",
        "YZZZZYIIYZZZZZZX",
        "XZZZZYIIYZZZZZZX",
        "YZZZZXIIYZZZZZZY",
        "XZZZZXIIYZZZZZZY",
        "YZZZZYIIYZZZZZZY",
        "XZZZZYIIYZZZZZZY",
        "YZZZZXIIXZZZZZZX",
        "XZZZZXIIXZZZZZZX",
        "YZZZZYIIXZZZZZZX",
        "XZZZZYIIXZZZZZZX",
        "YZZZZXIIXZZZZZZY",
        "XZZZZXIIXZZZZZZY",
        "YZZZZYIIXZZZZZZY",
        "XZZZZYIIXZZZZZZY"
    ],
    [
        "YZZZZXIIIYXIIIII",
        "XZZZZXIIIYXIIIII",
        "YZZZZYIIIYXIIIII",
        "XZZZZYIIIYXIIIII",
        "YZZZZXIIIYYIIIII",
        "XZZZZXIIIYYIIIII",
        "YZZZZYIIIYYIIIII",
        "XZZZZYIIIYYIIIII",
        "YZZZZXIIIXXIIIII",
        "XZZZZXIIIXXIIIII",
        "YZZZZYIIIXXIIIII",
        "XZZZZYIIIXXIIIII",
        "YZZZZXIIIXYIIIII",
        "XZZZZXIIIXYIIIII",
        "YZZZZYIIIXYIIIII",
        "XZZZZYIIIXYIIIII"
    ],
    [
        "YZZZZXIIIYZZXIII",
        "XZZZZXIIIYZZXIII",
        "YZZZZYIIIYZZXIII",
        "XZZZZYIIIYZZXIII",
        "YZZZZXIIIYZZYIII",
        "XZZZZXIIIYZZYIII",
        "YZZZZYIIIYZZYIII",
        "XZZZZYIIIYZZYIII",
        "YZZZZXIIIXZZXIII",
        "XZZZZXIIIXZZXIII",
        "YZZZZYIIIXZZXIII",
        "XZZZZYIIIXZZXIII",
        "YZZZZXIIIXZZYIII",
        "XZZZZXIIIXZZYIII",
        "YZZZZYIIIXZZYIII",
        "XZZZZYIIIXZZYIII"
    ],
    [
        "YZZZZXIIIYZZZZXI",
        "XZZZZXIIIYZZZZXI",
        "YZZZZYIIIYZZZZXI",
        "XZZZZYIIIYZZZZXI",
        "YZZZZXIIIYZZZZYI",
        "XZZZZXIIIYZZZZYI",
        "YZZZZYIIIYZZZZYI",
        "XZZZZYIIIYZZZZYI",
        "YZZZZXIIIXZZZZXI",
        "XZZZZXIIIXZZZZXI",
        "YZZZZYIIIXZZZZXI",
        "XZZZZYIIIXZZZZXI",
        "YZZZZXIIIXZZZZYI",
        "XZZZZXIIIXZZZZYI",
        "YZZZZYIIIXZZZZYI",
        "XZZZZYIIIXZZZZYI"
    ],
    [
        "YZZZZXIIIIYXIIII",
        "XZZZZXIIIIYXIIII",
        "YZZZZYIIIIYXIIII",
        "XZZZZYIIIIYXIIII",
        "YZZZZXIIIIYYIIII",
        "XZZZZXIIIIYYIIII",
        "YZZZZYIIIIYYIIII",
        "XZZZZYIIIIYYIIII",
        "YZZZZXIIIIXXIIII",
        "XZZZZXIIIIXXIIII",
        "YZZZZYIIIIXXIIII",
        "XZZZZYIIIIXXIIII",
        "YZZZZXIIIIXYIIII",
        "XZZZZXIIIIXYIIII",
        "YZZZZYIIIIXYIIII",
        "XZZZZYIIIIXYIIII"
    ],
    [
        "YZZZZXIIIIYZZXII",
        "XZZZZXIIIIYZZXII",
        "YZZZZYIIIIYZZXII",
        "XZZZZYIIIIYZZXII",
        "YZZZZXIIIIYZZYII",
        "XZZZZXIIIIYZZYII",
        "YZZZZYIIIIYZZYII",
        "XZZZZYIIIIYZZYII",
        "YZZZZXIIIIXZZXII",
        "XZZZZXIIIIXZZXII",
        "YZZZZYIIIIXZZXII",
        "XZZZZYIIIIXZZXII",
        "YZZZZXIIIIXZZYII",
        "XZZZZXIIIIXZZYII",
        "YZZZZYIIIIXZZYII",
        "XZZZZYIIIIXZZYII"
    ],
    [
        "YZZZZXIIIIYZZZZX",
        "XZZZZXIIIIYZZZZX",
        "YZZZZYIIIIYZZZZX",
        "XZZZZYIIIIYZZZZX",
        "YZZZZXIIIIYZZZZY",
        "XZZZZXIIIIYZZZZY",
        "YZZZZYIIIIYZZZZY",
        "XZZZZYIIIIYZZZZY",
        "YZZZZXIIIIXZZZZX",
        "XZZZZXIIIIXZZZZX",
        "YZZZZYIIIIXZZZZX",
        "XZZZZYIIIIXZZZZX",
        "YZZZZXIIIIXZZZZY",
        "XZZZZXIIIIXZZZZY",
        "YZZZZYIIIIXZZZZY",
        "XZZZZYIIIIXZZZZY"
    ],
    [
        "YZZZZXIIIIIYXIII",
        "XZZZZXIIIIIYXIII",
        "YZZZZYIIIIIYXIII",
        "XZZZZYIIIIIYXIII",
        "YZZZZXIIIIIYYIII",
        "XZZZZXIIIIIYYIII",
        "YZZZZYIIIIIYYIII",
        "XZZZZYIIIIIYYIII",
        "YZZZZXIIIIIXXIII",
        "XZZZZXIIIIIXXIII",
        "YZZZZYIIIIIXXIII",
        "XZZZZYIIIIIXXIII",
        "YZZZZXIIIIIXYIII",
        "XZZZZXIIIIIXYIII",
        "YZZZZYIIIIIXYIII",
        "XZZZZYIIIIIXYIII"
    ],
    [
        "YZZZZXIIIIIYZZXI",
        "XZZZZXIIIIIYZZXI",
        "YZZZZYIIIIIYZZXI",
        "XZZZZYIIIIIYZZXI",
        "YZZZZXIIIIIYZZYI",
        "XZZZZXIIIIIYZZYI",
        "YZZZZYIIIIIYZZYI",
        "XZZZZYIIIIIYZZYI",
        "YZZZZXIIIIIXZZXI",
        "XZZZZXIIIIIXZZXI",
        "YZZZZYIIIIIXZZXI",
        "XZZZZYIIIIIXZZXI",
        "YZZZZXIIIIIXZZYI",
        "XZZZZXIIIIIXZZYI",
        "YZZZZYIIIIIXZZYI",
        "XZZZZYIIIIIXZZYI"
    ],
    [
        "YZZZZXIIIIIIYXII",
        "XZZZZXIIIIIIYXII",
        "YZZZZYIIIIIIYXII",
        "XZZZZYIIIIIIYXII",
        "YZZZZXIIIIIIYYII",
        "XZZZZXIIIIIIYYII",
        "YZZZZYIIIIIIYYII",
        "XZZZZYIIIIIIYYII",
        "YZZZZXIIIIIIXXII",
        "XZZZZXIIIIIIXXII",
        "YZZZZYIIIIIIXXII",
        "XZZZZYIIIIIIXXII",
        "YZZZZXIIIIIIXYII",
        "XZZZZXIIIIIIXYII",
        "YZZZZYIIIIIIXYII",
        "XZZZZYIIIIIIXYII"
    ],
    [
        "YZZZZXIIIIIIYZZX",
        "XZZZZXIIIIIIYZZX",
        "YZZZZYIIIIIIYZZX",
        "XZZZZYIIIIIIYZZX",
        "YZZZZXIIIIIIYZZY",
        "XZZZZXIIIIIIYZZY",
        "YZZZZYIIIIIIYZZY",
        "XZZZZYIIIIIIYZZY",
        "YZZZZXIIIIIIXZZX",
        "XZZZZXIIIIIIXZZX",
        "YZZZZYIIIIIIXZZX",
        "XZZZZYIIIIIIXZZX",
        "YZZZZXIIIIIIXZZY",
        "XZZZZXIIIIIIXZZY",
        "YZZZZYIIIIIIXZZY",
        "XZZZZYIIIIIIXZZY"
    ],
    [
        "YZZZZXIIIIIIIYXI",
        "XZZZZXIIIIIIIYXI",
        "YZZZZYIIIIIIIYXI",
        "XZZZZYIIIIIIIYXI",
        "YZZZZXIIIIIIIYYI",
        "XZZZZXIIIIIIIYYI",
        "YZZZZYIIIIIIIYYI",
        "XZZZZYIIIIIIIYYI",
        "YZZZZXIIIIIIIXXI",
        "XZZZZXIIIIIIIXXI",
        "YZZZZYIIIIIIIXXI",
        "XZZZZYIIIIIIIXXI",
        "YZZZZXIIIIIIIXYI",
        "XZZZZXIIIIIIIXYI",
        "YZZZZYIIIIIIIXYI",
        "XZZZZYIIIIIIIXYI"
    ],
    [
        "YZZZZXIIIIIIIIYX",
        "XZZZZXIIIIIIIIYX",
        "YZZZZYIIIIIIIIYX",
        "XZZZZYIIIIIIIIYX",
        "YZZZZXIIIIIIIIYY",
        "XZZZZXIIIIIIIIYY",
        "YZZZZYIIIIIIIIYY",
        "XZZZZYIIIIIIIIYY",
        "YZZZZXIIIIIIIIXX",
        "XZZZZXIIIIIIIIXX",
        "YZZZZYIIIIIIIIXX",
        "XZZZZYIIIIIIIIXX",
        "YZZZZXIIIIIIIIXY",
        "XZZZZXIIIIIIIIXY",
        "YZZZZYIIIIIIIIXY",
        "XZZZZYIIIIIIIIXY"
    ],
    [
        "YZZZZZXIYZXIIIII",
        "XZZZZZXIYZXIIIII",
        "YZZZZZYIYZXIIIII",
        "XZZZZZYIYZXIIIII",
        "YZZZZZXIYZYIIIII",
        "XZZZZZXIYZYIIIII",
        "YZZZZZYIYZYIIIII",
        "XZZZZZYIYZYIIIII",
        "YZZZZZXIXZXIIIII",
        "XZZZZZXIXZXIIIII",
        "YZZZZZYIXZXIIIII",
        "XZZZZZYIXZXIIIII",
        "YZZZZZXIXZYIIIII",
        "XZZZZZXIXZYIIIII",
        "YZZZZZYIXZYIIIII",
        "XZZZZZYIXZYIIIII"
    ],
    [
        "YZZZZZXIYZZZXIII",
        "XZZZZZXIYZZZXIII",
        "YZZZZZYIYZZZXIII",
        "XZZZZZYIYZZZXIII",
        "YZZZZZXIYZZZYIII",
        "XZZZZZXIYZZZYIII",
        "YZZZZZYIYZZZYIII",
        "XZZZZZYIYZZZYIII",
        "YZZZZZXIXZZZXIII",
        "XZZZZZXIXZZZXIII",
        "YZZZZZYIXZZZXIII",
        "XZZZZZYIXZZZXIII",
        "YZZZZZXIXZZZYIII",
        "XZZZZZXIXZZZYIII",
        "YZZZZZYIXZZZYIII",
        "XZZZZZYIXZZZYIII"
    ],
    [
        "YZZZZZXIYZZZZZXI",
        "XZZZZZXIYZZZZZXI",
        "YZZZZZYIYZZZZZXI",
        "XZZZZZYIYZZZZZXI",
        "YZZZZZXIYZZZZZYI",
        "XZZZZZXIYZZZZZYI",
        "YZZZZZYIYZZZZZYI",
        "XZZZZZYIYZZZZZYI",
        "YZZZZZXIXZZZZZXI",
        "XZZZZZXIXZZZZZXI",
        "YZZZZZYIXZZZZZXI",
        "XZZZZZYIXZZZZZXI",
        "YZZZZZXIXZZZZZYI",
        "XZZZZZXIXZZZZZYI",
        "YZZZZZYIXZZZZZYI",
        "XZZZZZYIXZZZZZYI"
    ],
    [
        "YZZZZZXIIIYZXIII",
        "XZZZZZXIIIYZXIII",
        "YZZZZZYIIIYZXIII",
        "XZZZZZYIIIYZXIII",
        "YZZZZZXIIIYZYIII",
        "XZZZZZXIIIYZYIII",
        "YZZZZZYIIIYZYIII",
        "XZZZZZYIIIYZYIII",
        "YZZZZZXIIIXZXIII",
        "XZZZZZXIIIXZXIII",
        "YZZZZZYIIIXZXIII",
        "XZZZZZYIIIXZXIII",
        "YZZZZZXIIIXZYIII",
        "XZZZZZXIIIXZYIII",
        "YZZZZZYIIIXZYIII",
        "XZZZZZYIIIXZYIII"
    ],
    [
        "YZZZZZXIIIYZZZXI",
        "XZZZZZXIIIYZZZXI",
        "YZZZZZYIIIYZZZXI",
        "XZZZZZYIIIYZZZXI",
        "YZZZZZXIIIYZZZYI",
        "XZZZZZXIIIYZZZYI",
        "YZZZZZYIIIYZZZYI",
        "XZZZZZYIIIYZZZYI",
        "YZZZZZXIIIXZZZXI",
        "XZZZZZXIIIXZZZXI",
        "YZZZZZYIIIXZZZXI",
        "XZZZZZYIIIXZZZXI",
        "YZZZZZXIIIXZZZYI",
        "XZZZZZXIIIXZZZYI",
        "YZZZZZYIIIXZZZYI",
        "XZZZZZYIIIXZZZYI"
    ],
    [
        "YZZZZZXIIIIIYZXI",
        "XZZZZZXIIIIIYZXI",
        "YZZZZZYIIIIIYZXI",
        "XZZZZZYIIIIIYZXI",
        "YZZZZZXIIIIIYZYI",
        "XZZZZZXIIIIIYZYI",
        "YZZZZZYIIIIIYZYI",
        "XZZZZZYIIIIIYZYI",
        "YZZZZZXIIIIIXZXI",
        "XZZZZZXIIIIIXZXI",
        "YZZZZZYIIIIIXZXI",
        "XZZZZZYIIIIIXZXI",
        "YZZZZZXIIIIIXZYI",
        "XZZZZZXIIIIIXZYI",
        "YZZZZZYIIIIIXZYI",
        "XZZZZZYIIIIIXZYI"
    ],
    [
        "YZZZZZZXYXIIIIII",
        "XZZZZZZXYXIIIIII",
        "YZZZZZZYYXIIIIII",
        "XZZZZZZYYXIIIIII",
        "YZZZZZZXYYIIIIII",
        "XZZZZZZXYYIIIIII",
        "YZZZZZZYYYIIIIII",
        "XZZZZZZYYYIIIIII",
        "YZZZZZZXXXIIIIII",
        "XZZZZZZXXXIIIIII",
        "YZZZZZZYXXIIIIII",
        "XZZZZZZYXXIIIIII",
        "YZZZZZZXXYIIIIII",
        "XZZZZZZXXYIIIIII",
        "YZZZZZZYXYIIIIII",
        "XZZZZZZYXYIIIIII"
    ],
    [
        "YZZZZZZXYZZXIIII",
        "XZZZZZZXYZZXIIII",
        "YZZZZZZYYZZXIIII",
        "XZZZZZZYYZZXIIII",
        "YZZZZZZXYZZYIIII",
        "XZZZZZZXYZZYIIII",
        "YZZZZZZYYZZYIIII",
        "XZZZZZZYYZZYIIII",
        "YZZZZZZXXZZXIIII",
        "XZZZZZZXXZZXIIII",
        "YZZZZZZYXZZXIIII",
        "XZZZZZZYXZZXIIII",
        "YZZZZZZXXZZYIIII",
        "XZZZZZZXXZZYIIII",
        "YZZZZZZYXZZYIIII",
        "XZZZZZZYXZZYIIII"
    ],
    [
        "YZZZZZZXYZZZZXII",
        "XZZZZZZXYZZZZXII",
        "YZZZZZZYYZZZZXII",
        "XZZZZZZYYZZZZXII",
        "YZZZZZZXYZZZZYII",
        "XZZZZZZXYZZZZYII",
        "YZZZZZZYYZZZZYII",
        "XZZZZZZYYZZZZYII",
        "YZZZZZZXXZZZZXII",
        "XZZZZZZXXZZZZXII",
        "YZZZZZZYXZZZZXII",
        "XZZZZZZYXZZZZXII",
        "YZZZZZZXXZZZZYII",
        "XZZZZZZXXZZZZYII",
        "YZZZZZZYXZZZZYII",
        "XZZZZZZYXZZZZYII"
    ],
    [
        "YZZZZZZXYZZZZZZX",
        "XZZZZZZXYZZZZZZX",
        "YZZZZZZYYZZZZZZX",
        "XZZZZZZYYZZZZZZX",
        "YZZZZZZXYZZZZZZY",
        "XZZZZZZXYZZZZZZY",
        "YZZZZZZYYZZZZZZY",
        "XZZZZZZYYZZZZZZY",
        "YZZZZZZXXZZZZZZX",
        "XZZZZZZXXZZZZZZX",
        "YZZZZZZYXZZZZZZX",
        "XZZZZZZYXZZZZZZX",
        "YZZZZZZXXZZZZZZY",
        "XZZZZZZXXZZZZZZY",
        "YZZZZZZYXZZZZZZY",
        "XZZZZZZYXZZZZZZY"
    ],
    [
        "YZZZZZZXIYXIIIII",
        "XZZZZZZXIYXIIIII",
        "YZZZZZZYIYXIIIII",
        "XZZZZZZYIYXIIIII",
        "YZZZZZZXIYYIIIII",
        "XZZZZZZXIYYIIIII",
        "YZZZZZZYIYYIIIII",
        "XZZZZZZYIYYIIIII",
        "YZZZZZZXIXXIIIII",
        "XZZZZZZXIXXIIIII",
        "YZZZZZZYIXXIIIII",
        "XZZZZZZYIXXIIIII",
        "YZZZZZZXIXYIIIII",
        "XZZZZZZXIXYIIIII",
        "YZZZZZZYIXYIIIII",
        "XZZZZZZYIXYIIIII"
    ],
    [
        "YZZZZZZXIYZZXIII",
        "XZZZZZZXIYZZXIII",
        "YZZZZZZYIYZZXIII",
        "XZZZZZZYIYZZXIII",
        "YZZZZZZXIYZZYIII",
        "XZZZZZZXIYZZYIII",
        "YZZZZZZYIYZZYIII",
        "XZZZZZZYIYZZYIII",
        "YZZZZZZXIXZZXIII",
        "XZZZZZZXIXZZXIII",
        "YZZZZZZYIXZZXIII",
        "XZZZZZZYIXZZXIII",
        "YZZZZZZXIXZZYIII",
        "XZZZZZZXIXZZYIII",
        "YZZZZZZYIXZZYIII",
        "XZZZZZZYIXZZYIII"
    ],
    [
        "YZZZZZZXIYZZZZXI",
        "XZZZZZZXIYZZZZXI",
        "YZZZZZZYIYZZZZXI",
        "XZZZZZZYIYZZZZXI",
        "YZZZZZZXIYZZZZYI",
        "XZZZZZZXIYZZZZYI",
        "YZZZZZZYIYZZZZYI",
        "XZZZZZZYIYZZZZYI",
        "YZZZZZZXIXZZZZXI",
        "XZZZZZZXIXZZZZXI",
        "YZZZZZZYIXZZZZXI",
        "XZZZZZZYIXZZZZXI",
        "YZZZZZZXIXZZZZYI",
        "XZZZZZZXIXZZZZYI",
        "YZZZZZZYIXZZZZYI",
        "XZZZZZZYIXZZZZYI"
    ],
    [
        "YZZZZZZXIIYXIIII",
        "XZZZZZZXIIYXIIII",
        "YZZZZZZYIIYXIIII",
        "XZZZZZZYIIYXIIII",
        "YZZZZZZXIIYYIIII",
        "XZZZZZZXIIYYIIII",
        "YZZZZZZYIIYYIIII",
        "XZZZZZZYIIYYIIII",
        "YZZZZZZXIIXXIIII",
        "XZZZZZZXIIXXIIII",
        "YZZZZZZYIIXXIIII",
        "XZZZZZZYIIXXIIII",
        "YZZZZZZXIIXYIIII",
        "XZZZZZZXIIXYIIII",
        "YZZZZZZYIIXYIIII",
        "XZZZZZZYIIXYIIII"
    ],
    [
        "YZZZZZZXIIYZZXII",
        "XZZZZZZXIIYZZXII",
        "YZZZZZZYIIYZZXII",
        "XZZZZZZYIIYZZXII",
        "YZZZZZZXIIYZZYII",
        "XZZZZZZXIIYZZYII",
        "YZZZZZZYIIYZZYII",
        "XZZZZZZYIIYZZYII",
        "YZZZZZZXIIXZZXII",
        "XZZZZZZXIIXZZXII",
        "YZZZZZZYIIXZZXII",
        "XZZZZZZYIIXZZXII",
        "YZZZZZZXIIXZZYII",
        "XZZZZZZXIIXZZYII",
        "YZZZZZZYIIXZZYII",
        "XZZZZZZYIIXZZYII"
    ],
    [
        "YZZZZZZXIIYZZZZX",
        "XZZZZZZXIIYZZZZX",
        "YZZZZZZYIIYZZZZX",
        "XZZZZZZYIIYZZZZX",
        "YZZZZZZXIIYZZZZY",
        "XZZZZZZXIIYZZZZY",
        "YZZZZZZYIIYZZZZY",
        "XZZZZZZYIIYZZZZY",
        "YZZZZZZXIIXZZZZX",
        "XZZZZZZXIIXZZZZX",
        "YZZZZZZYIIXZZZZX",
        "XZZZZZZYIIXZZZZX",
        "YZZZZZZXIIXZZZZY",
        "XZZZZZZXIIXZZZZY",
        "YZZZZZZYIIXZZZZY",
        "XZZZZZZYIIXZZZZY"
    ],
    [
        "YZZZZZZXIIIYXIII",
        "XZZZZZZXIIIYXIII",
        "YZZZZZZYIIIYXIII",
        "XZZZZZZYIIIYXIII",
        "YZZZZZZXIIIYYIII",
        "XZZZZZZXIIIYYIII",
        "YZZZZZZYIIIYYIII",
        "XZZZZZZYIIIYYIII",
        "YZZZZZZXIIIXXIII",
        "XZZZZZZXIIIXXIII",
        "YZZZZZZYIIIXXIII",
        "XZZZZZZYIIIXXIII",
        "YZZZZZZXIIIXYIII",
        "XZZZZZZXIIIXYIII",
        "YZZZZZZYIIIXYIII",
        "XZZZZZZYIIIXYIII"
    ],
    [
        "YZZZZZZXIIIYZZXI",
        "XZZZZZZXIIIYZZXI",
        "YZZZZZZYIIIYZZXI",
        "XZZZZZZYIIIYZZXI",
        "YZZZZZZXIIIYZZYI",
        "XZZZZZZXIIIYZZYI",
        "YZZZZZZYIIIYZZYI",
        "XZZZZZZYIIIYZZYI",
        "YZZZZZZXIIIXZZXI",
        "XZZZZZZXIIIXZZXI",
        "YZZZZZZYIIIXZZXI",
        "XZZZZZZYIIIXZZXI",
        "YZZZZZZXIIIXZZYI",
        "XZZZZZZXIIIXZZYI",
        "YZZZZZZYIIIXZZYI",
        "XZZZZZZYIIIXZZYI"
    ],
    [
        "YZZZZZZXIIIIYXII",
        "XZZZZZZXIIIIYXII",
        "YZZZZZZYIIIIYXII",
        "XZZZZZZYIIIIYXII",
        "YZZZZZZXIIIIYYII",
        "XZZZZZZXIIIIYYII",
        "YZZZZZZYIIIIYYII",
        "XZZZZZZYIIIIYYII",
        "YZZZZZZXIIIIXXII",
        "XZZZZZZXIIIIXXII",
        "YZZZZZZYIIIIXXII",
        "XZZZZZZYIIIIXXII",
        "YZZZZZZXIIIIXYII",
        "XZZZZZZXIIIIXYII",
        "YZZZZZZYIIIIXYII",
        "XZZZZZZYIIIIXYII"
    ],
    [
        "YZZZZZZXIIIIYZZX",
        "XZZZZZZXIIIIYZZX",
        "YZZZZZZYIIIIYZZX",
        "XZZZZZZYIIIIYZZX",
        "YZZZZZZXIIIIYZZY",
        "XZZZZZZXIIIIYZZY",
        "YZZZZZZYIIIIYZZY",
        "XZZZZZZYIIIIYZZY",
        "YZZZZZZXIIIIXZZX",
        "XZZZZZZXIIIIXZZX",
        "YZZZZZZYIIIIXZZX",
        "XZZZZZZYIIIIXZZX",
        "YZZZZZZXIIIIXZZY",
        "XZZZZZZXIIIIXZZY",
        "YZZZZZZYIIIIXZZY",
        "XZZZZZZYIIIIXZZY"
    ],
    [
        "YZZZZZZXIIIIIYXI",
        "XZZZZZZXIIIIIYXI",
        "YZZZZZZYIIIIIYXI",
        "XZZZZZZYIIIIIYXI",
        "YZZZZZZXIIIIIYYI",
        "XZZZZZZXIIIIIYYI",
        "YZZZZZZYIIIIIYYI",
        "XZZZZZZYIIIIIYYI",
        "YZZZZZZXIIIIIXXI",
        "XZZZZZZXIIIIIXXI",
        "YZZZZZZYIIIIIXXI",
        "XZZZZZZYIIIIIXXI",
        "YZZZZZZXIIIIIXYI",
        "XZZZZZZXIIIIIXYI",
        "YZZZZZZYIIIIIXYI",
        "XZZZZZZYIIIIIXYI"
    ],
    [
        "YZZZZZZXIIIIIIYX",
        "XZZZZZZXIIIIIIYX",
        "YZZZZZZYIIIIIIYX",
        "XZZZZZZYIIIIIIYX",
        "YZZZZZZXIIIIIIYY",
        "XZZZZZZXIIIIIIYY",
        "YZZZZZZYIIIIIIYY",
        "XZZZZZZYIIIIIIYY",
        "YZZZZZZXIIIIIIXX",
        "XZZZZZZXIIIIIIXX",
        "YZZZZZZYIIIIIIXX",
        "XZZZZZZYIIIIIIXX",
        "YZZZZZZXIIIIIIXY",
        "XZZZZZZXIIIIIIXY",
        "YZZZZZZYIIIIIIXY",
        "XZZZZZZYIIIIIIXY"
    ],
    [
        "IYXIIIIIYXIIIIII",
        "IXXIIIIIYXIIIIII",
        "IYYIIIIIYXIIIIII",
        "IXYIIIIIYXIIIIII",
        "IYXIIIIIYYIIIIII",
        "IXXIIIIIYYIIIIII",
        "IYYIIIIIYYIIIIII",
        "IXYIIIIIYYIIIIII",
        "IYXIIIIIXXIIIIII",
        "IXXIIIIIXXIIIIII",
        "IYYIIIIIXXIIIIII",
        "IXYIIIIIXXIIIIII",
        "IYXIIIIIXYIIIIII",
        "IXXIIIIIXYIIIIII",
        "IYYIIIIIXYIIIIII",
        "IXYIIIIIXYIIIIII"
    ],
    [
        "IYXIIIIIYZZXIIII",
        "IXXIIIIIYZZXIIII",
        "IYYIIIIIYZZXIIII",
        "IXYIIIIIYZZXIIII",
        "IYXIIIIIYZZYIIII",
        "IXXIIIIIYZZYIIII",
        "IYYIIIIIYZZYIIII",
        "IXYIIIIIYZZYIIII",
        "IYXIIIIIXZZXIIII",
        "IXXIIIIIXZZXIIII",
        "IYYIIIIIXZZXIIII",
        "IXYIIIIIXZZXIIII",
        "IYXIIIIIXZZYIIII",
        "IXXIIIIIXZZYIIII",
        "IYYIIIIIXZZYIIII",
        "IXYIIIIIXZZYIIII"
    ],
    [
        "IYXIIIIIYZZZZXII",
        "IXXIIIIIYZZZZXII",
        "IYYIIIIIYZZZZXII",
        "IXYIIIIIYZZZZXII",
        "IYXIIIIIYZZZZYII",
        "IXXIIIIIYZZZZYII",
        "IYYIIIIIYZZZZYII",
        "IXYIIIIIYZZZZYII",
        "IYXIIIIIXZZZZXII",
        "IXXIIIIIXZZZZXII",
        "IYYIIIIIXZZZZXII",
        "IXYIIIIIXZZZZXII",
        "IYXIIIIIXZZZZYII",
        "IXXIIIIIXZZZZYII",
        "IYYIIIIIXZZZZYII",
        "IXYIIIIIXZZZZYII"
    ],
    [
        "IYXIIIIIYZZZZZZX",
        "IXXIIIIIYZZZZZZX",
        "IYYIIIIIYZZZZZZX",
        "IXYIIIIIYZZZZZZX",
        "IYXIIIIIYZZZZZZY",
        "IXXIIIIIYZZZZZZY",
        "IYYIIIIIYZZZZZZY",
        "IXYIIIIIYZZZZZZY",
        "IYXIIIIIXZZZZZZX",
        "IXXIIIIIXZZZZZZX",
        "IYYIIIIIXZZZZZZX",
        "IXYIIIIIXZZZZZZX",
        "IYXIIIIIXZZZZZZY",
        "IXXIIIIIXZZZZZZY",
        "IYYIIIIIXZZZZZZY",
        "IXYIIIIIXZZZZZZY"
    ],
    [
        "IYXIIIIIIYXIIIII",
        "IXXIIIIIIYXIIIII",
        "IYYIIIIIIYXIIIII",
        "IXYIIIIIIYXIIIII",
        "IYXIIIIIIYYIIIII",
        "IXXIIIIIIYYIIIII",
        "IYYIIIIIIYYIIIII",
        "IXYIIIIIIYYIIIII",
        "IYXIIIIIIXXIIIII",
        "IXXIIIIIIXXIIIII",
        "IYYIIIIIIXXIIIII",
        "IXYIIIIIIXXIIIII",
        "IYXIIIIIIXYIIIII",
        "IXXIIIIIIXYIIIII",
        "IYYIIIIIIXYIIIII",
        "IXYIIIIIIXYIIIII"
    ],
    [
        "IYXIIIIIIYZZXIII",
        "IXXIIIIIIYZZXIII",
        "IYYIIIIIIYZZXIII",
        "IXYIIIIIIYZZXIII",
        "IYXIIIIIIYZZYIII",
        "IXXIIIIIIYZZYIII",
        "IYYIIIIIIYZZYIII",
        "IXYIIIIIIYZZYIII",
        "IYXIIIIIIXZZXIII",
        "IXXIIIIIIXZZXIII",
        "IYYIIIIIIXZZXIII",
        "IXYIIIIIIXZZXIII",
        "IYXIIIIIIXZZYIII",
        "IXXIIIIIIXZZYIII",
        "IYYIIIIIIXZZYIII",
        "IXYIIIIIIXZZYIII"
    ],
    [
        "IYXIIIIIIYZZZZXI",
        "IXXIIIIIIYZZZZXI",
        "IYYIIIIIIYZZZZXI",
        "IXYIIIIIIYZZZZXI",
        "IYXIIIIIIYZZZZYI",
        "IXXIIIIIIYZZZZYI",
        "IYYIIIIIIYZZZZYI",
        "IXYIIIIIIYZZZZYI",
        "IYXIIIIIIXZZZZXI",
        "IXXIIIIIIXZZZZXI",
        "IYYIIIIIIXZZZZXI",
        "IXYIIIIIIXZZZZXI",
        "IYXIIIIIIXZZZZYI",
        "IXXIIIIIIXZZZZYI",
        "IYYIIIIIIXZZZZYI",
        "IXYIIIIIIXZZZZYI"
    ],
    [
        "IYXIIIIIIIYXIIII",
        "IXXIIIIIIIYXIIII",
        "IYYIIIIIIIYXIIII",
        "IXYIIIIIIIYXIIII",
        "IYXIIIIIIIYYIIII",
        "IXXIIIIIIIYYIIII",
        "IYYIIIIIIIYYIIII",
        "IXYIIIIIIIYYIIII",
        "IYXIIIIIIIXXIIII",
        "IXXIIIIIIIXXIIII",
        "IYYIIIIIIIXXIIII",
        "IXYIIIIIIIXXIIII",
        "IYXIIIIIIIXYIIII",
        "IXXIIIIIIIXYIIII",
        "IYYIIIIIIIXYIIII",
        "IXYIIIIIIIXYIIII"
    ],
    [
        "IYXIIIIIIIYZZXII",
        "IXXIIIIIIIYZZXII",
        "IYYIIIIIIIYZZXII",
        "IXYIIIIIIIYZZXII",
        "IYXIIIIIIIYZZYII",
        "IXXIIIIIIIYZZYII",
        "IYYIIIIIIIYZZYII",
        "IXYIIIIIIIYZZYII",
        "IYXIIIIIIIXZZXII",
        "IXXIIIIIIIXZZXII",
        "IYYIIIIIIIXZZXII",
        "IXYIIIIIIIXZZXII",
        "IYXIIIIIIIXZZYII",
        "IXXIIIIIIIXZZYII",
        "IYYIIIIIIIXZZYII",
        "IXYIIIIIIIXZZYII"
    ],
    [
        "IYXIIIIIIIYZZZZX",
        "IXXIIIIIIIYZZZZX",
        "IYYIIIIIIIYZZZZX",
        "IXYIIIIIIIYZZZZX",
        "IYXIIIIIIIYZZZZY",
        "IXXIIIIIIIYZZZZY",
        "IYYIIIIIIIYZZZZY",
        "IXYIIIIIIIYZZZZY",
        "IYXIIIIIIIXZZZZX",
        "IXXIIIIIIIXZZZZX",
        "IYYIIIIIIIXZZZZX",
        "IXYIIIIIIIXZZZZX",
        "IYXIIIIIIIXZZZZY",
        "IXXIIIIIIIXZZZZY",
        "IYYIIIIIIIXZZZZY",
        "IXYIIIIIIIXZZZZY"
    ],
    [
        "IYXIIIIIIIIYXIII",
        "IXXIIIIIIIIYXIII",
        "IYYIIIIIIIIYXIII",
        "IXYIIIIIIIIYXIII",
        "IYXIIIIIIIIYYIII",
        "IXXIIIIIIIIYYIII",
        "IYYIIIIIIIIYYIII",
        "IXYIIIIIIIIYYIII",
        "IYXIIIIIIIIXXIII",
        "IXXIIIIIIIIXXIII",
        "IYYIIIIIIIIXXIII",
        "IXYIIIIIIIIXXIII",
        "IYXIIIIIIIIXYIII",
        "IXXIIIIIIIIXYIII",
        "IYYIIIIIIIIXYIII",
        "IXYIIIIIIIIXYIII"
    ],
    [
        "IYXIIIIIIIIYZZXI",
        "IXXIIIIIIIIYZZXI",
        "IYYIIIIIIIIYZZXI",
        "IXYIIIIIIIIYZZXI",
        "IYXIIIIIIIIYZZYI",
        "IXXIIIIIIIIYZZYI",
        "IYYIIIIIIIIYZZYI",
        "IXYIIIIIIIIYZZYI",
        "IYXIIIIIIIIXZZXI",
        "IXXIIIIIIIIXZZXI",
        "IYYIIIIIIIIXZZXI",
        "IXYIIIIIIIIXZZXI",
        "IYXIIIIIIIIXZZYI",
        "IXXIIIIIIIIXZZYI",
        "IYYIIIIIIIIXZZYI",
        "IXYIIIIIIIIXZZYI"
    ],
    [
        "IYXIIIIIIIIIYXII",
        "IXXIIIIIIIIIYXII",
        "IYYIIIIIIIIIYXII",
        "IXYIIIIIIIIIYXII",
        "IYXIIIIIIIIIYYII",
        "IXXIIIIIIIIIYYII",
        "IYYIIIIIIIIIYYII",
        "IXYIIIIIIIIIYYII",
        "IYXIIIIIIIIIXXII",
        "IXXIIIIIIIIIXXII",
        "IYYIIIIIIIIIXXII",
        "IXYIIIIIIIIIXXII",
        "IYXIIIIIIIIIXYII",
        "IXXIIIIIIIIIXYII",
        "IYYIIIIIIIIIXYII",
        "IXYIIIIIIIIIXYII"
    ],
    [
        "IYXIIIIIIIIIYZZX",
        "IXXIIIIIIIIIYZZX",
        "IYYIIIIIIIIIYZZX",
        "IXYIIIIIIIIIYZZX",
        "IYXIIIIIIIIIYZZY",
        "IXXIIIIIIIIIYZZY",
        "IYYIIIIIIIIIYZZY",
        "IXYIIIIIIIIIYZZY",
        "IYXIIIIIIIIIXZZX",
        "IXXIIIIIIIIIXZZX",
        "IYYIIIIIIIIIXZZX",
        "IXYIIIIIIIIIXZZX",
        "IYXIIIIIIIIIXZZY",
        "IXXIIIIIIIIIXZZY",
        "IYYIIIIIIIIIXZZY",
        "IXYIIIIIIIIIXZZY"
    ],
    [
        "IYXIIIIIIIIIIYXI",
        "IXXIIIIIIIIIIYXI",
        "IYYIIIIIIIIIIYXI",
        "IXYIIIIIIIIIIYXI",
        "IYXIIIIIIIIIIYYI",
        "IXXIIIIIIIIIIYYI",
        "IYYIIIIIIIIIIYYI",
        "IXYIIIIIIIIIIYYI",
        "IYXIIIIIIIIIIXXI",
        "IXXIIIIIIIIIIXXI",
        "IYYIIIIIIIIIIXXI",
        "IXYIIIIIIIIIIXXI",
        "IYXIIIIIIIIIIXYI",
        "IXXIIIIIIIIIIXYI",
        "IYYIIIIIIIIIIXYI",
        "IXYIIIIIIIIIIXYI"
    ],
    [
        "IYXIIIIIIIIIIIYX",
        "IXXIIIIIIIIIIIYX",
        "IYYIIIIIIIIIIIYX",
        "IXYIIIIIIIIIIIYX",
        "IYXIIIIIIIIIIIYY",
        "IXXIIIIIIIIIIIYY",
        "IYYIIIIIIIIIIIYY",
        "IXYIIIIIIIIIIIYY",
        "IYXIIIIIIIIIIIXX",
        "IXXIIIIIIIIIIIXX",
        "IYYIIIIIIIIIIIXX",
        "IXYIIIIIIIIIIIXX",
        "IYXIIIIIIIIIIIXY",
        "IXXIIIIIIIIIIIXY",
        "IYYIIIIIIIIIIIXY",
        "IXYIIIIIIIIIIIXY"
    ],
    [
        "IYZXIIIIIYZXIIII",
        "IXZXIIIIIYZXIIII",
        "IYZYIIIIIYZXIIII",
        "IXZYIIIIIYZXIIII",
        "IYZXIIIIIYZYIIII",
        "IXZXIIIIIYZYIIII",
        "IYZYIIIIIYZYIIII",
        "IXZYIIIIIYZYIIII",
        "IYZXIIIIIXZXIIII",
        "IXZXIIIIIXZXIIII",
        "IYZYIIIIIXZXIIII",
        "IXZYIIIIIXZXIIII",
        "IYZXIIIIIXZYIIII",
        "IXZXIIIIIXZYIIII",
        "IYZYIIIIIXZYIIII",
        "IXZYIIIIIXZYIIII"
    ],
    [
        "IYZXIIIIIYZZZXII",
        "IXZXIIIIIYZZZXII",
        "IYZYIIIIIYZZZXII",
        "IXZYIIIIIYZZZXII",
        "IYZXIIIIIYZZZYII",
        "IXZXIIIIIYZZZYII",
        "IYZYIIIIIYZZZYII",
        "IXZYIIIIIYZZZYII",
        "IYZXIIIIIXZZZXII",
        "IXZXIIIIIXZZZXII",
        "IYZYIIIIIXZZZXII",
        "IXZYIIIIIXZZZXII",
        "IYZXIIIIIXZZZYII",
        "IXZXIIIIIXZZZYII",
        "IYZYIIIIIXZZZYII",
        "IXZYIIIIIXZZZYII"
    ],
    [
        "IYZXIIIIIYZZZZZX",
        "IXZXIIIIIYZZZZZX",
        "IYZYIIIIIYZZZZZX",
        "IXZYIIIIIYZZZZZX",
        "IYZXIIIIIYZZZZZY",
        "IXZXIIIIIYZZZZZY",
        "IYZYIIIIIYZZZZZY",
        "IXZYIIIIIYZZZZZY",
        "IYZXIIIIIXZZZZZX",
        "IXZXIIIIIXZZZZZX",
        "IYZYIIIIIXZZZZZX",
        "IXZYIIIIIXZZZZZX",
        "IYZXIIIIIXZZZZZY",
        "IXZXIIIIIXZZZZZY",
        "IYZYIIIIIXZZZZZY",
        "IXZYIIIIIXZZZZZY"
    ],
    [
        "IYZXIIIIIIIYZXII",
        "IXZXIIIIIIIYZXII",
        "IYZYIIIIIIIYZXII",
        "IXZYIIIIIIIYZXII",
        "IYZXIIIIIIIYZYII",
        "IXZXIIIIIIIYZYII",
        "IYZYIIIIIIIYZYII",
        "IXZYIIIIIIIYZYII",
        "IYZXIIIIIIIXZXII",
        "IXZXIIIIIIIXZXII",
        "IYZYIIIIIIIXZXII",
        "IXZYIIIIIIIXZXII",
        "IYZXIIIIIIIXZYII",
        "IXZXIIIIIIIXZYII",
        "IYZYIIIIIIIXZYII",
        "IXZYIIIIIIIXZYII"
    ],
    [
        "IYZXIIIIIIIYZZZX",
        "IXZXIIIIIIIYZZZX",
        "IYZYIIIIIIIYZZZX",
        "IXZYIIIIIIIYZZZX",
        "IYZXIIIIIIIYZZZY",
        "IXZXIIIIIIIYZZZY",
        "IYZYIIIIIIIYZZZY",
        "IXZYIIIIIIIYZZZY",
        "IYZXIIIIIIIXZZZX",
        "IXZXIIIIIIIXZZZX",
        "IYZYIIIIIIIXZZZX",
        "IXZYIIIIIIIXZZZX",
        "IYZXIIIIIIIXZZZY",
        "IXZXIIIIIIIXZZZY",
        "IYZYIIIIIIIXZZZY",
        "IXZYIIIIIIIXZZZY"
    ],
    [
        "IYZXIIIIIIIIIYZX",
        "IXZXIIIIIIIIIYZX",
        "IYZYIIIIIIIIIYZX",
        "IXZYIIIIIIIIIYZX",
        "IYZXIIIIIIIIIYZY",
        "IXZXIIIIIIIIIYZY",
        "IYZYIIIIIIIIIYZY",
        "IXZYIIIIIIIIIYZY",
        "IYZXIIIIIIIIIXZX",
        "IXZXIIIIIIIIIXZX",
        "IYZYIIIIIIIIIXZX",
        "IXZYIIIIIIIIIXZX",
        "IYZXIIIIIIIIIXZY",
        "IXZXIIIIIIIIIXZY",
        "IYZYIIIIIIIIIXZY",
        "IXZYIIIIIIIIIXZY"
    ],
    [
        "IYZZXIIIYXIIIIII",
        "IXZZXIIIYXIIIIII",
        "IYZZYIIIYXIIIIII",
        "IXZZYIIIYXIIIIII",
        "IYZZXIIIYYIIIIII",
        "IXZZXIIIYYIIIIII",
        "IYZZYIIIYYIIIIII",
        "IXZZYIIIYYIIIIII",
        "IYZZXIIIXXIIIIII",
        "IXZZXIIIXXIIIIII",
        "IYZZYIIIXXIIIIII",
        "IXZZYIIIXXIIIIII",
        "IYZZXIIIXYIIIIII",
        "IXZZXIIIXYIIIIII",
        "IYZZYIIIXYIIIIII",
        "IXZZYIIIXYIIIIII"
    ],
    [
        "IYZZXIIIYZZXIIII",
        "IXZZXIIIYZZXIIII",
        "IYZZYIIIYZZXIIII",
        "IXZZYIIIYZZXIIII",
        "IYZZXIIIYZZYIIII",
        "IXZZXIIIYZZYIIII",
        "IYZZYIIIYZZYIIII",
        "IXZZYIIIYZZYIIII",
        "IYZZXIIIXZZXIIII",
        "IXZZXIIIXZZXIIII",
        "IYZZYIIIXZZXIIII",
        "IXZZYIIIXZZXIIII",
        "IYZZXIIIXZZYIIII",
        "IXZZXIIIXZZYIIII",
        "IYZZYIIIXZZYIIII",
        "IXZZYIIIXZZYIIII"
    ],
    [
        "IYZZXIIIYZZZZXII",
        "IXZZXIIIYZZZZXII",
        "IYZZYIIIYZZZZXII",
        "IXZZYIIIYZZZZXII",
        "IYZZXIIIYZZZZYII",
        "IXZZXIIIYZZZZYII",
        "IYZZYIIIYZZZZYII",
        "IXZZYIIIYZZZZYII",
        "IYZZXIIIXZZZZXII",
        "IXZZXIIIXZZZZXII",
        "IYZZYIIIXZZZZXII",
        "IXZZYIIIXZZZZXII",
        "IYZZXIIIXZZZZYII",
        "IXZZXIIIXZZZZYII",
        "IYZZYIIIXZZZZYII",
        "IXZZYIIIXZZZZYII"
    ],
    [
        "IYZZXIIIYZZZZZZX",
        "IXZZXIIIYZZZZZZX",
        "IYZZYIIIYZZZZZZX",
        "IXZZYIIIYZZZZZZX",
        "IYZZXIIIYZZZZZZY",
        "IXZZXIIIYZZZZZZY",
        "IYZZYIIIYZZZZZZY",
        "IXZZYIIIYZZZZZZY",
        "IYZZXIIIXZZZZZZX",
        "IXZZXIIIXZZZZZZX",
        "IYZZYIIIXZZZZZZX",
        "IXZZYIIIXZZZZZZX",
        "IYZZXIIIXZZZZZZY",
        "IXZZXIIIXZZZZZZY",
        "IYZZYIIIXZZZZZZY",
        "IXZZYIIIXZZZZZZY"
    ],
    [
        "IYZZXIIIIYXIIIII",
        "IXZZXIIIIYXIIIII",
        "IYZZYIIIIYXIIIII",
        "IXZZYIIIIYXIIIII",
        "IYZZXIIIIYYIIIII",
        "IXZZXIIIIYYIIIII",
        "IYZZYIIIIYYIIIII",
        "IXZZYIIIIYYIIIII",
        "IYZZXIIIIXXIIIII",
        "IXZZXIIIIXXIIIII",
        "IYZZYIIIIXXIIIII",
        "IXZZYIIIIXXIIIII",
        "IYZZXIIIIXYIIIII",
        "IXZZXIIIIXYIIIII",
        "IYZZYIIIIXYIIIII",
        "IXZZYIIIIXYIIIII"
    ],
    [
        "IYZZXIIIIYZZXIII",
        "IXZZXIIIIYZZXIII",
        "IYZZYIIIIYZZXIII",
        "IXZZYIIIIYZZXIII",
        "IYZZXIIIIYZZYIII",
        "IXZZXIIIIYZZYIII",
        "IYZZYIIIIYZZYIII",
        "IXZZYIIIIYZZYIII",
        "IYZZXIIIIXZZXIII",
        "IXZZXIIIIXZZXIII",
        "IYZZYIIIIXZZXIII",
        "IXZZYIIIIXZZXIII",
        "IYZZXIIIIXZZYIII",
        "IXZZXIIIIXZZYIII",
        "IYZZYIIIIXZZYIII",
        "IXZZYIIIIXZZYIII"
    ],
    [
        "IYZZXIIIIYZZZZXI",
        "IXZZXIIIIYZZZZXI",
        "IYZZYIIIIYZZZZXI",
        "IXZZYIIIIYZZZZXI",
        "IYZZXIIIIYZZZZYI",
        "IXZZXIIIIYZZZZYI",
        "IYZZYIIIIYZZZZYI",
        "IXZZYIIIIYZZZZYI",
        "IYZZXIIIIXZZZZXI",
        "IXZZXIIIIXZZZZXI",
        "IYZZYIIIIXZZZZXI",
        "IXZZYIIIIXZZZZXI",
        "IYZZXIIIIXZZZZYI",
        "IXZZXIIIIXZZZZYI",
        "IYZZYIIIIXZZZZYI",
        "IXZZYIIIIXZZZZYI"
    ],
    [
        "IYZZXIIIIIYXIIII",
        "IXZZXIIIIIYXIIII",
        "IYZZYIIIIIYXIIII",
        "IXZZYIIIIIYXIIII",
        "IYZZXIIIIIYYIIII",
        "IXZZXIIIIIYYIIII",
        "IYZZYIIIIIYYIIII",
        "IXZZYIIIIIYYIIII",
        "IYZZXIIIIIXXIIII",
        "IXZZXIIIIIXXIIII",
        "IYZZYIIIIIXXIIII",
        "IXZZYIIIIIXXIIII",
        "IYZZXIIIIIXYIIII",
        "IXZZXIIIIIXYIIII",
        "IYZZYIIIIIXYIIII",
        "IXZZYIIIIIXYIIII"
    ],
    [
        "IYZZXIIIIIYZZXII",
        "IXZZXIIIIIYZZXII",
        "IYZZYIIIIIYZZXII",
        "IXZZYIIIIIYZZXII",
        "IYZZXIIIIIYZZYII",
        "IXZZXIIIIIYZZYII",
        "IYZZYIIIIIYZZYII",
        "IXZZYIIIIIYZZYII",
        "IYZZXIIIIIXZZXII",
        "IXZZXIIIIIXZZXII",
        "IYZZYIIIIIXZZXII",
        "IXZZYIIIIIXZZXII",
        "IYZZXIIIIIXZZYII",
        "IXZZXIIIIIXZZYII",
        "IYZZYIIIIIXZZYII",
        "IXZZYIIIIIXZZYII"
    ],
    [
        "IYZZXIIIIIYZZZZX",
        "IXZZXIIIIIYZZZZX",
        "IYZZYIIIIIYZZZZX",
        "IXZZYIIIIIYZZZZX",
        "IYZZXIIIIIYZZZZY",
        "IXZZXIIIIIYZZZZY",
        "IYZZYIIIIIYZZZZY",
        "IXZZYIIIIIYZZZZY",
        "IYZZXIIIIIXZZZZX",
        "IXZZXIIIIIXZZZZX",
        "IYZZYIIIIIXZZZZX",
        "IXZZYIIIIIXZZZZX",
        "IYZZXIIIIIXZZZZY",
        "IXZZXIIIIIXZZZZY",
        "IYZZYIIIIIXZZZZY",
        "IXZZYIIIIIXZZZZY"
    ],
    [
        "IYZZXIIIIIIYXIII",
        "IXZZXIIIIIIYXIII",
        "IYZZYIIIIIIYXIII",
        "IXZZYIIIIIIYXIII",
        "IYZZXIIIIIIYYIII",
        "IXZZXIIIIIIYYIII",
        "IYZZYIIIIIIYYIII",
        "IXZZYIIIIIIYYIII",
        "IYZZXIIIIIIXXIII",
        "IXZZXIIIIIIXXIII",
        "IYZZYIIIIIIXXIII",
        "IXZZYIIIIIIXXIII",
        "IYZZXIIIIIIXYIII",
        "IXZZXIIIIIIXYIII",
        "IYZZYIIIIIIXYIII",
        "IXZZYIIIIIIXYIII"
    ],
    [
        "IYZZXIIIIIIYZZXI",
        "IXZZXIIIIIIYZZXI",
        "IYZZYIIIIIIYZZXI",
        "IXZZYIIIIIIYZZXI",
        "IYZZXIIIIIIYZZYI",
        "IXZZXIIIIIIYZZYI",
        "IYZZYIIIIIIYZZYI",
        "IXZZYIIIIIIYZZYI",
        "IYZZXIIIIIIXZZXI",
        "IXZZXIIIIIIXZZXI",
        "IYZZYIIIIIIXZZXI",
        "IXZZYIIIIIIXZZXI",
        "IYZZXIIIIIIXZZYI",
        "IXZZXIIIIIIXZZYI",
        "IYZZYIIIIIIXZZYI",
        "IXZZYIIIIIIXZZYI"
    ],
    [
        "IYZZXIIIIIIIYXII",
        "IXZZXIIIIIIIYXII",
        "IYZZYIIIIIIIYXII",
        "IXZZYIIIIIIIYXII",
        "IYZZXIIIIIIIYYII",
        "IXZZXIIIIIIIYYII",
        "IYZZYIIIIIIIYYII",
        "IXZZYIIIIIIIYYII",
        "IYZZXIIIIIIIXXII",
        "IXZZXIIIIIIIXXII",
        "IYZZYIIIIIIIXXII",
        "IXZZYIIIIIIIXXII",
        "IYZZXIIIIIIIXYII",
        "IXZZXIIIIIIIXYII",
        "IYZZYIIIIIIIXYII",
        "IXZZYIIIIIIIXYII"
    ],
    [
        "IYZZXIIIIIIIYZZX",
        "IXZZXIIIIIIIYZZX",
        "IYZZYIIIIIIIYZZX",
        "IXZZYIIIIIIIYZZX",
        "IYZZXIIIIIIIYZZY",
        "IXZZXIIIIIIIYZZY",
        "IYZZYIIIIIIIYZZY",
        "IXZZYIIIIIIIYZZY",
        "IYZZXIIIIIIIXZZX",
        "IXZZXIIIIIIIXZZX",
        "IYZZYIIIIIIIXZZX",
        "IXZZYIIIIIIIXZZX",
        "IYZZXIIIIIIIXZZY",
        "IXZZXIIIIIIIXZZY",
        "IYZZYIIIIIIIXZZY",
        "IXZZYIIIIIIIXZZY"
    ],
    [
        "IYZZXIIIIIIIIYXI",
        "IXZZXIIIIIIIIYXI",
        "IYZZYIIIIIIIIYXI",
        "IXZZYIIIIIIIIYXI",
        "IYZZXIIIIIIIIYYI",
        "IXZZXIIIIIIIIYYI",
        "IYZZYIIIIIIIIYYI",
        "IXZZYIIIIIIIIYYI",
        "IYZZXIIIIIIIIXXI",
        "IXZZXIIIIIIIIXXI",
        "IYZZYIIIIIIIIXXI",
        "IXZZYIIIIIIIIXXI",
        "IYZZXIIIIIIIIXYI",
        "IXZZXIIIIIIIIXYI",
        "IYZZYIIIIIIIIXYI",
        "IXZZYIIIIIIIIXYI"
    ],
    [
        "IYZZXIIIIIIIIIYX",
        "IXZZXIIIIIIIIIYX",
        "IYZZYIIIIIIIIIYX",
        "IXZZYIIIIIIIIIYX",
        "IYZZXIIIIIIIIIYY",
        "IXZZXIIIIIIIIIYY",
        "IYZZYIIIIIIIIIYY",
        "IXZZYIIIIIIIIIYY",
        "IYZZXIIIIIIIIIXX",
        "IXZZXIIIIIIIIIXX",
        "IYZZYIIIIIIIIIXX",
        "IXZZYIIIIIIIIIXX",
        "IYZZXIIIIIIIIIXY",
        "IXZZXIIIIIIIIIXY",
        "IYZZYIIIIIIIIIXY",
        "IXZZYIIIIIIIIIXY"
    ],
    [
        "IYZZZXIIIYZXIIII",
        "IXZZZXIIIYZXIIII",
        "IYZZZYIIIYZXIIII",
        "IXZZZYIIIYZXIIII",
        "IYZZZXIIIYZYIIII",
        "IXZZZXIIIYZYIIII",
        "IYZZZYIIIYZYIIII",
        "IXZZZYIIIYZYIIII",
        "IYZZZXIIIXZXIIII",
        "IXZZZXIIIXZXIIII",
        "IYZZZYIIIXZXIIII",
        "IXZZZYIIIXZXIIII",
        "IYZZZXIIIXZYIIII",
        "IXZZZXIIIXZYIIII",
        "IYZZZYIIIXZYIIII",
        "IXZZZYIIIXZYIIII"
    ],
    [
        "IYZZZXIIIYZZZXII",
        "IXZZZXIIIYZZZXII",
        "IYZZZYIIIYZZZXII",
        "IXZZZYIIIYZZZXII",
        "IYZZZXIIIYZZZYII",
        "IXZZZXIIIYZZZYII",
        "IYZZZYIIIYZZZYII",
        "IXZZZYIIIYZZZYII",
        "IYZZZXIIIXZZZXII",
        "IXZZZXIIIXZZZXII",
        "IYZZZYIIIXZZZXII",
        "IXZZZYIIIXZZZXII",
        "IYZZZXIIIXZZZYII",
        "IXZZZXIIIXZZZYII",
        "IYZZZYIIIXZZZYII",
        "IXZZZYIIIXZZZYII"
    ],
    [
        "IYZZZXIIIYZZZZZX",
        "IXZZZXIIIYZZZZZX",
        "IYZZZYIIIYZZZZZX",
        "IXZZZYIIIYZZZZZX",
        "IYZZZXIIIYZZZZZY",
        "IXZZZXIIIYZZZZZY",
        "IYZZZYIIIYZZZZZY",
        "IXZZZYIIIYZZZZZY",
        "IYZZZXIIIXZZZZZX",
        "IXZZZXIIIXZZZZZX",
        "IYZZZYIIIXZZZZZX",
        "IXZZZYIIIXZZZZZX",
        "IYZZZXIIIXZZZZZY",
        "IXZZZXIIIXZZZZZY",
        "IYZZZYIIIXZZZZZY",
        "IXZZZYIIIXZZZZZY"
    ],
    [
        "IYZZZXIIIIIYZXII",
        "IXZZZXIIIIIYZXII",
        "IYZZZYIIIIIYZXII",
        "IXZZZYIIIIIYZXII",
        "IYZZZXIIIIIYZYII",
        "IXZZZXIIIIIYZYII",
        "IYZZZYIIIIIYZYII",
        "IXZZZYIIIIIYZYII",
        "IYZZZXIIIIIXZXII",
        "IXZZZXIIIIIXZXII",
        "IYZZZYIIIIIXZXII",
        "IXZZZYIIIIIXZXII",
        "IYZZZXIIIIIXZYII",
        "IXZZZXIIIIIXZYII",
        "IYZZZYIIIIIXZYII",
        "IXZZZYIIIIIXZYII"
    ],
    [
        "IYZZZXIIIIIYZZZX",
        "IXZZZXIIIIIYZZZX",
        "IYZZZYIIIIIYZZZX",
        "IXZZZYIIIIIYZZZX",
        "IYZZZXIIIIIYZZZY",
        "IXZZZXIIIIIYZZZY",
        "IYZZZYIIIIIYZZZY",
        "IXZZZYIIIIIYZZZY",
        "IYZZZXIIIIIXZZZX",
        "IXZZZXIIIIIXZZZX",
        "IYZZZYIIIIIXZZZX",
        "IXZZZYIIIIIXZZZX",
        "IYZZZXIIIIIXZZZY",
        "IXZZZXIIIIIXZZZY",
        "IYZZZYIIIIIXZZZY",
        "IXZZZYIIIIIXZZZY"
    ],
    [
        "IYZZZXIIIIIIIYZX",
        "IXZZZXIIIIIIIYZX",
        "IYZZZYIIIIIIIYZX",
        "IXZZZYIIIIIIIYZX",
        "IYZZZXIIIIIIIYZY",
        "IXZZZXIIIIIIIYZY",
        "IYZZZYIIIIIIIYZY",
        "IXZZZYIIIIIIIYZY",
        "IYZZZXIIIIIIIXZX",
        "IXZZZXIIIIIIIXZX",
        "IYZZZYIIIIIIIXZX",
        "IXZZZYIIIIIIIXZX",
        "IYZZZXIIIIIIIXZY",
        "IXZZZXIIIIIIIXZY",
        "IYZZZYIIIIIIIXZY",
        "IXZZZYIIIIIIIXZY"
    ],
    [
        "IYZZZZXIYXIIIIII",
        "IXZZZZXIYXIIIIII",
        "IYZZZZYIYXIIIIII",
        "IXZZZZYIYXIIIIII",
        "IYZZZZXIYYIIIIII",
        "IXZZZZXIYYIIIIII",
        "IYZZZZYIYYIIIIII",
        "IXZZZZYIYYIIIIII",
        "IYZZZZXIXXIIIIII",
        "IXZZZZXIXXIIIIII",
        "IYZZZZYIXXIIIIII",
        "IXZZZZYIXXIIIIII",
        "IYZZZZXIXYIIIIII",
        "IXZZZZXIXYIIIIII",
        "IYZZZZYIXYIIIIII",
        "IXZZZZYIXYIIIIII"
    ],
    [
        "IYZZZZXIYZZXIIII",
        "IXZZZZXIYZZXIIII",
        "IYZZZZYIYZZXIIII",
        "IXZZZZYIYZZXIIII",
        "IYZZZZXIYZZYIIII",
        "IXZZZZXIYZZYIIII",
        "IYZZZZYIYZZYIIII",
        "IXZZZZYIYZZYIIII",
        "IYZZZZXIXZZXIIII",
        "IXZZZZXIXZZXIIII",
        "IYZZZZYIXZZXIIII",
        "IXZZZZYIXZZXIIII",
        "IYZZZZXIXZZYIIII",
        "IXZZZZXIXZZYIIII",
        "IYZZZZYIXZZYIIII",
        "IXZZZZYIXZZYIIII"
    ],
    [
        "IYZZZZXIYZZZZXII",
        "IXZZZZXIYZZZZXII",
        "IYZZZZYIYZZZZXII",
        "IXZZZZYIYZZZZXII",
        "IYZZZZXIYZZZZYII",
        "IXZZZZXIYZZZZYII",
        "IYZZZZYIYZZZZYII",
        "IXZZZZYIYZZZZYII",
        "IYZZZZXIXZZZZXII",
        "IXZZZZXIXZZZZXII",
        "IYZZZZYIXZZZZXII",
        "IXZZZZYIXZZZZXII",
        "IYZZZZXIXZZZZYII",
        "IXZZZZXIXZZZZYII",
        "IYZZZZYIXZZZZYII",
        "IXZZZZYIXZZZZYII"
    ],
    [
        "IYZZZZXIYZZZZZZX",
        "IXZZZZXIYZZZZZZX",
        "IYZZZZYIYZZZZZZX",
        "IXZZZZYIYZZZZZZX",
        "IYZZZZXIYZZZZZZY",
        "IXZZZZXIYZZZZZZY",
        "IYZZZZYIYZZZZZZY",
        "IXZZZZYIYZZZZZZY",
        "IYZZZZXIXZZZZZZX",
        "IXZZZZXIXZZZZZZX",
        "IYZZZZYIXZZZZZZX",
        "IXZZZZYIXZZZZZZX",
        "IYZZZZXIXZZZZZZY",
        "IXZZZZXIXZZZZZZY",
        "IYZZZZYIXZZZZZZY",
        "IXZZZZYIXZZZZZZY"
    ],
    [
        "IYZZZZXIIYXIIIII",
        "IXZZZZXIIYXIIIII",
        "IYZZZZYIIYXIIIII",
        "IXZZZZYIIYXIIIII",
        "IYZZZZXIIYYIIIII",
        "IXZZZZXIIYYIIIII",
        "IYZZZZYIIYYIIIII",
        "IXZZZZYIIYYIIIII",
        "IYZZZZXIIXXIIIII",
        "IXZZZZXIIXXIIIII",
        "IYZZZZYIIXXIIIII",
        "IXZZZZYIIXXIIIII",
        "IYZZZZXIIXYIIIII",
        "IXZZZZXIIXYIIIII",
        "IYZZZZYIIXYIIIII",
        "IXZZZZYIIXYIIIII"
    ],
    [
        "IYZZZZXIIYZZXIII",
        "IXZZZZXIIYZZXIII",
        "IYZZZZYIIYZZXIII",
        "IXZZZZYIIYZZXIII",
        "IYZZZZXIIYZZYIII",
        "IXZZZZXIIYZZYIII",
        "IYZZZZYIIYZZYIII",
        "IXZZZZYIIYZZYIII",
        "IYZZZZXIIXZZXIII",
        "IXZZZZXIIXZZXIII",
        "IYZZZZYIIXZZXIII",
        "IXZZZZYIIXZZXIII",
        "IYZZZZXIIXZZYIII",
        "IXZZZZXIIXZZYIII",
        "IYZZZZYIIXZZYIII",
        "IXZZZZYIIXZZYIII"
    ],
    [
        "IYZZZZXIIYZZZZXI",
        "IXZZZZXIIYZZZZXI",
        "IYZZZZYIIYZZZZXI",
        "IXZZZZYIIYZZZZXI",
        "IYZZZZXIIYZZZZYI",
        "IXZZZZXIIYZZZZYI",
        "IYZZZZYIIYZZZZYI",
        "IXZZZZYIIYZZZZYI",
        "IYZZZZXIIXZZZZXI",
        "IXZZZZXIIXZZZZXI",
        "IYZZZZYIIXZZZZXI",
        "IXZZZZYIIXZZZZXI",
        "IYZZZZXIIXZZZZYI",
        "IXZZZZXIIXZZZZYI",
        "IYZZZZYIIXZZZZYI",
        "IXZZZZYIIXZZZZYI"
    ],
    [
        "IYZZZZXIIIYXIIII",
        "IXZZZZXIIIYXIIII",
        "IYZZZZYIIIYXIIII",
        "IXZZZZYIIIYXIIII",
        "IYZZZZXIIIYYIIII",
        "IXZZZZXIIIYYIIII",
        "IYZZZZYIIIYYIIII",
        "IXZZZZYIIIYYIIII",
        "IYZZZZXIIIXXIIII",
        "IXZZZZXIIIXXIIII",
        "IYZZZZYIIIXXIIII",
        "IXZZZZYIIIXXIIII",
        "IYZZZZXIIIXYIIII",
        "IXZZZZXIIIXYIIII",
        "IYZZZZYIIIXYIIII",
        "IXZZZZYIIIXYIIII"
    ],
    [
        "IYZZZZXIIIYZZXII",
        "IXZZZZXIIIYZZXII",
        "IYZZZZYIIIYZZXII",
        "IXZZZZYIIIYZZXII",
        "IYZZZZXIIIYZZYII",
        "IXZZZZXIIIYZZYII",
        "IYZZZZYIIIYZZYII",
        "IXZZZZYIIIYZZYII",
        "IYZZZZXIIIXZZXII",
        "IXZZZZXIIIXZZXII",
        "IYZZZZYIIIXZZXII",
        "IXZZZZYIIIXZZXII",
        "IYZZZZXIIIXZZYII",
        "IXZZZZXIIIXZZYII",
        "IYZZZZYIIIXZZYII",
        "IXZZZZYIIIXZZYII"
    ],
    [
        "IYZZZZXIIIYZZZZX",
        "IXZZZZXIIIYZZZZX",
        "IYZZZZYIIIYZZZZX",
        "IXZZZZYIIIYZZZZX",
        "IYZZZZXIIIYZZZZY",
        "IXZZZZXIIIYZZZZY",
        "IYZZZZYIIIYZZZZY",
        "IXZZZZYIIIYZZZZY",
        "IYZZZZXIIIXZZZZX",
        "IXZZZZXIIIXZZZZX",
        "IYZZZZYIIIXZZZZX",
        "IXZZZZYIIIXZZZZX",
        "IYZZZZXIIIXZZZZY",
        "IXZZZZXIIIXZZZZY",
        "IYZZZZYIIIXZZZZY",
        "IXZZZZYIIIXZZZZY"
    ],
    [
        "IYZZZZXIIIIYXIII",
        "IXZZZZXIIIIYXIII",
        "IYZZZZYIIIIYXIII",
        "IXZZZZYIIIIYXIII",
        "IYZZZZXIIIIYYIII",
        "IXZZZZXIIIIYYIII",
        "IYZZZZYIIIIYYIII",
        "IXZZZZYIIIIYYIII",
        "IYZZZZXIIIIXXIII",
        "IXZZZZXIIIIXXIII",
        "IYZZZZYIIIIXXIII",
        "IXZZZZYIIIIXXIII",
        "IYZZZZXIIIIXYIII",
        "IXZZZZXIIIIXYIII",
        "IYZZZZYIIIIXYIII",
        "IXZZZZYIIIIXYIII"
    ],
    [
        "IYZZZZXIIIIYZZXI",
        "IXZZZZXIIIIYZZXI",
        "IYZZZZYIIIIYZZXI",
        "IXZZZZYIIIIYZZXI",
        "IYZZZZXIIIIYZZYI",
        "IXZZZZXIIIIYZZYI",
        "IYZZZZYIIIIYZZYI",
        "IXZZZZYIIIIYZZYI",
        "IYZZZZXIIIIXZZXI",
        "IXZZZZXIIIIXZZXI",
        "IYZZZZYIIIIXZZXI",
        "IXZZZZYIIIIXZZXI",
        "IYZZZZXIIIIXZZYI",
        "IXZZZZXIIIIXZZYI",
        "IYZZZZYIIIIXZZYI",
        "IXZZZZYIIIIXZZYI"
    ],
    [
        "IYZZZZXIIIIIYXII",
        "IXZZZZXIIIIIYXII",
        "IYZZZZYIIIIIYXII",
        "IXZZZZYIIIIIYXII",
        "IYZZZZXIIIIIYYII",
        "IXZZZZXIIIIIYYII",
        "IYZZZZYIIIIIYYII",
        "IXZZZZYIIIIIYYII",
        "IYZZZZXIIIIIXXII",
        "IXZZZZXIIIIIXXII",
        "IYZZZZYIIIIIXXII",
        "IXZZZZYIIIIIXXII",
        "IYZZZZXIIIIIXYII",
        "IXZZZZXIIIIIXYII",
        "IYZZZZYIIIIIXYII",
        "IXZZZZYIIIIIXYII"
    ],
    [
        "IYZZZZXIIIIIYZZX",
        "IXZZZZXIIIIIYZZX",
        "IYZZZZYIIIIIYZZX",
        "IXZZZZYIIIIIYZZX",
        "IYZZZZXIIIIIYZZY",
        "IXZZZZXIIIIIYZZY",
        "IYZZZZYIIIIIYZZY",
        "IXZZZZYIIIIIYZZY",
        "IYZZZZXIIIIIXZZX",
        "IXZZZZXIIIIIXZZX",
        "IYZZZZYIIIIIXZZX",
        "IXZZZZYIIIIIXZZX",
        "IYZZZZXIIIIIXZZY",
        "IXZZZZXIIIIIXZZY",
        "IYZZZZYIIIIIXZZY",
        "IXZZZZYIIIIIXZZY"
    ],
    [
        "IYZZZZXIIIIIIYXI",
        "IXZZZZXIIIIIIYXI",
        "IYZZZZYIIIIIIYXI",
        "IXZZZZYIIIIIIYXI",
        "IYZZZZXIIIIIIYYI",
        "IXZZZZXIIIIIIYYI",
        "IYZZZZYIIIIIIYYI",
        "IXZZZZYIIIIIIYYI",
        "IYZZZZXIIIIIIXXI",
        "IXZZZZXIIIIIIXXI",
        "IYZZZZYIIIIIIXXI",
        "IXZZZZYIIIIIIXXI",
        "IYZZZZXIIIIIIXYI",
        "IXZZZZXIIIIIIXYI",
        "IYZZZZYIIIIIIXYI",
        "IXZZZZYIIIIIIXYI"
    ],
    [
        "IYZZZZXIIIIIIIYX",
        "IXZZZZXIIIIIIIYX",
        "IYZZZZYIIIIIIIYX",
        "IXZZZZYIIIIIIIYX",
        "IYZZZZXIIIIIIIYY",
        "IXZZZZXIIIIIIIYY",
        "IYZZZZYIIIIIIIYY",
        "IXZZZZYIIIIIIIYY",
        "IYZZZZXIIIIIIIXX",
        "IXZZZZXIIIIIIIXX",
        "IYZZZZYIIIIIIIXX",
        "IXZZZZYIIIIIIIXX",
        "IYZZZZXIIIIIIIXY",
        "IXZZZZXIIIIIIIXY",
        "IYZZZZYIIIIIIIXY",
        "IXZZZZYIIIIIIIXY"
    ],
    [
        "IYZZZZZXIYZXIIII",
        "IXZZZZZXIYZXIIII",
        "IYZZZZZYIYZXIIII",
        "IXZZZZZYIYZXIIII",
        "IYZZZZZXIYZYIIII",
        "IXZZZZZXIYZYIIII",
        "IYZZZZZYIYZYIIII",
        "IXZZZZZYIYZYIIII",
        "IYZZZZZXIXZXIIII",
        "IXZZZZZXIXZXIIII",
        "IYZZZZZYIXZXIIII",
        "IXZZZZZYIXZXIIII",
        "IYZZZZZXIXZYIIII",
        "IXZZZZZXIXZYIIII",
        "IYZZZZZYIXZYIIII",
        "IXZZZZZYIXZYIIII"
    ],
    [
        "IYZZZZZXIYZZZXII",
        "IXZZZZZXIYZZZXII",
        "IYZZZZZYIYZZZXII",
        "IXZZZZZYIYZZZXII",
        "IYZZZZZXIYZZZYII",
        "IXZZZZZXIYZZZYII",
        "IYZZZZZYIYZZZYII",
        "IXZZZZZYIYZZZYII",
        "IYZZZZZXIXZZZXII",
        "IXZZZZZXIXZZZXII",
        "IYZZZZZYIXZZZXII",
        "IXZZZZZYIXZZZXII",
        "IYZZZZZXIXZZZYII",
        "IXZZZZZXIXZZZYII",
        "IYZZZZZYIXZZZYII",
        "IXZZZZZYIXZZZYII"
    ],
    [
        "IYZZZZZXIYZZZZZX",
        "IXZZZZZXIYZZZZZX",
        "IYZZZZZYIYZZZZZX",
        "IXZZZZZYIYZZZZZX",
        "IYZZZZZXIYZZZZZY",
        "IXZZZZZXIYZZZZZY",
        "IYZZZZZYIYZZZZZY",
        "IXZZZZZYIYZZZZZY",
        "IYZZZZZXIXZZZZZX",
        "IXZZZZZXIXZZZZZX",
        "IYZZZZZYIXZZZZZX",
        "IXZZZZZYIXZZZZZX",
        "IYZZZZZXIXZZZZZY",
        "IXZZZZZXIXZZZZZY",
        "IYZZZZZYIXZZZZZY",
        "IXZZZZZYIXZZZZZY"
    ],
    [
        "IYZZZZZXIIIYZXII",
        "IXZZZZZXIIIYZXII",
        "IYZZZZZYIIIYZXII",
        "IXZZZZZYIIIYZXII",
        "IYZZZZZXIIIYZYII",
        "IXZZZZZXIIIYZYII",
        "IYZZZZZYIIIYZYII",
        "IXZZZZZYIIIYZYII",
        "IYZZZZZXIIIXZXII",
        "IXZZZZZXIIIXZXII",
        "IYZZZZZYIIIXZXII",
        "IXZZZZZYIIIXZXII",
        "IYZZZZZXIIIXZYII",
        "IXZZZZZXIIIXZYII",
        "IYZZZZZYIIIXZYII",
        "IXZZZZZYIIIXZYII"
    ],
    [
        "IYZZZZZXIIIYZZZX",
        "IXZZZZZXIIIYZZZX",
        "IYZZZZZYIIIYZZZX",
        "IXZZZZZYIIIYZZZX",
        "IYZZZZZXIIIYZZZY",
        "IXZZZZZXIIIYZZZY",
        "IYZZZZZYIIIYZZZY",
        "IXZZZZZYIIIYZZZY",
        "IYZZZZZXIIIXZZZX",
        "IXZZZZZXIIIXZZZX",
        "IYZZZZZYIIIXZZZX",
        "IXZZZZZYIIIXZZZX",
        "IYZZZZZXIIIXZZZY",
        "IXZZZZZXIIIXZZZY",
        "IYZZZZZYIIIXZZZY",
        "IXZZZZZYIIIXZZZY"
    ],
    [
        "IYZZZZZXIIIIIYZX",
        "IXZZZZZXIIIIIYZX",
        "IYZZZZZYIIIIIYZX",
        "IXZZZZZYIIIIIYZX",
        "IYZZZZZXIIIIIYZY",
        "IXZZZZZXIIIIIYZY",
        "IYZZZZZYIIIIIYZY",
        "IXZZZZZYIIIIIYZY",
        "IYZZZZZXIIIIIXZX",
        "IXZZZZZXIIIIIXZX",
        "IYZZZZZYIIIIIXZX",
        "IXZZZZZYIIIIIXZX",
        "IYZZZZZXIIIIIXZY",
        "IXZZZZZXIIIIIXZY",
        "IYZZZZZYIIIIIXZY",
        "IXZZZZZYIIIIIXZY"
    ],
    [
        "IIYXIIIIYXIIIIII",
        "IIXXIIIIYXIIIIII",
        "IIYYIIIIYXIIIIII",
        "IIXYIIIIYXIIIIII",
        "IIYXIIIIYYIIIIII",
        "IIXXIIIIYYIIIIII",
        "IIYYIIIIYYIIIIII",
        "IIXYIIIIYYIIIIII",
        "IIYXIIIIXXIIIIII",
        "IIXXIIIIXXIIIIII",
        "IIYYIIIIXXIIIIII",
        "IIXYIIIIXXIIIIII",
        "IIYXIIIIXYIIIIII",
        "IIXXIIIIXYIIIIII",
        "IIYYIIIIXYIIIIII",
        "IIXYIIIIXYIIIIII"
    ],
    [
        "IIYXIIIIYZZXIIII",
        "IIXXIIIIYZZXIIII",
        "IIYYIIIIYZZXIIII",
        "IIXYIIIIYZZXIIII",
        "IIYXIIIIYZZYIIII",
        "IIXXIIIIYZZYIIII",
        "IIYYIIIIYZZYIIII",
        "IIXYIIIIYZZYIIII",
        "IIYXIIIIXZZXIIII",
        "IIXXIIIIXZZXIIII",
        "IIYYIIIIXZZXIIII",
        "IIXYIIIIXZZXIIII",
        "IIYXIIIIXZZYIIII",
        "IIXXIIIIXZZYIIII",
        "IIYYIIIIXZZYIIII",
        "IIXYIIIIXZZYIIII"
    ],
    [
        "IIYXIIIIYZZZZXII",
        "IIXXIIIIYZZZZXII",
        "IIYYIIIIYZZZZXII",
        "IIXYIIIIYZZZZXII",
        "IIYXIIIIYZZZZYII",
        "IIXXIIIIYZZZZYII",
        "IIYYIIIIYZZZZYII",
        "IIXYIIIIYZZZZYII",
        "IIYXIIIIXZZZZXII",
        "IIXXIIIIXZZZZXII",
        "IIYYIIIIXZZZZXII",
        "IIXYIIIIXZZZZXII",
        "IIYXIIIIXZZZZYII",
        "IIXXIIIIXZZZZYII",
        "IIYYIIIIXZZZZYII",
        "IIXYIIIIXZZZZYII"
    ],
    [
        "IIYXIIIIYZZZZZZX",
        "IIXXIIIIYZZZZZZX",
        "IIYYIIIIYZZZZZZX",
        "IIXYIIIIYZZZZZZX",
        "IIYXIIIIYZZZZZZY",
        "IIXXIIIIYZZZZZZY",
        "IIYYIIIIYZZZZZZY",
        "IIXYIIIIYZZZZZZY",
        "IIYXIIIIXZZZZZZX",
        "IIXXIIIIXZZZZZZX",
        "IIYYIIIIXZZZZZZX",
        "IIXYIIIIXZZZZZZX",
        "IIYXIIIIXZZZZZZY",
        "IIXXIIIIXZZZZZZY",
        "IIYYIIIIXZZZZZZY",
        "IIXYIIIIXZZZZZZY"
    ],
    [
        "IIYXIIIIIYXIIIII",
        "IIXXIIIIIYXIIIII",
        "IIYYIIIIIYXIIIII",
        "IIXYIIIIIYXIIIII",
        "IIYXIIIIIYYIIIII",
        "IIXXIIIIIYYIIIII",
        "IIYYIIIIIYYIIIII",
        "IIXYIIIIIYYIIIII",
        "IIYXIIIIIXXIIIII",
        "IIXXIIIIIXXIIIII",
        "IIYYIIIIIXXIIIII",
        "IIXYIIIIIXXIIIII",
        "IIYXIIIIIXYIIIII",
        "IIXXIIIIIXYIIIII",
        "IIYYIIIIIXYIIIII",
        "IIXYIIIIIXYIIIII"
    ],
    [
        "IIYXIIIIIYZZXIII",
        "IIXXIIIIIYZZXIII",
        "IIYYIIIIIYZZXIII",
        "IIXYIIIIIYZZXIII",
        "IIYXIIIIIYZZYIII",
        "IIXXIIIIIYZZYIII",
        "IIYYIIIIIYZZYIII",
        "IIXYIIIIIYZZYIII",
        "IIYXIIIIIXZZXIII",
        "IIXXIIIIIXZZXIII",
        "IIYYIIIIIXZZXIII",
        "IIXYIIIIIXZZXIII",
        "IIYXIIIIIXZZYIII",
        "IIXXIIIIIXZZYIII",
        "IIYYIIIIIXZZYIII",
        "IIXYIIIIIXZZYIII"
    ],
    [
        "IIYXIIIIIYZZZZXI",
        "IIXXIIIIIYZZZZXI",
        "IIYYIIIIIYZZZZXI",
        "IIXYIIIIIYZZZZXI",
        "IIYXIIIIIYZZZZYI",
        "IIXXIIIIIYZZZZYI",
        "IIYYIIIIIYZZZZYI",
        "IIXYIIIIIYZZZZYI",
        "IIYXIIIIIXZZZZXI",
        "IIXXIIIIIXZZZZXI",
        "IIYYIIIIIXZZZZXI",
        "IIXYIIIIIXZZZZXI",
        "IIYXIIIIIXZZZZYI",
        "IIXXIIIIIXZZZZYI",
        "IIYYIIIIIXZZZZYI",
        "IIXYIIIIIXZZZZYI"
    ],
    [
        "IIYXIIIIIIYXIIII",
        "IIXXIIIIIIYXIIII",
        "IIYYIIIIIIYXIIII",
        "IIXYIIIIIIYXIIII",
        "IIYXIIIIIIYYIIII",
        "IIXXIIIIIIYYIIII",
        "IIYYIIIIIIYYIIII",
        "IIXYIIIIIIYYIIII",
        "IIYXIIIIIIXXIIII",
        "IIXXIIIIIIXXIIII",
        "IIYYIIIIIIXXIIII",
        "IIXYIIIIIIXXIIII",
        "IIYXIIIIIIXYIIII",
        "IIXXIIIIIIXYIIII",
        "IIYYIIIIIIXYIIII",
        "IIXYIIIIIIXYIIII"
    ],
    [
        "IIYXIIIIIIYZZXII",
        "IIXXIIIIIIYZZXII",
        "IIYYIIIIIIYZZXII",
        "IIXYIIIIIIYZZXII",
        "IIYXIIIIIIYZZYII",
        "IIXXIIIIIIYZZYII",
        "IIYYIIIIIIYZZYII",
        "IIXYIIIIIIYZZYII",
        "IIYXIIIIIIXZZXII",
        "IIXXIIIIIIXZZXII",
        "IIYYIIIIIIXZZXII",
        "IIXYIIIIIIXZZXII",
        "IIYXIIIIIIXZZYII",
        "IIXXIIIIIIXZZYII",
        "IIYYIIIIIIXZZYII",
        "IIXYIIIIIIXZZYII"
    ],
    [
        "IIYXIIIIIIYZZZZX",
        "IIXXIIIIIIYZZZZX",
        "IIYYIIIIIIYZZZZX",
        "IIXYIIIIIIYZZZZX",
        "IIYXIIIIIIYZZZZY",
        "IIXXIIIIIIYZZZZY",
        "IIYYIIIIIIYZZZZY",
        "IIXYIIIIIIYZZZZY",
        "IIYXIIIIIIXZZZZX",
        "IIXXIIIIIIXZZZZX",
        "IIYYIIIIIIXZZZZX",
        "IIXYIIIIIIXZZZZX",
        "IIYXIIIIIIXZZZZY",
        "IIXXIIIIIIXZZZZY",
        "IIYYIIIIIIXZZZZY",
        "IIXYIIIIIIXZZZZY"
    ],
    [
        "IIYXIIIIIIIYXIII",
        "IIXXIIIIIIIYXIII",
        "IIYYIIIIIIIYXIII",
        "IIXYIIIIIIIYXIII",
        "IIYXIIIIIIIYYIII",
        "IIXXIIIIIIIYYIII",
        "IIYYIIIIIIIYYIII",
        "IIXYIIIIIIIYYIII",
        "IIYXIIIIIIIXXIII",
        "IIXXIIIIIIIXXIII",
        "IIYYIIIIIIIXXIII",
        "IIXYIIIIIIIXXIII",
        "IIYXIIIIIIIXYIII",
        "IIXXIIIIIIIXYIII",
        "IIYYIIIIIIIXYIII",
        "IIXYIIIIIIIXYIII"
    ],
    [
        "IIYXIIIIIIIYZZXI",
        "IIXXIIIIIIIYZZXI",
        "IIYYIIIIIIIYZZXI",
        "IIXYIIIIIIIYZZXI",
        "IIYXIIIIIIIYZZYI",
        "IIXXIIIIIIIYZZYI",
        "IIYYIIIIIIIYZZYI",
        "IIXYIIIIIIIYZZYI",
        "IIYXIIIIIIIXZZXI",
        "IIXXIIIIIIIXZZXI",
        "IIYYIIIIIIIXZZXI",
        "IIXYIIIIIIIXZZXI",
        "IIYXIIIIIIIXZZYI",
        "IIXXIIIIIIIXZZYI",
        "IIYYIIIIIIIXZZYI",
        "IIXYIIIIIIIXZZYI"
    ],
    [
        "IIYXIIIIIIIIYXII",
        "IIXXIIIIIIIIYXII",
        "IIYYIIIIIIIIYXII",
        "IIXYIIIIIIIIYXII",
        "IIYXIIIIIIIIYYII",
        "IIXXIIIIIIIIYYII",
        "IIYYIIIIIIIIYYII",
        "IIXYIIIIIIIIYYII",
        "IIYXIIIIIIIIXXII",
        "IIXXIIIIIIIIXXII",
        "IIYYIIIIIIIIXXII",
        "IIXYIIIIIIIIXXII",
        "IIYXIIIIIIIIXYII",
        "IIXXIIIIIIIIXYII",
        "IIYYIIIIIIIIXYII",
        "IIXYIIIIIIIIXYII"
    ],
    [
        "IIYXIIIIIIIIYZZX",
        "IIXXIIIIIIIIYZZX",
        "IIYYIIIIIIIIYZZX",
        "IIXYIIIIIIIIYZZX",
        "IIYXIIIIIIIIYZZY",
        "IIXXIIIIIIIIYZZY",
        "IIYYIIIIIIIIYZZY",
        "IIXYIIIIIIIIYZZY",
        "IIYXIIIIIIIIXZZX",
        "IIXXIIIIIIIIXZZX",
        "IIYYIIIIIIIIXZZX",
        "IIXYIIIIIIIIXZZX",
        "IIYXIIIIIIIIXZZY",
        "IIXXIIIIIIIIXZZY",
        "IIYYIIIIIIIIXZZY",
        "IIXYIIIIIIIIXZZY"
    ],
    [
        "IIYXIIIIIIIIIYXI",
        "IIXXIIIIIIIIIYXI",
        "IIYYIIIIIIIIIYXI",
        "IIXYIIIIIIIIIYXI",
        "IIYXIIIIIIIIIYYI",
        "IIXXIIIIIIIIIYYI",
        "IIYYIIIIIIIIIYYI",
        "IIXYIIIIIIIIIYYI",
        "IIYXIIIIIIIIIXXI",
        "IIXXIIIIIIIIIXXI",
        "IIYYIIIIIIIIIXXI",
        "IIXYIIIIIIIIIXXI",
        "IIYXIIIIIIIIIXYI",
        "IIXXIIIIIIIIIXYI",
        "IIYYIIIIIIIIIXYI",
        "IIXYIIIIIIIIIXYI"
    ],
    [
        "IIYXIIIIIIIIIIYX",
        "IIXXIIIIIIIIIIYX",
        "IIYYIIIIIIIIIIYX",
        "IIXYIIIIIIIIIIYX",
        "IIYXIIIIIIIIIIYY",
        "IIXXIIIIIIIIIIYY",
        "IIYYIIIIIIIIIIYY",
        "IIXYIIIIIIIIIIYY",
        "IIYXIIIIIIIIIIXX",
        "IIXXIIIIIIIIIIXX",
        "IIYYIIIIIIIIIIXX",
        "IIXYIIIIIIIIIIXX",
        "IIYXIIIIIIIIIIXY",
        "IIXXIIIIIIIIIIXY",
        "IIYYIIIIIIIIIIXY",
        "IIXYIIIIIIIIIIXY"
    ],
    [
        "IIYZXIIIYZXIIIII",
        "IIXZXIIIYZXIIIII",
        "IIYZYIIIYZXIIIII",
        "IIXZYIIIYZXIIIII",
        "IIYZXIIIYZYIIIII",
        "IIXZXIIIYZYIIIII",
        "IIYZYIIIYZYIIIII",
        "IIXZYIIIYZYIIIII",
        "IIYZXIIIXZXIIIII",
        "IIXZXIIIXZXIIIII",
        "IIYZYIIIXZXIIIII",
        "IIXZYIIIXZXIIIII",
        "IIYZXIIIXZYIIIII",
        "IIXZXIIIXZYIIIII",
        "IIYZYIIIXZYIIIII",
        "IIXZYIIIXZYIIIII"
    ],
    [
        "IIYZXIIIYZZZXIII",
        "IIXZXIIIYZZZXIII",
        "IIYZYIIIYZZZXIII",
        "IIXZYIIIYZZZXIII",
        "IIYZXIIIYZZZYIII",
        "IIXZXIIIYZZZYIII",
        "IIYZYIIIYZZZYIII",
        "IIXZYIIIYZZZYIII",
        "IIYZXIIIXZZZXIII",
        "IIXZXIIIXZZZXIII",
        "IIYZYIIIXZZZXIII",
        "IIXZYIIIXZZZXIII",
        "IIYZXIIIXZZZYIII",
        "IIXZXIIIXZZZYIII",
        "IIYZYIIIXZZZYIII",
        "IIXZYIIIXZZZYIII"
    ],
    [
        "IIYZXIIIYZZZZZXI",
        "IIXZXIIIYZZZZZXI",
        "IIYZYIIIYZZZZZXI",
        "IIXZYIIIYZZZZZXI",
        "IIYZXIIIYZZZZZYI",
        "IIXZXIIIYZZZZZYI",
        "IIYZYIIIYZZZZZYI",
        "IIXZYIIIYZZZZZYI",
        "IIYZXIIIXZZZZZXI",
        "IIXZXIIIXZZZZZXI",
        "IIYZYIIIXZZZZZXI",
        "IIXZYIIIXZZZZZXI",
        "IIYZXIIIXZZZZZYI",
        "IIXZXIIIXZZZZZYI",
        "IIYZYIIIXZZZZZYI",
        "IIXZYIIIXZZZZZYI"
    ],
    [
        "IIYZXIIIIIYZXIII",
        "IIXZXIIIIIYZXIII",
        "IIYZYIIIIIYZXIII",
        "IIXZYIIIIIYZXIII",
        "IIYZXIIIIIYZYIII",
        "IIXZXIIIIIYZYIII",
        "IIYZYIIIIIYZYIII",
        "IIXZYIIIIIYZYIII",
        "IIYZXIIIIIXZXIII",
        "IIXZXIIIIIXZXIII",
        "IIYZYIIIIIXZXIII",
        "IIXZYIIIIIXZXIII",
        "IIYZXIIIIIXZYIII",
        "IIXZXIIIIIXZYIII",
        "IIYZYIIIIIXZYIII",
        "IIXZYIIIIIXZYIII"
    ],
    [
        "IIYZXIIIIIYZZZXI",
        "IIXZXIIIIIYZZZXI",
        "IIYZYIIIIIYZZZXI",
        "IIXZYIIIIIYZZZXI",
        "IIYZXIIIIIYZZZYI",
        "IIXZXIIIIIYZZZYI",
        "IIYZYIIIIIYZZZYI",
        "IIXZYIIIIIYZZZYI",
        "IIYZXIIIIIXZZZXI",
        "IIXZXIIIIIXZZZXI",
        "IIYZYIIIIIXZZZXI",
        "IIXZYIIIIIXZZZXI",
        "IIYZXIIIIIXZZZYI",
        "IIXZXIIIIIXZZZYI",
        "IIYZYIIIIIXZZZYI",
        "IIXZYIIIIIXZZZYI"
    ],
    [
        "IIYZXIIIIIIIYZXI",
        "IIXZXIIIIIIIYZXI",
        "IIYZYIIIIIIIYZXI",
        "IIXZYIIIIIIIYZXI",
        "IIYZXIIIIIIIYZYI",
        "IIXZXIIIIIIIYZYI",
        "IIYZYIIIIIIIYZYI",
        "IIXZYIIIIIIIYZYI",
        "IIYZXIIIIIIIXZXI",
        "IIXZXIIIIIIIXZXI",
        "IIYZYIIIIIIIXZXI",
        "IIXZYIIIIIIIXZXI",
        "IIYZXIIIIIIIXZYI",
        "IIXZXIIIIIIIXZYI",
        "IIYZYIIIIIIIXZYI",
        "IIXZYIIIIIIIXZYI"
    ],
    [
        "IIYZZXIIYXIIIIII",
        "IIXZZXIIYXIIIIII",
        "IIYZZYIIYXIIIIII",
        "IIXZZYIIYXIIIIII",
        "IIYZZXIIYYIIIIII",
        "IIXZZXIIYYIIIIII",
        "IIYZZYIIYYIIIIII",
        "IIXZZYIIYYIIIIII",
        "IIYZZXIIXXIIIIII",
        "IIXZZXIIXXIIIIII",
        "IIYZZYIIXXIIIIII",
        "IIXZZYIIXXIIIIII",
        "IIYZZXIIXYIIIIII",
        "IIXZZXIIXYIIIIII",
        "IIYZZYIIXYIIIIII",
        "IIXZZYIIXYIIIIII"
    ],
    [
        "IIYZZXIIYZZXIIII",
        "IIXZZXIIYZZXIIII",
        "IIYZZYIIYZZXIIII",
        "IIXZZYIIYZZXIIII",
        "IIYZZXIIYZZYIIII",
        "IIXZZXIIYZZYIIII",
        "IIYZZYIIYZZYIIII",
        "IIXZZYIIYZZYIIII",
        "IIYZZXIIXZZXIIII",
        "IIXZZXIIXZZXIIII",
        "IIYZZYIIXZZXIIII",
        "IIXZZYIIXZZXIIII",
        "IIYZZXIIXZZYIIII",
        "IIXZZXIIXZZYIIII",
        "IIYZZYIIXZZYIIII",
        "IIXZZYIIXZZYIIII"
    ],
    [
        "IIYZZXIIYZZZZXII",
        "IIXZZXIIYZZZZXII",
        "IIYZZYIIYZZZZXII",
        "IIXZZYIIYZZZZXII",
        "IIYZZXIIYZZZZYII",
        "IIXZZXIIYZZZZYII",
        "IIYZZYIIYZZZZYII",
        "IIXZZYIIYZZZZYII",
        "IIYZZXIIXZZZZXII",
        "IIXZZXIIXZZZZXII",
        "IIYZZYIIXZZZZXII",
        "IIXZZYIIXZZZZXII",
        "IIYZZXIIXZZZZYII",
        "IIXZZXIIXZZZZYII",
        "IIYZZYIIXZZZZYII",
        "IIXZZYIIXZZZZYII"
    ],
    [
        "IIYZZXIIYZZZZZZX",
        "IIXZZXIIYZZZZZZX",
        "IIYZZYIIYZZZZZZX",
        "IIXZZYIIYZZZZZZX",
        "IIYZZXIIYZZZZZZY",
        "IIXZZXIIYZZZZZZY",
        "IIYZZYIIYZZZZZZY",
        "IIXZZYIIYZZZZZZY",
        "IIYZZXIIXZZZZZZX",
        "IIXZZXIIXZZZZZZX",
        "IIYZZYIIXZZZZZZX",
        "IIXZZYIIXZZZZZZX",
        "IIYZZXIIXZZZZZZY",
        "IIXZZXIIXZZZZZZY",
        "IIYZZYIIXZZZZZZY",
        "IIXZZYIIXZZZZZZY"
    ],
    [
        "IIYZZXIIIYXIIIII",
        "IIXZZXIIIYXIIIII",
        "IIYZZYIIIYXIIIII",
        "IIXZZYIIIYXIIIII",
        "IIYZZXIIIYYIIIII",
        "IIXZZXIIIYYIIIII",
        "IIYZZYIIIYYIIIII",
        "IIXZZYIIIYYIIIII",
        "IIYZZXIIIXXIIIII",
        "IIXZZXIIIXXIIIII",
        "IIYZZYIIIXXIIIII",
        "IIXZZYIIIXXIIIII",
        "IIYZZXIIIXYIIIII",
        "IIXZZXIIIXYIIIII",
        "IIYZZYIIIXYIIIII",
        "IIXZZYIIIXYIIIII"
    ],
    [
        "IIYZZXIIIYZZXIII",
        "IIXZZXIIIYZZXIII",
        "IIYZZYIIIYZZXIII",
        "IIXZZYIIIYZZXIII",
        "IIYZZXIIIYZZYIII",
        "IIXZZXIIIYZZYIII",
        "IIYZZYIIIYZZYIII",
        "IIXZZYIIIYZZYIII",
        "IIYZZXIIIXZZXIII",
        "IIXZZXIIIXZZXIII",
        "IIYZZYIIIXZZXIII",
        "IIXZZYIIIXZZXIII",
        "IIYZZXIIIXZZYIII",
        "IIXZZXIIIXZZYIII",
        "IIYZZYIIIXZZYIII",
        "IIXZZYIIIXZZYIII"
    ],
    [
        "IIYZZXIIIYZZZZXI",
        "IIXZZXIIIYZZZZXI",
        "IIYZZYIIIYZZZZXI",
        "IIXZZYIIIYZZZZXI",
        "IIYZZXIIIYZZZZYI",
        "IIXZZXIIIYZZZZYI",
        "IIYZZYIIIYZZZZYI",
        "IIXZZYIIIYZZZZYI",
        "IIYZZXIIIXZZZZXI",
        "IIXZZXIIIXZZZZXI",
        "IIYZZYIIIXZZZZXI",
        "IIXZZYIIIXZZZZXI",
        "IIYZZXIIIXZZZZYI",
        "IIXZZXIIIXZZZZYI",
        "IIYZZYIIIXZZZZYI",
        "IIXZZYIIIXZZZZYI"
    ],
    [
        "IIYZZXIIIIYXIIII",
        "IIXZZXIIIIYXIIII",
        "IIYZZYIIIIYXIIII",
        "IIXZZYIIIIYXIIII",
        "IIYZZXIIIIYYIIII",
        "IIXZZXIIIIYYIIII",
        "IIYZZYIIIIYYIIII",
        "IIXZZYIIIIYYIIII",
        "IIYZZXIIIIXXIIII",
        "IIXZZXIIIIXXIIII",
        "IIYZZYIIIIXXIIII",
        "IIXZZYIIIIXXIIII",
        "IIYZZXIIIIXYIIII",
        "IIXZZXIIIIXYIIII",
        "IIYZZYIIIIXYIIII",
        "IIXZZYIIIIXYIIII"
    ],
    [
        "IIYZZXIIIIYZZXII",
        "IIXZZXIIIIYZZXII",
        "IIYZZYIIIIYZZXII",
        "IIXZZYIIIIYZZXII",
        "IIYZZXIIIIYZZYII",
        "IIXZZXIIIIYZZYII",
        "IIYZZYIIIIYZZYII",
        "IIXZZYIIIIYZZYII",
        "IIYZZXIIIIXZZXII",
        "IIXZZXIIIIXZZXII",
        "IIYZZYIIIIXZZXII",
        "IIXZZYIIIIXZZXII",
        "IIYZZXIIIIXZZYII",
        "IIXZZXIIIIXZZYII",
        "IIYZZYIIIIXZZYII",
        "IIXZZYIIIIXZZYII"
    ],
    [
        "IIYZZXIIIIYZZZZX",
        "IIXZZXIIIIYZZZZX",
        "IIYZZYIIIIYZZZZX",
        "IIXZZYIIIIYZZZZX",
        "IIYZZXIIIIYZZZZY",
        "IIXZZXIIIIYZZZZY",
        "IIYZZYIIIIYZZZZY",
        "IIXZZYIIIIYZZZZY",
        "IIYZZXIIIIXZZZZX",
        "IIXZZXIIIIXZZZZX",
        "IIYZZYIIIIXZZZZX",
        "IIXZZYIIIIXZZZZX",
        "IIYZZXIIIIXZZZZY",
        "IIXZZXIIIIXZZZZY",
        "IIYZZYIIIIXZZZZY",
        "IIXZZYIIIIXZZZZY"
    ],
    [
        "IIYZZXIIIIIYXIII",
        "IIXZZXIIIIIYXIII",
        "IIYZZYIIIIIYXIII",
        "IIXZZYIIIIIYXIII",
        "IIYZZXIIIIIYYIII",
        "IIXZZXIIIIIYYIII",
        "IIYZZYIIIIIYYIII",
        "IIXZZYIIIIIYYIII",
        "IIYZZXIIIIIXXIII",
        "IIXZZXIIIIIXXIII",
        "IIYZZYIIIIIXXIII",
        "IIXZZYIIIIIXXIII",
        "IIYZZXIIIIIXYIII",
        "IIXZZXIIIIIXYIII",
        "IIYZZYIIIIIXYIII",
        "IIXZZYIIIIIXYIII"
    ],
    [
        "IIYZZXIIIIIYZZXI",
        "IIXZZXIIIIIYZZXI",
        "IIYZZYIIIIIYZZXI",
        "IIXZZYIIIIIYZZXI",
        "IIYZZXIIIIIYZZYI",
        "IIXZZXIIIIIYZZYI",
        "IIYZZYIIIIIYZZYI",
        "IIXZZYIIIIIYZZYI",
        "IIYZZXIIIIIXZZXI",
        "IIXZZXIIIIIXZZXI",
        "IIYZZYIIIIIXZZXI",
        "IIXZZYIIIIIXZZXI",
        "IIYZZXIIIIIXZZYI",
        "IIXZZXIIIIIXZZYI",
        "IIYZZYIIIIIXZZYI",
        "IIXZZYIIIIIXZZYI"
    ],
    [
        "IIYZZXIIIIIIYXII",
        "IIXZZXIIIIIIYXII",
        "IIYZZYIIIIIIYXII",
        "IIXZZYIIIIIIYXII",
        "IIYZZXIIIIIIYYII",
        "IIXZZXIIIIIIYYII",
        "IIYZZYIIIIIIYYII",
        "IIXZZYIIIIIIYYII",
        "IIYZZXIIIIIIXXII",
        "IIXZZXIIIIIIXXII",
        "IIYZZYIIIIIIXXII",
        "IIXZZYIIIIIIXXII",
        "IIYZZXIIIIIIXYII",
        "IIXZZXIIIIIIXYII",
        "IIYZZYIIIIIIXYII",
        "IIXZZYIIIIIIXYII"
    ],
    [
        "IIYZZXIIIIIIYZZX",
        "IIXZZXIIIIIIYZZX",
        "IIYZZYIIIIIIYZZX",
        "IIXZZYIIIIIIYZZX",
        "IIYZZXIIIIIIYZZY",
        "IIXZZXIIIIIIYZZY",
        "IIYZZYIIIIIIYZZY",
        "IIXZZYIIIIIIYZZY",
        "IIYZZXIIIIIIXZZX",
        "IIXZZXIIIIIIXZZX",
        "IIYZZYIIIIIIXZZX",
        "IIXZZYIIIIIIXZZX",
        "IIYZZXIIIIIIXZZY",
        "IIXZZXIIIIIIXZZY",
        "IIYZZYIIIIIIXZZY",
        "IIXZZYIIIIIIXZZY"
    ],
    [
        "IIYZZXIIIIIIIYXI",
        "IIXZZXIIIIIIIYXI",
        "IIYZZYIIIIIIIYXI",
        "IIXZZYIIIIIIIYXI",
        "IIYZZXIIIIIIIYYI",
        "IIXZZXIIIIIIIYYI",
        "IIYZZYIIIIIIIYYI",
        "IIXZZYIIIIIIIYYI",
        "IIYZZXIIIIIIIXXI",
        "IIXZZXIIIIIIIXXI",
        "IIYZZYIIIIIIIXXI",
        "IIXZZYIIIIIIIXXI",
        "IIYZZXIIIIIIIXYI",
        "IIXZZXIIIIIIIXYI",
        "IIYZZYIIIIIIIXYI",
        "IIXZZYIIIIIIIXYI"
    ],
    [
        "IIYZZXIIIIIIIIYX",
        "IIXZZXIIIIIIIIYX",
        "IIYZZYIIIIIIIIYX",
        "IIXZZYIIIIIIIIYX",
        "IIYZZXIIIIIIIIYY",
        "IIXZZXIIIIIIIIYY",
        "IIYZZYIIIIIIIIYY",
        "IIXZZYIIIIIIIIYY",
        "IIYZZXIIIIIIIIXX",
        "IIXZZXIIIIIIIIXX",
        "IIYZZYIIIIIIIIXX",
        "IIXZZYIIIIIIIIXX",
        "IIYZZXIIIIIIIIXY",
        "IIXZZXIIIIIIIIXY",
        "IIYZZYIIIIIIIIXY",
        "IIXZZYIIIIIIIIXY"
    ],
    [
        "IIYZZZXIYZXIIIII",
        "IIXZZZXIYZXIIIII",
        "IIYZZZYIYZXIIIII",
        "IIXZZZYIYZXIIIII",
        "IIYZZZXIYZYIIIII",
        "IIXZZZXIYZYIIIII",
        "IIYZZZYIYZYIIIII",
        "IIXZZZYIYZYIIIII",
        "IIYZZZXIXZXIIIII",
        "IIXZZZXIXZXIIIII",
        "IIYZZZYIXZXIIIII",
        "IIXZZZYIXZXIIIII",
        "IIYZZZXIXZYIIIII",
        "IIXZZZXIXZYIIIII",
        "IIYZZZYIXZYIIIII",
        "IIXZZZYIXZYIIIII"
    ],
    [
        "IIYZZZXIYZZZXIII",
        "IIXZZZXIYZZZXIII",
        "IIYZZZYIYZZZXIII",
        "IIXZZZYIYZZZXIII",
        "IIYZZZXIYZZZYIII",
        "IIXZZZXIYZZZYIII",
        "IIYZZZYIYZZZYIII",
        "IIXZZZYIYZZZYIII",
        "IIYZZZXIXZZZXIII",
        "IIXZZZXIXZZZXIII",
        "IIYZZZYIXZZZXIII",
        "IIXZZZYIXZZZXIII",
        "IIYZZZXIXZZZYIII",
        "IIXZZZXIXZZZYIII",
        "IIYZZZYIXZZZYIII",
        "IIXZZZYIXZZZYIII"
    ],
    [
        "IIYZZZXIYZZZZZXI",
        "IIXZZZXIYZZZZZXI",
        "IIYZZZYIYZZZZZXI",
        "IIXZZZYIYZZZZZXI",
        "IIYZZZXIYZZZZZYI",
        "IIXZZZXIYZZZZZYI",
        "IIYZZZYIYZZZZZYI",
        "IIXZZZYIYZZZZZYI",
        "IIYZZZXIXZZZZZXI",
        "IIXZZZXIXZZZZZXI",
        "IIYZZZYIXZZZZZXI",
        "IIXZZZYIXZZZZZXI",
        "IIYZZZXIXZZZZZYI",
        "IIXZZZXIXZZZZZYI",
        "IIYZZZYIXZZZZZYI",
        "IIXZZZYIXZZZZZYI"
    ],
    [
        "IIYZZZXIIIYZXIII",
        "IIXZZZXIIIYZXIII",
        "IIYZZZYIIIYZXIII",
        "IIXZZZYIIIYZXIII",
        "IIYZZZXIIIYZYIII",
        "IIXZZZXIIIYZYIII",
        "IIYZZZYIIIYZYIII",
        "IIXZZZYIIIYZYIII",
        "IIYZZZXIIIXZXIII",
        "IIXZZZXIIIXZXIII",
        "IIYZZZYIIIXZXIII",
        "IIXZZZYIIIXZXIII",
        "IIYZZZXIIIXZYIII",
        "IIXZZZXIIIXZYIII",
        "IIYZZZYIIIXZYIII",
        "IIXZZZYIIIXZYIII"
    ],
    [
        "IIYZZZXIIIYZZZXI",
        "IIXZZZXIIIYZZZXI",
        "IIYZZZYIIIYZZZXI",
        "IIXZZZYIIIYZZZXI",
        "IIYZZZXIIIYZZZYI",
        "IIXZZZXIIIYZZZYI",
        "IIYZZZYIIIYZZZYI",
        "IIXZZZYIIIYZZZYI",
        "IIYZZZXIIIXZZZXI",
        "IIXZZZXIIIXZZZXI",
        "IIYZZZYIIIXZZZXI",
        "IIXZZZYIIIXZZZXI",
        "IIYZZZXIIIXZZZYI",
        "IIXZZZXIIIXZZZYI",
        "IIYZZZYIIIXZZZYI",
        "IIXZZZYIIIXZZZYI"
    ],
    [
        "IIYZZZXIIIIIYZXI",
        "IIXZZZXIIIIIYZXI",
        "IIYZZZYIIIIIYZXI",
        "IIXZZZYIIIIIYZXI",
        "IIYZZZXIIIIIYZYI",
        "IIXZZZXIIIIIYZYI",
        "IIYZZZYIIIIIYZYI",
        "IIXZZZYIIIIIYZYI",
        "IIYZZZXIIIIIXZXI",
        "IIXZZZXIIIIIXZXI",
        "IIYZZZYIIIIIXZXI",
        "IIXZZZYIIIIIXZXI",
        "IIYZZZXIIIIIXZYI",
        "IIXZZZXIIIIIXZYI",
        "IIYZZZYIIIIIXZYI",
        "IIXZZZYIIIIIXZYI"
    ],
    [
        "IIYZZZZXYXIIIIII",
        "IIXZZZZXYXIIIIII",
        "IIYZZZZYYXIIIIII",
        "IIXZZZZYYXIIIIII",
        "IIYZZZZXYYIIIIII",
        "IIXZZZZXYYIIIIII",
        "IIYZZZZYYYIIIIII",
        "IIXZZZZYYYIIIIII",
        "IIYZZZZXXXIIIIII",
        "IIXZZZZXXXIIIIII",
        "IIYZZZZYXXIIIIII",
        "IIXZZZZYXXIIIIII",
        "IIYZZZZXXYIIIIII",
        "IIXZZZZXXYIIIIII",
        "IIYZZZZYXYIIIIII",
        "IIXZZZZYXYIIIIII"
    ],
    [
        "IIYZZZZXYZZXIIII",
        "IIXZZZZXYZZXIIII",
        "IIYZZZZYYZZXIIII",
        "IIXZZZZYYZZXIIII",
        "IIYZZZZXYZZYIIII",
        "IIXZZZZXYZZYIIII",
        "IIYZZZZYYZZYIIII",
        "IIXZZZZYYZZYIIII",
        "IIYZZZZXXZZXIIII",
        "IIXZZZZXXZZXIIII",
        "IIYZZZZYXZZXIIII",
        "IIXZZZZYXZZXIIII",
        "IIYZZZZXXZZYIIII",
        "IIXZZZZXXZZYIIII",
        "IIYZZZZYXZZYIIII",
        "IIXZZZZYXZZYIIII"
    ],
    [
        "IIYZZZZXYZZZZXII",
        "IIXZZZZXYZZZZXII",
        "IIYZZZZYYZZZZXII",
        "IIXZZZZYYZZZZXII",
        "IIYZZZZXYZZZZYII",
        "IIXZZZZXYZZZZYII",
        "IIYZZZZYYZZZZYII",
        "IIXZZZZYYZZZZYII",
        "IIYZZZZXXZZZZXII",
        "IIXZZZZXXZZZZXII",
        "IIYZZZZYXZZZZXII",
        "IIXZZZZYXZZZZXII",
        "IIYZZZZXXZZZZYII",
        "IIXZZZZXXZZZZYII",
        "IIYZZZZYXZZZZYII",
        "IIXZZZZYXZZZZYII"
    ],
    [
        "IIYZZZZXYZZZZZZX",
        "IIXZZZZXYZZZZZZX",
        "IIYZZZZYYZZZZZZX",
        "IIXZZZZYYZZZZZZX",
        "IIYZZZZXYZZZZZZY",
        "IIXZZZZXYZZZZZZY",
        "IIYZZZZYYZZZZZZY",
        "IIXZZZZYYZZZZZZY",
        "IIYZZZZXXZZZZZZX",
        "IIXZZZZXXZZZZZZX",
        "IIYZZZZYXZZZZZZX",
        "IIXZZZZYXZZZZZZX",
        "IIYZZZZXXZZZZZZY",
        "IIXZZZZXXZZZZZZY",
        "IIYZZZZYXZZZZZZY",
        "IIXZZZZYXZZZZZZY"
    ],
    [
        "IIYZZZZXIYXIIIII",
        "IIXZZZZXIYXIIIII",
        "IIYZZZZYIYXIIIII",
        "IIXZZZZYIYXIIIII",
        "IIYZZZZXIYYIIIII",
        "IIXZZZZXIYYIIIII",
        "IIYZZZZYIYYIIIII",
        "IIXZZZZYIYYIIIII",
        "IIYZZZZXIXXIIIII",
        "IIXZZZZXIXXIIIII",
        "IIYZZZZYIXXIIIII",
        "IIXZZZZYIXXIIIII",
        "IIYZZZZXIXYIIIII",
        "IIXZZZZXIXYIIIII",
        "IIYZZZZYIXYIIIII",
        "IIXZZZZYIXYIIIII"
    ],
    [
        "IIYZZZZXIYZZXIII",
        "IIXZZZZXIYZZXIII",
        "IIYZZZZYIYZZXIII",
        "IIXZZZZYIYZZXIII",
        "IIYZZZZXIYZZYIII",
        "IIXZZZZXIYZZYIII",
        "IIYZZZZYIYZZYIII",
        "IIXZZZZYIYZZYIII",
        "IIYZZZZXIXZZXIII",
        "IIXZZZZXIXZZXIII",
        "IIYZZZZYIXZZXIII",
        "IIXZZZZYIXZZXIII",
        "IIYZZZZXIXZZYIII",
        "IIXZZZZXIXZZYIII",
        "IIYZZZZYIXZZYIII",
        "IIXZZZZYIXZZYIII"
    ],
    [
        "IIYZZZZXIYZZZZXI",
        "IIXZZZZXIYZZZZXI",
        "IIYZZZZYIYZZZZXI",
        "IIXZZZZYIYZZZZXI",
        "IIYZZZZXIYZZZZYI",
        "IIXZZZZXIYZZZZYI",
        "IIYZZZZYIYZZZZYI",
        "IIXZZZZYIYZZZZYI",
        "IIYZZZZXIXZZZZXI",
        "IIXZZZZXIXZZZZXI",
        "IIYZZZZYIXZZZZXI",
        "IIXZZZZYIXZZZZXI",
        "IIYZZZZXIXZZZZYI",
        "IIXZZZZXIXZZZZYI",
        "IIYZZZZYIXZZZZYI",
        "IIXZZZZYIXZZZZYI"
    ],
    [
        "IIYZZZZXIIYXIIII",
        "IIXZZZZXIIYXIIII",
        "IIYZZZZYIIYXIIII",
        "IIXZZZZYIIYXIIII",
        "IIYZZZZXIIYYIIII",
        "IIXZZZZXIIYYIIII",
        "IIYZZZZYIIYYIIII",
        "IIXZZZZYIIYYIIII",
        "IIYZZZZXIIXXIIII",
        "IIXZZZZXIIXXIIII",
        "IIYZZZZYIIXXIIII",
        "IIXZZZZYIIXXIIII",
        "IIYZZZZXIIXYIIII",
        "IIXZZZZXIIXYIIII",
        "IIYZZZZYIIXYIIII",
        "IIXZZZZYIIXYIIII"
    ],
    [
        "IIYZZZZXIIYZZXII",
        "IIXZZZZXIIYZZXII",
        "IIYZZZZYIIYZZXII",
        "IIXZZZZYIIYZZXII",
        "IIYZZZZXIIYZZYII",
        "IIXZZZZXIIYZZYII",
        "IIYZZZZYIIYZZYII",
        "IIXZZZZYIIYZZYII",
        "IIYZZZZXIIXZZXII",
        "IIXZZZZXIIXZZXII",
        "IIYZZZZYIIXZZXII",
        "IIXZZZZYIIXZZXII",
        "IIYZZZZXIIXZZYII",
        "IIXZZZZXIIXZZYII",
        "IIYZZZZYIIXZZYII",
        "IIXZZZZYIIXZZYII"
    ],
    [
        "IIYZZZZXIIYZZZZX",
        "IIXZZZZXIIYZZZZX",
        "IIYZZZZYIIYZZZZX",
        "IIXZZZZYIIYZZZZX",
        "IIYZZZZXIIYZZZZY",
        "IIXZZZZXIIYZZZZY",
        "IIYZZZZYIIYZZZZY",
        "IIXZZZZYIIYZZZZY",
        "IIYZZZZXIIXZZZZX",
        "IIXZZZZXIIXZZZZX",
        "IIYZZZZYIIXZZZZX",
        "IIXZZZZYIIXZZZZX",
        "IIYZZZZXIIXZZZZY",
        "IIXZZZZXIIXZZZZY",
        "IIYZZZZYIIXZZZZY",
        "IIXZZZZYIIXZZZZY"
    ],
    [
        "IIYZZZZXIIIYXIII",
        "IIXZZZZXIIIYXIII",
        "IIYZZZZYIIIYXIII",
        "IIXZZZZYIIIYXIII",
        "IIYZZZZXIIIYYIII",
        "IIXZZZZXIIIYYIII",
        "IIYZZZZYIIIYYIII",
        "IIXZZZZYIIIYYIII",
        "IIYZZZZXIIIXXIII",
        "IIXZZZZXIIIXXIII",
        "IIYZZZZYIIIXXIII",
        "IIXZZZZYIIIXXIII",
        "IIYZZZZXIIIXYIII",
        "IIXZZZZXIIIXYIII",
        "IIYZZZZYIIIXYIII",
        "IIXZZZZYIIIXYIII"
    ],
    [
        "IIYZZZZXIIIYZZXI",
        "IIXZZZZXIIIYZZXI",
        "IIYZZZZYIIIYZZXI",
        "IIXZZZZYIIIYZZXI",
        "IIYZZZZXIIIYZZYI",
        "IIXZZZZXIIIYZZYI",
        "IIYZZZZYIIIYZZYI",
        "IIXZZZZYIIIYZZYI",
        "IIYZZZZXIIIXZZXI",
        "IIXZZZZXIIIXZZXI",
        "IIYZZZZYIIIXZZXI",
        "IIXZZZZYIIIXZZXI",
        "IIYZZZZXIIIXZZYI",
        "IIXZZZZXIIIXZZYI",
        "IIYZZZZYIIIXZZYI",
        "IIXZZZZYIIIXZZYI"
    ],
    [
        "IIYZZZZXIIIIYXII",
        "IIXZZZZXIIIIYXII",
        "IIYZZZZYIIIIYXII",
        "IIXZZZZYIIIIYXII",
        "IIYZZZZXIIIIYYII",
        "IIXZZZZXIIIIYYII",
        "IIYZZZZYIIIIYYII",
        "IIXZZZZYIIIIYYII",
        "IIYZZZZXIIIIXXII",
        "IIXZZZZXIIIIXXII",
        "IIYZZZZYIIIIXXII",
        "IIXZZZZYIIIIXXII",
        "IIYZZZZXIIIIXYII",
        "IIXZZZZXIIIIXYII",
        "IIYZZZZYIIIIXYII",
        "IIXZZZZYIIIIXYII"
    ],
    [
        "IIYZZZZXIIIIYZZX",
        "IIXZZZZXIIIIYZZX",
        "IIYZZZZYIIIIYZZX",
        "IIXZZZZYIIIIYZZX",
        "IIYZZZZXIIIIYZZY",
        "IIXZZZZXIIIIYZZY",
        "IIYZZZZYIIIIYZZY",
        "IIXZZZZYIIIIYZZY",
        "IIYZZZZXIIIIXZZX",
        "IIXZZZZXIIIIXZZX",
        "IIYZZZZYIIIIXZZX",
        "IIXZZZZYIIIIXZZX",
        "IIYZZZZXIIIIXZZY",
        "IIXZZZZXIIIIXZZY",
        "IIYZZZZYIIIIXZZY",
        "IIXZZZZYIIIIXZZY"
    ],
    [
        "IIYZZZZXIIIIIYXI",
        "IIXZZZZXIIIIIYXI",
        "IIYZZZZYIIIIIYXI",
        "IIXZZZZYIIIIIYXI",
        "IIYZZZZXIIIIIYYI",
        "IIXZZZZXIIIIIYYI",
        "IIYZZZZYIIIIIYYI",
        "IIXZZZZYIIIIIYYI",
        "IIYZZZZXIIIIIXXI",
        "IIXZZZZXIIIIIXXI",
        "IIYZZZZYIIIIIXXI",
        "IIXZZZZYIIIIIXXI",
        "IIYZZZZXIIIIIXYI",
        "IIXZZZZXIIIIIXYI",
        "IIYZZZZYIIIIIXYI",
        "IIXZZZZYIIIIIXYI"
    ],
    [
        "IIYZZZZXIIIIIIYX",
        "IIXZZZZXIIIIIIYX",
        "IIYZZZZYIIIIIIYX",
        "IIXZZZZYIIIIIIYX",
        "IIYZZZZXIIIIIIYY",
        "IIXZZZZXIIIIIIYY",
        "IIYZZZZYIIIIIIYY",
        "IIXZZZZYIIIIIIYY",
        "IIYZZZZXIIIIIIXX",
        "IIXZZZZXIIIIIIXX",
        "IIYZZZZYIIIIIIXX",
        "IIXZZZZYIIIIIIXX",
        "IIYZZZZXIIIIIIXY",
        "IIXZZZZXIIIIIIXY",
        "IIYZZZZYIIIIIIXY",
        "IIXZZZZYIIIIIIXY"
    ],
    [
        "IIIYXIIIYXIIIIII",
        "IIIXXIIIYXIIIIII",
        "IIIYYIIIYXIIIIII",
        "IIIXYIIIYXIIIIII",
        "IIIYXIIIYYIIIIII",
        "IIIXXIIIYYIIIIII",
        "IIIYYIIIYYIIIIII",
        "IIIXYIIIYYIIIIII",
        "IIIYXIIIXXIIIIII",
        "IIIXXIIIXXIIIIII",
        "IIIYYIIIXXIIIIII",
        "IIIXYIIIXXIIIIII",
        "IIIYXIIIXYIIIIII",
        "IIIXXIIIXYIIIIII",
        "IIIYYIIIXYIIIIII",
        "IIIXYIIIXYIIIIII"
    ],
    [
        "IIIYXIIIYZZXIIII",
        "IIIXXIIIYZZXIIII",
        "IIIYYIIIYZZXIIII",
        "IIIXYIIIYZZXIIII",
        "IIIYXIIIYZZYIIII",
        "IIIXXIIIYZZYIIII",
        "IIIYYIIIYZZYIIII",
        "IIIXYIIIYZZYIIII",
        "IIIYXIIIXZZXIIII",
        "IIIXXIIIXZZXIIII",
        "IIIYYIIIXZZXIIII",
        "IIIXYIIIXZZXIIII",
        "IIIYXIIIXZZYIIII",
        "IIIXXIIIXZZYIIII",
        "IIIYYIIIXZZYIIII",
        "IIIXYIIIXZZYIIII"
    ],
    [
        "IIIYXIIIYZZZZXII",
        "IIIXXIIIYZZZZXII",
        "IIIYYIIIYZZZZXII",
        "IIIXYIIIYZZZZXII",
        "IIIYXIIIYZZZZYII",
        "IIIXXIIIYZZZZYII",
        "IIIYYIIIYZZZZYII",
        "IIIXYIIIYZZZZYII",
        "IIIYXIIIXZZZZXII",
        "IIIXXIIIXZZZZXII",
        "IIIYYIIIXZZZZXII",
        "IIIXYIIIXZZZZXII",
        "IIIYXIIIXZZZZYII",
        "IIIXXIIIXZZZZYII",
        "IIIYYIIIXZZZZYII",
        "IIIXYIIIXZZZZYII"
    ],
    [
        "IIIYXIIIYZZZZZZX",
        "IIIXXIIIYZZZZZZX",
        "IIIYYIIIYZZZZZZX",
        "IIIXYIIIYZZZZZZX",
        "IIIYXIIIYZZZZZZY",
        "IIIXXIIIYZZZZZZY",
        "IIIYYIIIYZZZZZZY",
        "IIIXYIIIYZZZZZZY",
        "IIIYXIIIXZZZZZZX",
        "IIIXXIIIXZZZZZZX",
        "IIIYYIIIXZZZZZZX",
        "IIIXYIIIXZZZZZZX",
        "IIIYXIIIXZZZZZZY",
        "IIIXXIIIXZZZZZZY",
        "IIIYYIIIXZZZZZZY",
        "IIIXYIIIXZZZZZZY"
    ],
    [
        "IIIYXIIIIYXIIIII",
        "IIIXXIIIIYXIIIII",
        "IIIYYIIIIYXIIIII",
        "IIIXYIIIIYXIIIII",
        "IIIYXIIIIYYIIIII",
        "IIIXXIIIIYYIIIII",
        "IIIYYIIIIYYIIIII",
        "IIIXYIIIIYYIIIII",
        "IIIYXIIIIXXIIIII",
        "IIIXXIIIIXXIIIII",
        "IIIYYIIIIXXIIIII",
        "IIIXYIIIIXXIIIII",
        "IIIYXIIIIXYIIIII",
        "IIIXXIIIIXYIIIII",
        "IIIYYIIIIXYIIIII",
        "IIIXYIIIIXYIIIII"
    ],
    [
        "IIIYXIIIIYZZXIII",
        "IIIXXIIIIYZZXIII",
        "IIIYYIIIIYZZXIII",
        "IIIXYIIIIYZZXIII",
        "IIIYXIIIIYZZYIII",
        "IIIXXIIIIYZZYIII",
        "IIIYYIIIIYZZYIII",
        "IIIXYIIIIYZZYIII",
        "IIIYXIIIIXZZXIII",
        "IIIXXIIIIXZZXIII",
        "IIIYYIIIIXZZXIII",
        "IIIXYIIIIXZZXIII",
        "IIIYXIIIIXZZYIII",
        "IIIXXIIIIXZZYIII",
        "IIIYYIIIIXZZYIII",
        "IIIXYIIIIXZZYIII"
    ],
    [
        "IIIYXIIIIYZZZZXI",
        "IIIXXIIIIYZZZZXI",
        "IIIYYIIIIYZZZZXI",
        "IIIXYIIIIYZZZZXI",
        "IIIYXIIIIYZZZZYI",
        "IIIXXIIIIYZZZZYI",
        "IIIYYIIIIYZZZZYI",
        "IIIXYIIIIYZZZZYI",
        "IIIYXIIIIXZZZZXI",
        "IIIXXIIIIXZZZZXI",
        "IIIYYIIIIXZZZZXI",
        "IIIXYIIIIXZZZZXI",
        "IIIYXIIIIXZZZZYI",
        "IIIXXIIIIXZZZZYI",
        "IIIYYIIIIXZZZZYI",
        "IIIXYIIIIXZZZZYI"
    ],
    [
        "IIIYXIIIIIYXIIII",
        "IIIXXIIIIIYXIIII",
        "IIIYYIIIIIYXIIII",
        "IIIXYIIIIIYXIIII",
        "IIIYXIIIIIYYIIII",
        "IIIXXIIIIIYYIIII",
        "IIIYYIIIIIYYIIII",
        "IIIXYIIIIIYYIIII",
        "IIIYXIIIIIXXIIII",
        "IIIXXIIIIIXXIIII",
        "IIIYYIIIIIXXIIII",
        "IIIXYIIIIIXXIIII",
        "IIIYXIIIIIXYIIII",
        "IIIXXIIIIIXYIIII",
        "IIIYYIIIIIXYIIII",
        "IIIXYIIIIIXYIIII"
    ],
    [
        "IIIYXIIIIIYZZXII",
        "IIIXXIIIIIYZZXII",
        "IIIYYIIIIIYZZXII",
        "IIIXYIIIIIYZZXII",
        "IIIYXIIIIIYZZYII",
        "IIIXXIIIIIYZZYII",
        "IIIYYIIIIIYZZYII",
        "IIIXYIIIIIYZZYII",
        "IIIYXIIIIIXZZXII",
        "IIIXXIIIIIXZZXII",
        "IIIYYIIIIIXZZXII",
        "IIIXYIIIIIXZZXII",
        "IIIYXIIIIIXZZYII",
        "IIIXXIIIIIXZZYII",
        "IIIYYIIIIIXZZYII",
        "IIIXYIIIIIXZZYII"
    ],
    [
        "IIIYXIIIIIYZZZZX",
        "IIIXXIIIIIYZZZZX",
        "IIIYYIIIIIYZZZZX",
        "IIIXYIIIIIYZZZZX",
        "IIIYXIIIIIYZZZZY",
        "IIIXXIIIIIYZZZZY",
        "IIIYYIIIIIYZZZZY",
        "IIIXYIIIIIYZZZZY",
        "IIIYXIIIIIXZZZZX",
        "IIIXXIIIIIXZZZZX",
        "IIIYYIIIIIXZZZZX",
        "IIIXYIIIIIXZZZZX",
        "IIIYXIIIIIXZZZZY",
        "IIIXXIIIIIXZZZZY",
        "IIIYYIIIIIXZZZZY",
        "IIIXYIIIIIXZZZZY"
    ],
    [
        "IIIYXIIIIIIYXIII",
        "IIIXXIIIIIIYXIII",
        "IIIYYIIIIIIYXIII",
        "IIIXYIIIIIIYXIII",
        "IIIYXIIIIIIYYIII",
        "IIIXXIIIIIIYYIII",
        "IIIYYIIIIIIYYIII",
        "IIIXYIIIIIIYYIII",
        "IIIYXIIIIIIXXIII",
        "IIIXXIIIIIIXXIII",
        "IIIYYIIIIIIXXIII",
        "IIIXYIIIIIIXXIII",
        "IIIYXIIIIIIXYIII",
        "IIIXXIIIIIIXYIII",
        "IIIYYIIIIIIXYIII",
        "IIIXYIIIIIIXYIII"
    ],
    [
        "IIIYXIIIIIIYZZXI",
        "IIIXXIIIIIIYZZXI",
        "IIIYYIIIIIIYZZXI",
        "IIIXYIIIIIIYZZXI",
        "IIIYXIIIIIIYZZYI",
        "IIIXXIIIIIIYZZYI",
        "IIIYYIIIIIIYZZYI",
        "IIIXYIIIIIIYZZYI",
        "IIIYXIIIIIIXZZXI",
        "IIIXXIIIIIIXZZXI",
        "IIIYYIIIIIIXZZXI",
        "IIIXYIIIIIIXZZXI",
        "IIIYXIIIIIIXZZYI",
        "IIIXXIIIIIIXZZYI",
        "IIIYYIIIIIIXZZYI",
        "IIIXYIIIIIIXZZYI"
    ],
    [
        "IIIYXIIIIIIIYXII",
        "IIIXXIIIIIIIYXII",
        "IIIYYIIIIIIIYXII",
        "IIIXYIIIIIIIYXII",
        "IIIYXIIIIIIIYYII",
        "IIIXXIIIIIIIYYII",
        "IIIYYIIIIIIIYYII",
        "IIIXYIIIIIIIYYII",
        "IIIYXIIIIIIIXXII",
        "IIIXXIIIIIIIXXII",
        "IIIYYIIIIIIIXXII",
        "IIIXYIIIIIIIXXII",
        "IIIYXIIIIIIIXYII",
        "IIIXXIIIIIIIXYII",
        "IIIYYIIIIIIIXYII",
        "IIIXYIIIIIIIXYII"
    ],
    [
        "IIIYXIIIIIIIYZZX",
        "IIIXXIIIIIIIYZZX",
        "IIIYYIIIIIIIYZZX",
        "IIIXYIIIIIIIYZZX",
        "IIIYXIIIIIIIYZZY",
        "IIIXXIIIIIIIYZZY",
        "IIIYYIIIIIIIYZZY",
        "IIIXYIIIIIIIYZZY",
        "IIIYXIIIIIIIXZZX",
        "IIIXXIIIIIIIXZZX",
        "IIIYYIIIIIIIXZZX",
        "IIIXYIIIIIIIXZZX",
        "IIIYXIIIIIIIXZZY",
        "IIIXXIIIIIIIXZZY",
        "IIIYYIIIIIIIXZZY",
        "IIIXYIIIIIIIXZZY"
    ],
    [
        "IIIYXIIIIIIIIYXI",
        "IIIXXIIIIIIIIYXI",
        "IIIYYIIIIIIIIYXI",
        "IIIXYIIIIIIIIYXI",
        "IIIYXIIIIIIIIYYI",
        "IIIXXIIIIIIIIYYI",
        "IIIYYIIIIIIIIYYI",
        "IIIXYIIIIIIIIYYI",
        "IIIYXIIIIIIIIXXI",
        "IIIXXIIIIIIIIXXI",
        "IIIYYIIIIIIIIXXI",
        "IIIXYIIIIIIIIXXI",
        "IIIYXIIIIIIIIXYI",
        "IIIXXIIIIIIIIXYI",
        "IIIYYIIIIIIIIXYI",
        "IIIXYIIIIIIIIXYI"
    ],
    [
        "IIIYXIIIIIIIIIYX",
        "IIIXXIIIIIIIIIYX",
        "IIIYYIIIIIIIIIYX",
        "IIIXYIIIIIIIIIYX",
        "IIIYXIIIIIIIIIYY",
        "IIIXXIIIIIIIIIYY",
        "IIIYYIIIIIIIIIYY",
        "IIIXYIIIIIIIIIYY",
        "IIIYXIIIIIIIIIXX",
        "IIIXXIIIIIIIIIXX",
        "IIIYYIIIIIIIIIXX",
        "IIIXYIIIIIIIIIXX",
        "IIIYXIIIIIIIIIXY",
        "IIIXXIIIIIIIIIXY",
        "IIIYYIIIIIIIIIXY",
        "IIIXYIIIIIIIIIXY"
    ],
    [
        "IIIYZXIIIYZXIIII",
        "IIIXZXIIIYZXIIII",
        "IIIYZYIIIYZXIIII",
        "IIIXZYIIIYZXIIII",
        "IIIYZXIIIYZYIIII",
        "IIIXZXIIIYZYIIII",
        "IIIYZYIIIYZYIIII",
        "IIIXZYIIIYZYIIII",
        "IIIYZXIIIXZXIIII",
        "IIIXZXIIIXZXIIII",
        "IIIYZYIIIXZXIIII",
        "IIIXZYIIIXZXIIII",
        "IIIYZXIIIXZYIIII",
        "IIIXZXIIIXZYIIII",
        "IIIYZYIIIXZYIIII",
        "IIIXZYIIIXZYIIII"
    ],
    [
        "IIIYZXIIIYZZZXII",
        "IIIXZXIIIYZZZXII",
        "IIIYZYIIIYZZZXII",
        "IIIXZYIIIYZZZXII",
        "IIIYZXIIIYZZZYII",
        "IIIXZXIIIYZZZYII",
        "IIIYZYIIIYZZZYII",
        "IIIXZYIIIYZZZYII",
        "IIIYZXIIIXZZZXII",
        "IIIXZXIIIXZZZXII",
        "IIIYZYIIIXZZZXII",
        "IIIXZYIIIXZZZXII",
        "IIIYZXIIIXZZZYII",
        "IIIXZXIIIXZZZYII",
        "IIIYZYIIIXZZZYII",
        "IIIXZYIIIXZZZYII"
    ],
    [
        "IIIYZXIIIYZZZZZX",
        "IIIXZXIIIYZZZZZX",
        "IIIYZYIIIYZZZZZX",
        "IIIXZYIIIYZZZZZX",
        "IIIYZXIIIYZZZZZY",
        "IIIXZXIIIYZZZZZY",
        "IIIYZYIIIYZZZZZY",
        "IIIXZYIIIYZZZZZY",
        "IIIYZXIIIXZZZZZX",
        "IIIXZXIIIXZZZZZX",
        "IIIYZYIIIXZZZZZX",
        "IIIXZYIIIXZZZZZX",
        "IIIYZXIIIXZZZZZY",
        "IIIXZXIIIXZZZZZY",
        "IIIYZYIIIXZZZZZY",
        "IIIXZYIIIXZZZZZY"
    ],
    [
        "IIIYZXIIIIIYZXII",
        "IIIXZXIIIIIYZXII",
        "IIIYZYIIIIIYZXII",
        "IIIXZYIIIIIYZXII",
        "IIIYZXIIIIIYZYII",
        "IIIXZXIIIIIYZYII",
        "IIIYZYIIIIIYZYII",
        "IIIXZYIIIIIYZYII",
        "IIIYZXIIIIIXZXII",
        "IIIXZXIIIIIXZXII",
        "IIIYZYIIIIIXZXII",
        "IIIXZYIIIIIXZXII",
        "IIIYZXIIIIIXZYII",
        "IIIXZXIIIIIXZYII",
        "IIIYZYIIIIIXZYII",
        "IIIXZYIIIIIXZYII"
    ],
    [
        "IIIYZXIIIIIYZZZX",
        "IIIXZXIIIIIYZZZX",
        "IIIYZYIIIIIYZZZX",
        "IIIXZYIIIIIYZZZX",
        "IIIYZXIIIIIYZZZY",
        "IIIXZXIIIIIYZZZY",
        "IIIYZYIIIIIYZZZY",
        "IIIXZYIIIIIYZZZY",
        "IIIYZXIIIIIXZZZX",
        "IIIXZXIIIIIXZZZX",
        "IIIYZYIIIIIXZZZX",
        "IIIXZYIIIIIXZZZX",
        "IIIYZXIIIIIXZZZY",
        "IIIXZXIIIIIXZZZY",
        "IIIYZYIIIIIXZZZY",
        "IIIXZYIIIIIXZZZY"
    ],
    [
        "IIIYZXIIIIIIIYZX",
        "IIIXZXIIIIIIIYZX",
        "IIIYZYIIIIIIIYZX",
        "IIIXZYIIIIIIIYZX",
        "IIIYZXIIIIIIIYZY",
        "IIIXZXIIIIIIIYZY",
        "IIIYZYIIIIIIIYZY",
        "IIIXZYIIIIIIIYZY",
        "IIIYZXIIIIIIIXZX",
        "IIIXZXIIIIIIIXZX",
        "IIIYZYIIIIIIIXZX",
        "IIIXZYIIIIIIIXZX",
        "IIIYZXIIIIIIIXZY",
        "IIIXZXIIIIIIIXZY",
        "IIIYZYIIIIIIIXZY",
        "IIIXZYIIIIIIIXZY"
    ],
    [
        "IIIYZZXIYXIIIIII",
        "IIIXZZXIYXIIIIII",
        "IIIYZZYIYXIIIIII",
        "IIIXZZYIYXIIIIII",
        "IIIYZZXIYYIIIIII",
        "IIIXZZXIYYIIIIII",
        "IIIYZZYIYYIIIIII",
        "IIIXZZYIYYIIIIII",
        "IIIYZZXIXXIIIIII",
        "IIIXZZXIXXIIIIII",
        "IIIYZZYIXXIIIIII",
        "IIIXZZYIXXIIIIII",
        "IIIYZZXIXYIIIIII",
        "IIIXZZXIXYIIIIII",
        "IIIYZZYIXYIIIIII",
        "IIIXZZYIXYIIIIII"
    ],
    [
        "IIIYZZXIYZZXIIII",
        "IIIXZZXIYZZXIIII",
        "IIIYZZYIYZZXIIII",
        "IIIXZZYIYZZXIIII",
        "IIIYZZXIYZZYIIII",
        "IIIXZZXIYZZYIIII",
        "IIIYZZYIYZZYIIII",
        "IIIXZZYIYZZYIIII",
        "IIIYZZXIXZZXIIII",
        "IIIXZZXIXZZXIIII",
        "IIIYZZYIXZZXIIII",
        "IIIXZZYIXZZXIIII",
        "IIIYZZXIXZZYIIII",
        "IIIXZZXIXZZYIIII",
        "IIIYZZYIXZZYIIII",
        "IIIXZZYIXZZYIIII"
    ],
    [
        "IIIYZZXIYZZZZXII",
        "IIIXZZXIYZZZZXII",
        "IIIYZZYIYZZZZXII",
        "IIIXZZYIYZZZZXII",
        "IIIYZZXIYZZZZYII",
        "IIIXZZXIYZZZZYII",
        "IIIYZZYIYZZZZYII",
        "IIIXZZYIYZZZZYII",
        "IIIYZZXIXZZZZXII",
        "IIIXZZXIXZZZZXII",
        "IIIYZZYIXZZZZXII",
        "IIIXZZYIXZZZZXII",
        "IIIYZZXIXZZZZYII",
        "IIIXZZXIXZZZZYII",
        "IIIYZZYIXZZZZYII",
        "IIIXZZYIXZZZZYII"
    ],
    [
        "IIIYZZXIYZZZZZZX",
        "IIIXZZXIYZZZZZZX",
        "IIIYZZYIYZZZZZZX",
        "IIIXZZYIYZZZZZZX",
        "IIIYZZXIYZZZZZZY",
        "IIIXZZXIYZZZZZZY",
        "IIIYZZYIYZZZZZZY",
        "IIIXZZYIYZZZZZZY",
        "IIIYZZXIXZZZZZZX",
        "IIIXZZXIXZZZZZZX",
        "IIIYZZYIXZZZZZZX",
        "IIIXZZYIXZZZZZZX",
        "IIIYZZXIXZZZZZZY",
        "IIIXZZXIXZZZZZZY",
        "IIIYZZYIXZZZZZZY",
        "IIIXZZYIXZZZZZZY"
    ],
    [
        "IIIYZZXIIYXIIIII",
        "IIIXZZXIIYXIIIII",
        "IIIYZZYIIYXIIIII",
        "IIIXZZYIIYXIIIII",
        "IIIYZZXIIYYIIIII",
        "IIIXZZXIIYYIIIII",
        "IIIYZZYIIYYIIIII",
        "IIIXZZYIIYYIIIII",
        "IIIYZZXIIXXIIIII",
        "IIIXZZXIIXXIIIII",
        "IIIYZZYIIXXIIIII",
        "IIIXZZYIIXXIIIII",
        "IIIYZZXIIXYIIIII",
        "IIIXZZXIIXYIIIII",
        "IIIYZZYIIXYIIIII",
        "IIIXZZYIIXYIIIII"
    ],
    [
        "IIIYZZXIIYZZXIII",
        "IIIXZZXIIYZZXIII",
        "IIIYZZYIIYZZXIII",
        "IIIXZZYIIYZZXIII",
        "IIIYZZXIIYZZYIII",
        "IIIXZZXIIYZZYIII",
        "IIIYZZYIIYZZYIII",
        "IIIXZZYIIYZZYIII",
        "IIIYZZXIIXZZXIII",
        "IIIXZZXIIXZZXIII",
        "IIIYZZYIIXZZXIII",
        "IIIXZZYIIXZZXIII",
        "IIIYZZXIIXZZYIII",
        "IIIXZZXIIXZZYIII",
        "IIIYZZYIIXZZYIII",
        "IIIXZZYIIXZZYIII"
    ],
    [
        "IIIYZZXIIYZZZZXI",
        "IIIXZZXIIYZZZZXI",
        "IIIYZZYIIYZZZZXI",
        "IIIXZZYIIYZZZZXI",
        "IIIYZZXIIYZZZZYI",
        "IIIXZZXIIYZZZZYI",
        "IIIYZZYIIYZZZZYI",
        "IIIXZZYIIYZZZZYI",
        "IIIYZZXIIXZZZZXI",
        "IIIXZZXIIXZZZZXI",
        "IIIYZZYIIXZZZZXI",
        "IIIXZZYIIXZZZZXI",
        "IIIYZZXIIXZZZZYI",
        "IIIXZZXIIXZZZZYI",
        "IIIYZZYIIXZZZZYI",
        "IIIXZZYIIXZZZZYI"
    ],
    [
        "IIIYZZXIIIYXIIII",
        "IIIXZZXIIIYXIIII",
        "IIIYZZYIIIYXIIII",
        "IIIXZZYIIIYXIIII",
        "IIIYZZXIIIYYIIII",
        "IIIXZZXIIIYYIIII",
        "IIIYZZYIIIYYIIII",
        "IIIXZZYIIIYYIIII",
        "IIIYZZXIIIXXIIII",
        "IIIXZZXIIIXXIIII",
        "IIIYZZYIIIXXIIII",
        "IIIXZZYIIIXXIIII",
        "IIIYZZXIIIXYIIII",
        "IIIXZZXIIIXYIIII",
        "IIIYZZYIIIXYIIII",
        "IIIXZZYIIIXYIIII"
    ],
    [
        "IIIYZZXIIIYZZXII",
        "IIIXZZXIIIYZZXII",
        "IIIYZZYIIIYZZXII",
        "IIIXZZYIIIYZZXII",
        "IIIYZZXIIIYZZYII",
        "IIIXZZXIIIYZZYII",
        "IIIYZZYIIIYZZYII",
        "IIIXZZYIIIYZZYII",
        "IIIYZZXIIIXZZXII",
        "IIIXZZXIIIXZZXII",
        "IIIYZZYIIIXZZXII",
        "IIIXZZYIIIXZZXII",
        "IIIYZZXIIIXZZYII",
        "IIIXZZXIIIXZZYII",
        "IIIYZZYIIIXZZYII",
        "IIIXZZYIIIXZZYII"
    ],
    [
        "IIIYZZXIIIYZZZZX",
        "IIIXZZXIIIYZZZZX",
        "IIIYZZYIIIYZZZZX",
        "IIIXZZYIIIYZZZZX",
        "IIIYZZXIIIYZZZZY",
        "IIIXZZXIIIYZZZZY",
        "IIIYZZYIIIYZZZZY",
        "IIIXZZYIIIYZZZZY",
        "IIIYZZXIIIXZZZZX",
        "IIIXZZXIIIXZZZZX",
        "IIIYZZYIIIXZZZZX",
        "IIIXZZYIIIXZZZZX",
        "IIIYZZXIIIXZZZZY",
        "IIIXZZXIIIXZZZZY",
        "IIIYZZYIIIXZZZZY",
        "IIIXZZYIIIXZZZZY"
    ],
    [
        "IIIYZZXIIIIYXIII",
        "IIIXZZXIIIIYXIII",
        "IIIYZZYIIIIYXIII",
        "IIIXZZYIIIIYXIII",
        "IIIYZZXIIIIYYIII",
        "IIIXZZXIIIIYYIII",
        "IIIYZZYIIIIYYIII",
        "IIIXZZYIIIIYYIII",
        "IIIYZZXIIIIXXIII",
        "IIIXZZXIIIIXXIII",
        "IIIYZZYIIIIXXIII",
        "IIIXZZYIIIIXXIII",
        "IIIYZZXIIIIXYIII",
        "IIIXZZXIIIIXYIII",
        "IIIYZZYIIIIXYIII",
        "IIIXZZYIIIIXYIII"
    ],
    [
        "IIIYZZXIIIIYZZXI",
        "IIIXZZXIIIIYZZXI",
        "IIIYZZYIIIIYZZXI",
        "IIIXZZYIIIIYZZXI",
        "IIIYZZXIIIIYZZYI",
        "IIIXZZXIIIIYZZYI",
        "IIIYZZYIIIIYZZYI",
        "IIIXZZYIIIIYZZYI",
        "IIIYZZXIIIIXZZXI",
        "IIIXZZXIIIIXZZXI",
        "IIIYZZYIIIIXZZXI",
        "IIIXZZYIIIIXZZXI",
        "IIIYZZXIIIIXZZYI",
        "IIIXZZXIIIIXZZYI",
        "IIIYZZYIIIIXZZYI",
        "IIIXZZYIIIIXZZYI"
    ],
    [
        "IIIYZZXIIIIIYXII",
        "IIIXZZXIIIIIYXII",
        "IIIYZZYIIIIIYXII",
        "IIIXZZYIIIIIYXII",
        "IIIYZZXIIIIIYYII",
        "IIIXZZXIIIIIYYII",
        "IIIYZZYIIIIIYYII",
        "IIIXZZYIIIIIYYII",
        "IIIYZZXIIIIIXXII",
        "IIIXZZXIIIIIXXII",
        "IIIYZZYIIIIIXXII",
        "IIIXZZYIIIIIXXII",
        "IIIYZZXIIIIIXYII",
        "IIIXZZXIIIIIXYII",
        "IIIYZZYIIIIIXYII",
        "IIIXZZYIIIIIXYII"
    ],
    [
        "IIIYZZXIIIIIYZZX",
        "IIIXZZXIIIIIYZZX",
        "IIIYZZYIIIIIYZZX",
        "IIIXZZYIIIIIYZZX",
        "IIIYZZXIIIIIYZZY",
        "IIIXZZXIIIIIYZZY",
        "IIIYZZYIIIIIYZZY",
        "IIIXZZYIIIIIYZZY",
        "IIIYZZXIIIIIXZZX",
        "IIIXZZXIIIIIXZZX",
        "IIIYZZYIIIIIXZZX",
        "IIIXZZYIIIIIXZZX",
        "IIIYZZXIIIIIXZZY",
        "IIIXZZXIIIIIXZZY",
        "IIIYZZYIIIIIXZZY",
        "IIIXZZYIIIIIXZZY"
    ],
    [
        "IIIYZZXIIIIIIYXI",
        "IIIXZZXIIIIIIYXI",
        "IIIYZZYIIIIIIYXI",
        "IIIXZZYIIIIIIYXI",
        "IIIYZZXIIIIIIYYI",
        "IIIXZZXIIIIIIYYI",
        "IIIYZZYIIIIIIYYI",
        "IIIXZZYIIIIIIYYI",
        "IIIYZZXIIIIIIXXI",
        "IIIXZZXIIIIIIXXI",
        "IIIYZZYIIIIIIXXI",
        "IIIXZZYIIIIIIXXI",
        "IIIYZZXIIIIIIXYI",
        "IIIXZZXIIIIIIXYI",
        "IIIYZZYIIIIIIXYI",
        "IIIXZZYIIIIIIXYI"
    ],
    [
        "IIIYZZXIIIIIIIYX",
        "IIIXZZXIIIIIIIYX",
        "IIIYZZYIIIIIIIYX",
        "IIIXZZYIIIIIIIYX",
        "IIIYZZXIIIIIIIYY",
        "IIIXZZXIIIIIIIYY",
        "IIIYZZYIIIIIIIYY",
        "IIIXZZYIIIIIIIYY",
        "IIIYZZXIIIIIIIXX",
        "IIIXZZXIIIIIIIXX",
        "IIIYZZYIIIIIIIXX",
        "IIIXZZYIIIIIIIXX",
        "IIIYZZXIIIIIIIXY",
        "IIIXZZXIIIIIIIXY",
        "IIIYZZYIIIIIIIXY",
        "IIIXZZYIIIIIIIXY"
    ],
    [
        "IIIYZZZXIYZXIIII",
        "IIIXZZZXIYZXIIII",
        "IIIYZZZYIYZXIIII",
        "IIIXZZZYIYZXIIII",
        "IIIYZZZXIYZYIIII",
        "IIIXZZZXIYZYIIII",
        "IIIYZZZYIYZYIIII",
        "IIIXZZZYIYZYIIII",
        "IIIYZZZXIXZXIIII",
        "IIIXZZZXIXZXIIII",
        "IIIYZZZYIXZXIIII",
        "IIIXZZZYIXZXIIII",
        "IIIYZZZXIXZYIIII",
        "IIIXZZZXIXZYIIII",
        "IIIYZZZYIXZYIIII",
        "IIIXZZZYIXZYIIII"
    ],
    [
        "IIIYZZZXIYZZZXII",
        "IIIXZZZXIYZZZXII",
        "IIIYZZZYIYZZZXII",
        "IIIXZZZYIYZZZXII",
        "IIIYZZZXIYZZZYII",
        "IIIXZZZXIYZZZYII",
        "IIIYZZZYIYZZZYII",
        "IIIXZZZYIYZZZYII",
        "IIIYZZZXIXZZZXII",
        "IIIXZZZXIXZZZXII",
        "IIIYZZZYIXZZZXII",
        "IIIXZZZYIXZZZXII",
        "IIIYZZZXIXZZZYII",
        "IIIXZZZXIXZZZYII",
        "IIIYZZZYIXZZZYII",
        "IIIXZZZYIXZZZYII"
    ],
    [
        "IIIYZZZXIYZZZZZX",
        "IIIXZZZXIYZZZZZX",
        "IIIYZZZYIYZZZZZX",
        "IIIXZZZYIYZZZZZX",
        "IIIYZZZXIYZZZZZY",
        "IIIXZZZXIYZZZZZY",
        "IIIYZZZYIYZZZZZY",
        "IIIXZZZYIYZZZZZY",
        "IIIYZZZXIXZZZZZX",
        "IIIXZZZXIXZZZZZX",
        "IIIYZZZYIXZZZZZX",
        "IIIXZZZYIXZZZZZX",
        "IIIYZZZXIXZZZZZY",
        "IIIXZZZXIXZZZZZY",
        "IIIYZZZYIXZZZZZY",
        "IIIXZZZYIXZZZZZY"
    ],
    [
        "IIIYZZZXIIIYZXII",
        "IIIXZZZXIIIYZXII",
        "IIIYZZZYIIIYZXII",
        "IIIXZZZYIIIYZXII",
        "IIIYZZZXIIIYZYII",
        "IIIXZZZXIIIYZYII",
        "IIIYZZZYIIIYZYII",
        "IIIXZZZYIIIYZYII",
        "IIIYZZZXIIIXZXII",
        "IIIXZZZXIIIXZXII",
        "IIIYZZZYIIIXZXII",
        "IIIXZZZYIIIXZXII",
        "IIIYZZZXIIIXZYII",
        "IIIXZZZXIIIXZYII",
        "IIIYZZZYIIIXZYII",
        "IIIXZZZYIIIXZYII"
    ],
    [
        "IIIYZZZXIIIYZZZX",
        "IIIXZZZXIIIYZZZX",
        "IIIYZZZYIIIYZZZX",
        "IIIXZZZYIIIYZZZX",
        "IIIYZZZXIIIYZZZY",
        "IIIXZZZXIIIYZZZY",
        "IIIYZZZYIIIYZZZY",
        "IIIXZZZYIIIYZZZY",
        "IIIYZZZXIIIXZZZX",
        "IIIXZZZXIIIXZZZX",
        "IIIYZZZYIIIXZZZX",
        "IIIXZZZYIIIXZZZX",
        "IIIYZZZXIIIXZZZY",
        "IIIXZZZXIIIXZZZY",
        "IIIYZZZYIIIXZZZY",
        "IIIXZZZYIIIXZZZY"
    ],
    [
        "IIIYZZZXIIIIIYZX",
        "IIIXZZZXIIIIIYZX",
        "IIIYZZZYIIIIIYZX",
        "IIIXZZZYIIIIIYZX",
        "IIIYZZZXIIIIIYZY",
        "IIIXZZZXIIIIIYZY",
        "IIIYZZZYIIIIIYZY",
        "IIIXZZZYIIIIIYZY",
        "IIIYZZZXIIIIIXZX",
        "IIIXZZZXIIIIIXZX",
        "IIIYZZZYIIIIIXZX",
        "IIIXZZZYIIIIIXZX",
        "IIIYZZZXIIIIIXZY",
        "IIIXZZZXIIIIIXZY",
        "IIIYZZZYIIIIIXZY",
        "IIIXZZZYIIIIIXZY"
    ],
    [
        "IIIIYXIIYXIIIIII",
        "IIIIXXIIYXIIIIII",
        "IIIIYYIIYXIIIIII",
        "IIIIXYIIYXIIIIII",
        "IIIIYXIIYYIIIIII",
        "IIIIXXIIYYIIIIII",
        "IIIIYYIIYYIIIIII",
        "IIIIXYIIYYIIIIII",
        "IIIIYXIIXXIIIIII",
        "IIIIXXIIXXIIIIII",
        "IIIIYYIIXXIIIIII",
        "IIIIXYIIXXIIIIII",
        "IIIIYXIIXYIIIIII",
        "IIIIXXIIXYIIIIII",
        "IIIIYYIIXYIIIIII",
        "IIIIXYIIXYIIIIII"
    ],
    [
        "IIIIYXIIYZZXIIII",
        "IIIIXXIIYZZXIIII",
        "IIIIYYIIYZZXIIII",
        "IIIIXYIIYZZXIIII",
        "IIIIYXIIYZZYIIII",
        "IIIIXXIIYZZYIIII",
        "IIIIYYIIYZZYIIII",
        "IIIIXYIIYZZYIIII",
        "IIIIYXIIXZZXIIII",
        "IIIIXXIIXZZXIIII",
        "IIIIYYIIXZZXIIII",
        "IIIIXYIIXZZXIIII",
        "IIIIYXIIXZZYIIII",
        "IIIIXXIIXZZYIIII",
        "IIIIYYIIXZZYIIII",
        "IIIIXYIIXZZYIIII"
    ],
    [
        "IIIIYXIIYZZZZXII",
        "IIIIXXIIYZZZZXII",
        "IIIIYYIIYZZZZXII",
        "IIIIXYIIYZZZZXII",
        "IIIIYXIIYZZZZYII",
        "IIIIXXIIYZZZZYII",
        "IIIIYYIIYZZZZYII",
        "IIIIXYIIYZZZZYII",
        "IIIIYXIIXZZZZXII",
        "IIIIXXIIXZZZZXII",
        "IIIIYYIIXZZZZXII",
        "IIIIXYIIXZZZZXII",
        "IIIIYXIIXZZZZYII",
        "IIIIXXIIXZZZZYII",
        "IIIIYYIIXZZZZYII",
        "IIIIXYIIXZZZZYII"
    ],
    [
        "IIIIYXIIYZZZZZZX",
        "IIIIXXIIYZZZZZZX",
        "IIIIYYIIYZZZZZZX",
        "IIIIXYIIYZZZZZZX",
        "IIIIYXIIYZZZZZZY",
        "IIIIXXIIYZZZZZZY",
        "IIIIYYIIYZZZZZZY",
        "IIIIXYIIYZZZZZZY",
        "IIIIYXIIXZZZZZZX",
        "IIIIXXIIXZZZZZZX",
        "IIIIYYIIXZZZZZZX",
        "IIIIXYIIXZZZZZZX",
        "IIIIYXIIXZZZZZZY",
        "IIIIXXIIXZZZZZZY",
        "IIIIYYIIXZZZZZZY",
        "IIIIXYIIXZZZZZZY"
    ],
    [
        "IIIIYXIIIYXIIIII",
        "IIIIXXIIIYXIIIII",
        "IIIIYYIIIYXIIIII",
        "IIIIXYIIIYXIIIII",
        "IIIIYXIIIYYIIIII",
        "IIIIXXIIIYYIIIII",
        "IIIIYYIIIYYIIIII",
        "IIIIXYIIIYYIIIII",
        "IIIIYXIIIXXIIIII",
        "IIIIXXIIIXXIIIII",
        "IIIIYYIIIXXIIIII",
        "IIIIXYIIIXXIIIII",
        "IIIIYXIIIXYIIIII",
        "IIIIXXIIIXYIIIII",
        "IIIIYYIIIXYIIIII",
        "IIIIXYIIIXYIIIII"
    ],
    [
        "IIIIYXIIIYZZXIII",
        "IIIIXXIIIYZZXIII",
        "IIIIYYIIIYZZXIII",
        "IIIIXYIIIYZZXIII",
        "IIIIYXIIIYZZYIII",
        "IIIIXXIIIYZZYIII",
        "IIIIYYIIIYZZYIII",
        "IIIIXYIIIYZZYIII",
        "IIIIYXIIIXZZXIII",
        "IIIIXXIIIXZZXIII",
        "IIIIYYIIIXZZXIII",
        "IIIIXYIIIXZZXIII",
        "IIIIYXIIIXZZYIII",
        "IIIIXXIIIXZZYIII",
        "IIIIYYIIIXZZYIII",
        "IIIIXYIIIXZZYIII"
    ],
    [
        "IIIIYXIIIYZZZZXI",
        "IIIIXXIIIYZZZZXI",
        "IIIIYYIIIYZZZZXI",
        "IIIIXYIIIYZZZZXI",
        "IIIIYXIIIYZZZZYI",
        "IIIIXXIIIYZZZZYI",
        "IIIIYYIIIYZZZZYI",
        "IIIIXYIIIYZZZZYI",
        "IIIIYXIIIXZZZZXI",
        "IIIIXXIIIXZZZZXI",
        "IIIIYYIIIXZZZZXI",
        "IIIIXYIIIXZZZZXI",
        "IIIIYXIIIXZZZZYI",
        "IIIIXXIIIXZZZZYI",
        "IIIIYYIIIXZZZZYI",
        "IIIIXYIIIXZZZZYI"
    ],
    [
        "IIIIYXIIIIYXIIII",
        "IIIIXXIIIIYXIIII",
        "IIIIYYIIIIYXIIII",
        "IIIIXYIIIIYXIIII",
        "IIIIYXIIIIYYIIII",
        "IIIIXXIIIIYYIIII",
        "IIIIYYIIIIYYIIII",
        "IIIIXYIIIIYYIIII",
        "IIIIYXIIIIXXIIII",
        "IIIIXXIIIIXXIIII",
        "IIIIYYIIIIXXIIII",
        "IIIIXYIIIIXXIIII",
        "IIIIYXIIIIXYIIII",
        "IIIIXXIIIIXYIIII",
        "IIIIYYIIIIXYIIII",
        "IIIIXYIIIIXYIIII"
    ],
    [
        "IIIIYXIIIIYZZXII",
        "IIIIXXIIIIYZZXII",
        "IIIIYYIIIIYZZXII",
        "IIIIXYIIIIYZZXII",
        "IIIIYXIIIIYZZYII",
        "IIIIXXIIIIYZZYII",
        "IIIIYYIIIIYZZYII",
        "IIIIXYIIIIYZZYII",
        "IIIIYXIIIIXZZXII",
        "IIIIXXIIIIXZZXII",
        "IIIIYYIIIIXZZXII",
        "IIIIXYIIIIXZZXII",
        "IIIIYXIIIIXZZYII",
        "IIIIXXIIIIXZZYII",
        "IIIIYYIIIIXZZYII",
        "IIIIXYIIIIXZZYII"
    ],
    [
        "IIIIYXIIIIYZZZZX",
        "IIIIXXIIIIYZZZZX",
        "IIIIYYIIIIYZZZZX",
        "IIIIXYIIIIYZZZZX",
        "IIIIYXIIIIYZZZZY",
        "IIIIXXIIIIYZZZZY",
        "IIIIYYIIIIYZZZZY",
        "IIIIXYIIIIYZZZZY",
        "IIIIYXIIIIXZZZZX",
        "IIIIXXIIIIXZZZZX",
        "IIIIYYIIIIXZZZZX",
        "IIIIXYIIIIXZZZZX",
        "IIIIYXIIIIXZZZZY",
        "IIIIXXIIIIXZZZZY",
        "IIIIYYIIIIXZZZZY",
        "IIIIXYIIIIXZZZZY"
    ],
    [
        "IIIIYXIIIIIYXIII",
        "IIIIXXIIIIIYXIII",
        "IIIIYYIIIIIYXIII",
        "IIIIXYIIIIIYXIII",
        "IIIIYXIIIIIYYIII",
        "IIIIXXIIIIIYYIII",
        "IIIIYYIIIIIYYIII",
        "IIIIXYIIIIIYYIII",
        "IIIIYXIIIIIXXIII",
        "IIIIXXIIIIIXXIII",
        "IIIIYYIIIIIXXIII",
        "IIIIXYIIIIIXXIII",
        "IIIIYXIIIIIXYIII",
        "IIIIXXIIIIIXYIII",
        "IIIIYYIIIIIXYIII",
        "IIIIXYIIIIIXYIII"
    ],
    [
        "IIIIYXIIIIIYZZXI",
        "IIIIXXIIIIIYZZXI",
        "IIIIYYIIIIIYZZXI",
        "IIIIXYIIIIIYZZXI",
        "IIIIYXIIIIIYZZYI",
        "IIIIXXIIIIIYZZYI",
        "IIIIYYIIIIIYZZYI",
        "IIIIXYIIIIIYZZYI",
        "IIIIYXIIIIIXZZXI",
        "IIIIXXIIIIIXZZXI",
        "IIIIYYIIIIIXZZXI",
        "IIIIXYIIIIIXZZXI",
        "IIIIYXIIIIIXZZYI",
        "IIIIXXIIIIIXZZYI",
        "IIIIYYIIIIIXZZYI",
        "IIIIXYIIIIIXZZYI"
    ],
    [
        "IIIIYXIIIIIIYXII",
        "IIIIXXIIIIIIYXII",
        "IIIIYYIIIIIIYXII",
        "IIIIXYIIIIIIYXII",
        "IIIIYXIIIIIIYYII",
        "IIIIXXIIIIIIYYII",
        "IIIIYYIIIIIIYYII",
        "IIIIXYIIIIIIYYII",
        "IIIIYXIIIIIIXXII",
        "IIIIXXIIIIIIXXII",
        "IIIIYYIIIIIIXXII",
        "IIIIXYIIIIIIXXII",
        "IIIIYXIIIIIIXYII",
        "IIIIXXIIIIIIXYII",
        "IIIIYYIIIIIIXYII",
        "IIIIXYIIIIIIXYII"
    ],
    [
        "IIIIYXIIIIIIYZZX",
        "IIIIXXIIIIIIYZZX",
        "IIIIYYIIIIIIYZZX",
        "IIIIXYIIIIIIYZZX",
        "IIIIYXIIIIIIYZZY",
        "IIIIXXIIIIIIYZZY",
        "IIIIYYIIIIIIYZZY",
        "IIIIXYIIIIIIYZZY",
        "IIIIYXIIIIIIXZZX",
        "IIIIXXIIIIIIXZZX",
        "IIIIYYIIIIIIXZZX",
        "IIIIXYIIIIIIXZZX",
        "IIIIYXIIIIIIXZZY",
        "IIIIXXIIIIIIXZZY",
        "IIIIYYIIIIIIXZZY",
        "IIIIXYIIIIIIXZZY"
    ],
    [
        "IIIIYXIIIIIIIYXI",
        "IIIIXXIIIIIIIYXI",
        "IIIIYYIIIIIIIYXI",
        "IIIIXYIIIIIIIYXI",
        "IIIIYXIIIIIIIYYI",
        "IIIIXXIIIIIIIYYI",
        "IIIIYYIIIIIIIYYI",
        "IIIIXYIIIIIIIYYI",
        "IIIIYXIIIIIIIXXI",
        "IIIIXXIIIIIIIXXI",
        "IIIIYYIIIIIIIXXI",
        "IIIIXYIIIIIIIXXI",
        "IIIIYXIIIIIIIXYI",
        "IIIIXXIIIIIIIXYI",
        "IIIIYYIIIIIIIXYI",
        "IIIIXYIIIIIIIXYI"
    ],
    [
        "IIIIYXIIIIIIIIYX",
        "IIIIXXIIIIIIIIYX",
        "IIIIYYIIIIIIIIYX",
        "IIIIXYIIIIIIIIYX",
        "IIIIYXIIIIIIIIYY",
        "IIIIXXIIIIIIIIYY",
        "IIIIYYIIIIIIIIYY",
        "IIIIXYIIIIIIIIYY",
        "IIIIYXIIIIIIIIXX",
        "IIIIXXIIIIIIIIXX",
        "IIIIYYIIIIIIIIXX",
        "IIIIXYIIIIIIIIXX",
        "IIIIYXIIIIIIIIXY",
        "IIIIXXIIIIIIIIXY",
        "IIIIYYIIIIIIIIXY",
        "IIIIXYIIIIIIIIXY"
    ],
    [
        "IIIIYZXIYZXIIIII",
        "IIIIXZXIYZXIIIII",
        "IIIIYZYIYZXIIIII",
        "IIIIXZYIYZXIIIII",
        "IIIIYZXIYZYIIIII",
        "IIIIXZXIYZYIIIII",
        "IIIIYZYIYZYIIIII",
        "IIIIXZYIYZYIIIII",
        "IIIIYZXIXZXIIIII",
        "IIIIXZXIXZXIIIII",
        "IIIIYZYIXZXIIIII",
        "IIIIXZYIXZXIIIII",
        "IIIIYZXIXZYIIIII",
        "IIIIXZXIXZYIIIII",
        "IIIIYZYIXZYIIIII",
        "IIIIXZYIXZYIIIII"
    ],
    [
        "IIIIYZXIYZZZXIII",
        "IIIIXZXIYZZZXIII",
        "IIIIYZYIYZZZXIII",
        "IIIIXZYIYZZZXIII",
        "IIIIYZXIYZZZYIII",
        "IIIIXZXIYZZZYIII",
        "IIIIYZYIYZZZYIII",
        "IIIIXZYIYZZZYIII",
        "IIIIYZXIXZZZXIII",
        "IIIIXZXIXZZZXIII",
        "IIIIYZYIXZZZXIII",
        "IIIIXZYIXZZZXIII",
        "IIIIYZXIXZZZYIII",
        "IIIIXZXIXZZZYIII",
        "IIIIYZYIXZZZYIII",
        "IIIIXZYIXZZZYIII"
    ],
    [
        "IIIIYZXIYZZZZZXI",
        "IIIIXZXIYZZZZZXI",
        "IIIIYZYIYZZZZZXI",
        "IIIIXZYIYZZZZZXI",
        "IIIIYZXIYZZZZZYI",
        "IIIIXZXIYZZZZZYI",
        "IIIIYZYIYZZZZZYI",
        "IIIIXZYIYZZZZZYI",
        "IIIIYZXIXZZZZZXI",
        "IIIIXZXIXZZZZZXI",
        "IIIIYZYIXZZZZZXI",
        "IIIIXZYIXZZZZZXI",
        "IIIIYZXIXZZZZZYI",
        "IIIIXZXIXZZZZZYI",
        "IIIIYZYIXZZZZZYI",
        "IIIIXZYIXZZZZZYI"
    ],
    [
        "IIIIYZXIIIYZXIII",
        "IIIIXZXIIIYZXIII",
        "IIIIYZYIIIYZXIII",
        "IIIIXZYIIIYZXIII",
        "IIIIYZXIIIYZYIII",
        "IIIIXZXIIIYZYIII",
        "IIIIYZYIIIYZYIII",
        "IIIIXZYIIIYZYIII",
        "IIIIYZXIIIXZXIII",
        "IIIIXZXIIIXZXIII",
        "IIIIYZYIIIXZXIII",
        "IIIIXZYIIIXZXIII",
        "IIIIYZXIIIXZYIII",
        "IIIIXZXIIIXZYIII",
        "IIIIYZYIIIXZYIII",
        "IIIIXZYIIIXZYIII"
    ],
    [
        "IIIIYZXIIIYZZZXI",
        "IIIIXZXIIIYZZZXI",
        "IIIIYZYIIIYZZZXI",
        "IIIIXZYIIIYZZZXI",
        "IIIIYZXIIIYZZZYI",
        "IIIIXZXIIIYZZZYI",
        "IIIIYZYIIIYZZZYI",
        "IIIIXZYIIIYZZZYI",
        "IIIIYZXIIIXZZZXI",
        "IIIIXZXIIIXZZZXI",
        "IIIIYZYIIIXZZZXI",
        "IIIIXZYIIIXZZZXI",
        "IIIIYZXIIIXZZZYI",
        "IIIIXZXIIIXZZZYI",
        "IIIIYZYIIIXZZZYI",
        "IIIIXZYIIIXZZZYI"
    ],
    [
        "IIIIYZXIIIIIYZXI",
        "IIIIXZXIIIIIYZXI",
        "IIIIYZYIIIIIYZXI",
        "IIIIXZYIIIIIYZXI",
        "IIIIYZXIIIIIYZYI",
        "IIIIXZXIIIIIYZYI",
        "IIIIYZYIIIIIYZYI",
        "IIIIXZYIIIIIYZYI",
        "IIIIYZXIIIIIXZXI",
        "IIIIXZXIIIIIXZXI",
        "IIIIYZYIIIIIXZXI",
        "IIIIXZYIIIIIXZXI",
        "IIIIYZXIIIIIXZYI",
        "IIIIXZXIIIIIXZYI",
        "IIIIYZYIIIIIXZYI",
        "IIIIXZYIIIIIXZYI"
    ],
    [
        "IIIIYZZXYXIIIIII",
        "IIIIXZZXYXIIIIII",
        "IIIIYZZYYXIIIIII",
        "IIIIXZZYYXIIIIII",
        "IIIIYZZXYYIIIIII",
        "IIIIXZZXYYIIIIII",
        "IIIIYZZYYYIIIIII",
        "IIIIXZZYYYIIIIII",
        "IIIIYZZXXXIIIIII",
        "IIIIXZZXXXIIIIII",
        "IIIIYZZYXXIIIIII",
        "IIIIXZZYXXIIIIII",
        "IIIIYZZXXYIIIIII",
        "IIIIXZZXXYIIIIII",
        "IIIIYZZYXYIIIIII",
        "IIIIXZZYXYIIIIII"
    ],
    [
        "IIIIYZZXYZZXIIII",
        "IIIIXZZXYZZXIIII",
        "IIIIYZZYYZZXIIII",
        "IIIIXZZYYZZXIIII",
        "IIIIYZZXYZZYIIII",
        "IIIIXZZXYZZYIIII",
        "IIIIYZZYYZZYIIII",
        "IIIIXZZYYZZYIIII",
        "IIIIYZZXXZZXIIII",
        "IIIIXZZXXZZXIIII",
        "IIIIYZZYXZZXIIII",
        "IIIIXZZYXZZXIIII",
        "IIIIYZZXXZZYIIII",
        "IIIIXZZXXZZYIIII",
        "IIIIYZZYXZZYIIII",
        "IIIIXZZYXZZYIIII"
    ],
    [
        "IIIIYZZXYZZZZXII",
        "IIIIXZZXYZZZZXII",
        "IIIIYZZYYZZZZXII",
        "IIIIXZZYYZZZZXII",
        "IIIIYZZXYZZZZYII",
        "IIIIXZZXYZZZZYII",
        "IIIIYZZYYZZZZYII",
        "IIIIXZZYYZZZZYII",
        "IIIIYZZXXZZZZXII",
        "IIIIXZZXXZZZZXII",
        "IIIIYZZYXZZZZXII",
        "IIIIXZZYXZZZZXII",
        "IIIIYZZXXZZZZYII",
        "IIIIXZZXXZZZZYII",
        "IIIIYZZYXZZZZYII",
        "IIIIXZZYXZZZZYII"
    ],
    [
        "IIIIYZZXYZZZZZZX",
        "IIIIXZZXYZZZZZZX",
        "IIIIYZZYYZZZZZZX",
        "IIIIXZZYYZZZZZZX",
        "IIIIYZZXYZZZZZZY",
        "IIIIXZZXYZZZZZZY",
        "IIIIYZZYYZZZZZZY",
        "IIIIXZZYYZZZZZZY",
        "IIIIYZZXXZZZZZZX",
        "IIIIXZZXXZZZZZZX",
        "IIIIYZZYXZZZZZZX",
        "IIIIXZZYXZZZZZZX",
        "IIIIYZZXXZZZZZZY",
        "IIIIXZZXXZZZZZZY",
        "IIIIYZZYXZZZZZZY",
        "IIIIXZZYXZZZZZZY"
    ],
    [
        "IIIIYZZXIYXIIIII",
        "IIIIXZZXIYXIIIII",
        "IIIIYZZYIYXIIIII",
        "IIIIXZZYIYXIIIII",
        "IIIIYZZXIYYIIIII",
        "IIIIXZZXIYYIIIII",
        "IIIIYZZYIYYIIIII",
        "IIIIXZZYIYYIIIII",
        "IIIIYZZXIXXIIIII",
        "IIIIXZZXIXXIIIII",
        "IIIIYZZYIXXIIIII",
        "IIIIXZZYIXXIIIII",
        "IIIIYZZXIXYIIIII",
        "IIIIXZZXIXYIIIII",
        "IIIIYZZYIXYIIIII",
        "IIIIXZZYIXYIIIII"
    ],
    [
        "IIIIYZZXIYZZXIII",
        "IIIIXZZXIYZZXIII",
        "IIIIYZZYIYZZXIII",
        "IIIIXZZYIYZZXIII",
        "IIIIYZZXIYZZYIII",
        "IIIIXZZXIYZZYIII",
        "IIIIYZZYIYZZYIII",
        "IIIIXZZYIYZZYIII",
        "IIIIYZZXIXZZXIII",
        "IIIIXZZXIXZZXIII",
        "IIIIYZZYIXZZXIII",
        "IIIIXZZYIXZZXIII",
        "IIIIYZZXIXZZYIII",
        "IIIIXZZXIXZZYIII",
        "IIIIYZZYIXZZYIII",
        "IIIIXZZYIXZZYIII"
    ],
    [
        "IIIIYZZXIYZZZZXI",
        "IIIIXZZXIYZZZZXI",
        "IIIIYZZYIYZZZZXI",
        "IIIIXZZYIYZZZZXI",
        "IIIIYZZXIYZZZZYI",
        "IIIIXZZXIYZZZZYI",
        "IIIIYZZYIYZZZZYI",
        "IIIIXZZYIYZZZZYI",
        "IIIIYZZXIXZZZZXI",
        "IIIIXZZXIXZZZZXI",
        "IIIIYZZYIXZZZZXI",
        "IIIIXZZYIXZZZZXI",
        "IIIIYZZXIXZZZZYI",
        "IIIIXZZXIXZZZZYI",
        "IIIIYZZYIXZZZZYI",
        "IIIIXZZYIXZZZZYI"
    ],
    [
        "IIIIYZZXIIYXIIII",
        "IIIIXZZXIIYXIIII",
        "IIIIYZZYIIYXIIII",
        "IIIIXZZYIIYXIIII",
        "IIIIYZZXIIYYIIII",
        "IIIIXZZXIIYYIIII",
        "IIIIYZZYIIYYIIII",
        "IIIIXZZYIIYYIIII",
        "IIIIYZZXIIXXIIII",
        "IIIIXZZXIIXXIIII",
        "IIIIYZZYIIXXIIII",
        "IIIIXZZYIIXXIIII",
        "IIIIYZZXIIXYIIII",
        "IIIIXZZXIIXYIIII",
        "IIIIYZZYIIXYIIII",
        "IIIIXZZYIIXYIIII"
    ],
    [
        "IIIIYZZXIIYZZXII",
        "IIIIXZZXIIYZZXII",
        "IIIIYZZYIIYZZXII",
        "IIIIXZZYIIYZZXII",
        "IIIIYZZXIIYZZYII",
        "IIIIXZZXIIYZZYII",
        "IIIIYZZYIIYZZYII",
        "IIIIXZZYIIYZZYII",
        "IIIIYZZXIIXZZXII",
        "IIIIXZZXIIXZZXII",
        "IIIIYZZYIIXZZXII",
        "IIIIXZZYIIXZZXII",
        "IIIIYZZXIIXZZYII",
        "IIIIXZZXIIXZZYII",
        "IIIIYZZYIIXZZYII",
        "IIIIXZZYIIXZZYII"
    ],
    [
        "IIIIYZZXIIYZZZZX",
        "IIIIXZZXIIYZZZZX",
        "IIIIYZZYIIYZZZZX",
        "IIIIXZZYIIYZZZZX",
        "IIIIYZZXIIYZZZZY",
        "IIIIXZZXIIYZZZZY",
        "IIIIYZZYIIYZZZZY",
        "IIIIXZZYIIYZZZZY",
        "IIIIYZZXIIXZZZZX",
        "IIIIXZZXIIXZZZZX",
        "IIIIYZZYIIXZZZZX",
        "IIIIXZZYIIXZZZZX",
        "IIIIYZZXIIXZZZZY",
        "IIIIXZZXIIXZZZZY",
        "IIIIYZZYIIXZZZZY",
        "IIIIXZZYIIXZZZZY"
    ],
    [
        "IIIIYZZXIIIYXIII",
        "IIIIXZZXIIIYXIII",
        "IIIIYZZYIIIYXIII",
        "IIIIXZZYIIIYXIII",
        "IIIIYZZXIIIYYIII",
        "IIIIXZZXIIIYYIII",
        "IIIIYZZYIIIYYIII",
        "IIIIXZZYIIIYYIII",
        "IIIIYZZXIIIXXIII",
        "IIIIXZZXIIIXXIII",
        "IIIIYZZYIIIXXIII",
        "IIIIXZZYIIIXXIII",
        "IIIIYZZXIIIXYIII",
        "IIIIXZZXIIIXYIII",
        "IIIIYZZYIIIXYIII",
        "IIIIXZZYIIIXYIII"
    ],
    [
        "IIIIYZZXIIIYZZXI",
        "IIIIXZZXIIIYZZXI",
        "IIIIYZZYIIIYZZXI",
        "IIIIXZZYIIIYZZXI",
        "IIIIYZZXIIIYZZYI",
        "IIIIXZZXIIIYZZYI",
        "IIIIYZZYIIIYZZYI",
        "IIIIXZZYIIIYZZYI",
        "IIIIYZZXIIIXZZXI",
        "IIIIXZZXIIIXZZXI",
        "IIIIYZZYIIIXZZXI",
        "IIIIXZZYIIIXZZXI",
        "IIIIYZZXIIIXZZYI",
        "IIIIXZZXIIIXZZYI",
        "IIIIYZZYIIIXZZYI",
        "IIIIXZZYIIIXZZYI"
    ],
    [
        "IIIIYZZXIIIIYXII",
        "IIIIXZZXIIIIYXII",
        "IIIIYZZYIIIIYXII",
        "IIIIXZZYIIIIYXII",
        "IIIIYZZXIIIIYYII",
        "IIIIXZZXIIIIYYII",
        "IIIIYZZYIIIIYYII",
        "IIIIXZZYIIIIYYII",
        "IIIIYZZXIIIIXXII",
        "IIIIXZZXIIIIXXII",
        "IIIIYZZYIIIIXXII",
        "IIIIXZZYIIIIXXII",
        "IIIIYZZXIIIIXYII",
        "IIIIXZZXIIIIXYII",
        "IIIIYZZYIIIIXYII",
        "IIIIXZZYIIIIXYII"
    ],
    [
        "IIIIYZZXIIIIYZZX",
        "IIIIXZZXIIIIYZZX",
        "IIIIYZZYIIIIYZZX",
        "IIIIXZZYIIIIYZZX",
        "IIIIYZZXIIIIYZZY",
        "IIIIXZZXIIIIYZZY",
        "IIIIYZZYIIIIYZZY",
        "IIIIXZZYIIIIYZZY",
        "IIIIYZZXIIIIXZZX",
        "IIIIXZZXIIIIXZZX",
        "IIIIYZZYIIIIXZZX",
        "IIIIXZZYIIIIXZZX",
        "IIIIYZZXIIIIXZZY",
        "IIIIXZZXIIIIXZZY",
        "IIIIYZZYIIIIXZZY",
        "IIIIXZZYIIIIXZZY"
    ],
    [
        "IIIIYZZXIIIIIYXI",
        "IIIIXZZXIIIIIYXI",
        "IIIIYZZYIIIIIYXI",
        "IIIIXZZYIIIIIYXI",
        "IIIIYZZXIIIIIYYI",
        "IIIIXZZXIIIIIYYI",
        "IIIIYZZYIIIIIYYI",
        "IIIIXZZYIIIIIYYI",
        "IIIIYZZXIIIIIXXI",
        "IIIIXZZXIIIIIXXI",
        "IIIIYZZYIIIIIXXI",
        "IIIIXZZYIIIIIXXI",
        "IIIIYZZXIIIIIXYI",
        "IIIIXZZXIIIIIXYI",
        "IIIIYZZYIIIIIXYI",
        "IIIIXZZYIIIIIXYI"
    ],
    [
        "IIIIYZZXIIIIIIYX",
        "IIIIXZZXIIIIIIYX",
        "IIIIYZZYIIIIIIYX",
        "IIIIXZZYIIIIIIYX",
        "IIIIYZZXIIIIIIYY",
        "IIIIXZZXIIIIIIYY",
        "IIIIYZZYIIIIIIYY",
        "IIIIXZZYIIIIIIYY",
        "IIIIYZZXIIIIIIXX",
        "IIIIXZZXIIIIIIXX",
        "IIIIYZZYIIIIIIXX",
        "IIIIXZZYIIIIIIXX",
        "IIIIYZZXIIIIIIXY",
        "IIIIXZZXIIIIIIXY",
        "IIIIYZZYIIIIIIXY",
        "IIIIXZZYIIIIIIXY"
    ],
    [
        "IIIIIYXIYXIIIIII",
        "IIIIIXXIYXIIIIII",
        "IIIIIYYIYXIIIIII",
        "IIIIIXYIYXIIIIII",
        "IIIIIYXIYYIIIIII",
        "IIIIIXXIYYIIIIII",
        "IIIIIYYIYYIIIIII",
        "IIIIIXYIYYIIIIII",
        "IIIIIYXIXXIIIIII",
        "IIIIIXXIXXIIIIII",
        "IIIIIYYIXXIIIIII",
        "IIIIIXYIXXIIIIII",
        "IIIIIYXIXYIIIIII",
        "IIIIIXXIXYIIIIII",
        "IIIIIYYIXYIIIIII",
        "IIIIIXYIXYIIIIII"
    ],
    [
        "IIIIIYXIYZZXIIII",
        "IIIIIXXIYZZXIIII",
        "IIIIIYYIYZZXIIII",
        "IIIIIXYIYZZXIIII",
        "IIIIIYXIYZZYIIII",
        "IIIIIXXIYZZYIIII",
        "IIIIIYYIYZZYIIII",
        "IIIIIXYIYZZYIIII",
        "IIIIIYXIXZZXIIII",
        "IIIIIXXIXZZXIIII",
        "IIIIIYYIXZZXIIII",
        "IIIIIXYIXZZXIIII",
        "IIIIIYXIXZZYIIII",
        "IIIIIXXIXZZYIIII",
        "IIIIIYYIXZZYIIII",
        "IIIIIXYIXZZYIIII"
    ],
    [
        "IIIIIYXIYZZZZXII",
        "IIIIIXXIYZZZZXII",
        "IIIIIYYIYZZZZXII",
        "IIIIIXYIYZZZZXII",
        "IIIIIYXIYZZZZYII",
        "IIIIIXXIYZZZZYII",
        "IIIIIYYIYZZZZYII",
        "IIIIIXYIYZZZZYII",
        "IIIIIYXIXZZZZXII",
        "IIIIIXXIXZZZZXII",
        "IIIIIYYIXZZZZXII",
        "IIIIIXYIXZZZZXII",
        "IIIIIYXIXZZZZYII",
        "IIIIIXXIXZZZZYII",
        "IIIIIYYIXZZZZYII",
        "IIIIIXYIXZZZZYII"
    ],
    [
        "IIIIIYXIYZZZZZZX",
        "IIIIIXXIYZZZZZZX",
        "IIIIIYYIYZZZZZZX",
        "IIIIIXYIYZZZZZZX",
        "IIIIIYXIYZZZZZZY",
        "IIIIIXXIYZZZZZZY",
        "IIIIIYYIYZZZZZZY",
        "IIIIIXYIYZZZZZZY",
        "IIIIIYXIXZZZZZZX",
        "IIIIIXXIXZZZZZZX",
        "IIIIIYYIXZZZZZZX",
        "IIIIIXYIXZZZZZZX",
        "IIIIIYXIXZZZZZZY",
        "IIIIIXXIXZZZZZZY",
        "IIIIIYYIXZZZZZZY",
        "IIIIIXYIXZZZZZZY"
    ],
    [
        "IIIIIYXIIYXIIIII",
        "IIIIIXXIIYXIIIII",
        "IIIIIYYIIYXIIIII",
        "IIIIIXYIIYXIIIII",
        "IIIIIYXIIYYIIIII",
        "IIIIIXXIIYYIIIII",
        "IIIIIYYIIYYIIIII",
        "IIIIIXYIIYYIIIII",
        "IIIIIYXIIXXIIIII",
        "IIIIIXXIIXXIIIII",
        "IIIIIYYIIXXIIIII",
        "IIIIIXYIIXXIIIII",
        "IIIIIYXIIXYIIIII",
        "IIIIIXXIIXYIIIII",
        "IIIIIYYIIXYIIIII",
        "IIIIIXYIIXYIIIII"
    ],
    [
        "IIIIIYXIIYZZXIII",
        "IIIIIXXIIYZZXIII",
        "IIIIIYYIIYZZXIII",
        "IIIIIXYIIYZZXIII",
        "IIIIIYXIIYZZYIII",
        "IIIIIXXIIYZZYIII",
        "IIIIIYYIIYZZYIII",
        "IIIIIXYIIYZZYIII",
        "IIIIIYXIIXZZXIII",
        "IIIIIXXIIXZZXIII",
        "IIIIIYYIIXZZXIII",
        "IIIIIXYIIXZZXIII",
        "IIIIIYXIIXZZYIII",
        "IIIIIXXIIXZZYIII",
        "IIIIIYYIIXZZYIII",
        "IIIIIXYIIXZZYIII"
    ],
    [
        "IIIIIYXIIYZZZZXI",
        "IIIIIXXIIYZZZZXI",
        "IIIIIYYIIYZZZZXI",
        "IIIIIXYIIYZZZZXI",
        "IIIIIYXIIYZZZZYI",
        "IIIIIXXIIYZZZZYI",
        "IIIIIYYIIYZZZZYI",
        "IIIIIXYIIYZZZZYI",
        "IIIIIYXIIXZZZZXI",
        "IIIIIXXIIXZZZZXI",
        "IIIIIYYIIXZZZZXI",
        "IIIIIXYIIXZZZZXI",
        "IIIIIYXIIXZZZZYI",
        "IIIIIXXIIXZZZZYI",
        "IIIIIYYIIXZZZZYI",
        "IIIIIXYIIXZZZZYI"
    ],
    [
        "IIIIIYXIIIYXIIII",
        "IIIIIXXIIIYXIIII",
        "IIIIIYYIIIYXIIII",
        "IIIIIXYIIIYXIIII",
        "IIIIIYXIIIYYIIII",
        "IIIIIXXIIIYYIIII",
        "IIIIIYYIIIYYIIII",
        "IIIIIXYIIIYYIIII",
        "IIIIIYXIIIXXIIII",
        "IIIIIXXIIIXXIIII",
        "IIIIIYYIIIXXIIII",
        "IIIIIXYIIIXXIIII",
        "IIIIIYXIIIXYIIII",
        "IIIIIXXIIIXYIIII",
        "IIIIIYYIIIXYIIII",
        "IIIIIXYIIIXYIIII"
    ],
    [
        "IIIIIYXIIIYZZXII",
        "IIIIIXXIIIYZZXII",
        "IIIIIYYIIIYZZXII",
        "IIIIIXYIIIYZZXII",
        "IIIIIYXIIIYZZYII",
        "IIIIIXXIIIYZZYII",
        "IIIIIYYIIIYZZYII",
        "IIIIIXYIIIYZZYII",
        "IIIIIYXIIIXZZXII",
        "IIIIIXXIIIXZZXII",
        "IIIIIYYIIIXZZXII",
        "IIIIIXYIIIXZZXII",
        "IIIIIYXIIIXZZYII",
        "IIIIIXXIIIXZZYII",
        "IIIIIYYIIIXZZYII",
        "IIIIIXYIIIXZZYII"
    ],
    [
        "IIIIIYXIIIYZZZZX",
        "IIIIIXXIIIYZZZZX",
        "IIIIIYYIIIYZZZZX",
        "IIIIIXYIIIYZZZZX",
        "IIIIIYXIIIYZZZZY",
        "IIIIIXXIIIYZZZZY",
        "IIIIIYYIIIYZZZZY",
        "IIIIIXYIIIYZZZZY",
        "IIIIIYXIIIXZZZZX",
        "IIIIIXXIIIXZZZZX",
        "IIIIIYYIIIXZZZZX",
        "IIIIIXYIIIXZZZZX",
        "IIIIIYXIIIXZZZZY",
        "IIIIIXXIIIXZZZZY",
        "IIIIIYYIIIXZZZZY",
        "IIIIIXYIIIXZZZZY"
    ],
    [
        "IIIIIYXIIIIYXIII",
        "IIIIIXXIIIIYXIII",
        "IIIIIYYIIIIYXIII",
        "IIIIIXYIIIIYXIII",
        "IIIIIYXIIIIYYIII",
        "IIIIIXXIIIIYYIII",
        "IIIIIYYIIIIYYIII",
        "IIIIIXYIIIIYYIII",
        "IIIIIYXIIIIXXIII",
        "IIIIIXXIIIIXXIII",
        "IIIIIYYIIIIXXIII",
        "IIIIIXYIIIIXXIII",
        "IIIIIYXIIIIXYIII",
        "IIIIIXXIIIIXYIII",
        "IIIIIYYIIIIXYIII",
        "IIIIIXYIIIIXYIII"
    ],
    [
        "IIIIIYXIIIIYZZXI",
        "IIIIIXXIIIIYZZXI",
        "IIIIIYYIIIIYZZXI",
        "IIIIIXYIIIIYZZXI",
        "IIIIIYXIIIIYZZYI",
        "IIIIIXXIIIIYZZYI",
        "IIIIIYYIIIIYZZYI",
        "IIIIIXYIIIIYZZYI",
        "IIIIIYXIIIIXZZXI",
        "IIIIIXXIIIIXZZXI",
        "IIIIIYYIIIIXZZXI",
        "IIIIIXYIIIIXZZXI",
        "IIIIIYXIIIIXZZYI",
        "IIIIIXXIIIIXZZYI",
        "IIIIIYYIIIIXZZYI",
        "IIIIIXYIIIIXZZYI"
    ],
    [
        "IIIIIYXIIIIIYXII",
        "IIIIIXXIIIIIYXII",
        "IIIIIYYIIIIIYXII",
        "IIIIIXYIIIIIYXII",
        "IIIIIYXIIIIIYYII",
        "IIIIIXXIIIIIYYII",
        "IIIIIYYIIIIIYYII",
        "IIIIIXYIIIIIYYII",
        "IIIIIYXIIIIIXXII",
        "IIIIIXXIIIIIXXII",
        "IIIIIYYIIIIIXXII",
        "IIIIIXYIIIIIXXII",
        "IIIIIYXIIIIIXYII",
        "IIIIIXXIIIIIXYII",
        "IIIIIYYIIIIIXYII",
        "IIIIIXYIIIIIXYII"
    ],
    [
        "IIIIIYXIIIIIYZZX",
        "IIIIIXXIIIIIYZZX",
        "IIIIIYYIIIIIYZZX",
        "IIIIIXYIIIIIYZZX",
        "IIIIIYXIIIIIYZZY",
        "IIIIIXXIIIIIYZZY",
        "IIIIIYYIIIIIYZZY",
        "IIIIIXYIIIIIYZZY",
        "IIIIIYXIIIIIXZZX",
        "IIIIIXXIIIIIXZZX",
        "IIIIIYYIIIIIXZZX",
        "IIIIIXYIIIIIXZZX",
        "IIIIIYXIIIIIXZZY",
        "IIIIIXXIIIIIXZZY",
        "IIIIIYYIIIIIXZZY",
        "IIIIIXYIIIIIXZZY"
    ],
    [
        "IIIIIYXIIIIIIYXI",
        "IIIIIXXIIIIIIYXI",
        "IIIIIYYIIIIIIYXI",
        "IIIIIXYIIIIIIYXI",
        "IIIIIYXIIIIIIYYI",
        "IIIIIXXIIIIIIYYI",
        "IIIIIYYIIIIIIYYI",
        "IIIIIXYIIIIIIYYI",
        "IIIIIYXIIIIIIXXI",
        "IIIIIXXIIIIIIXXI",
        "IIIIIYYIIIIIIXXI",
        "IIIIIXYIIIIIIXXI",
        "IIIIIYXIIIIIIXYI",
        "IIIIIXXIIIIIIXYI",
        "IIIIIYYIIIIIIXYI",
        "IIIIIXYIIIIIIXYI"
    ],
    [
        "IIIIIYXIIIIIIIYX",
        "IIIIIXXIIIIIIIYX",
        "IIIIIYYIIIIIIIYX",
        "IIIIIXYIIIIIIIYX",
        "IIIIIYXIIIIIIIYY",
        "IIIIIXXIIIIIIIYY",
        "IIIIIYYIIIIIIIYY",
        "IIIIIXYIIIIIIIYY",
        "IIIIIYXIIIIIIIXX",
        "IIIIIXXIIIIIIIXX",
        "IIIIIYYIIIIIIIXX",
        "IIIIIXYIIIIIIIXX",
        "IIIIIYXIIIIIIIXY",
        "IIIIIXXIIIIIIIXY",
        "IIIIIYYIIIIIIIXY",
        "IIIIIXYIIIIIIIXY"
    ],
    [
        "IIIIIYZXIYZXIIII",
        "IIIIIXZXIYZXIIII",
        "IIIIIYZYIYZXIIII",
        "IIIIIXZYIYZXIIII",
        "IIIIIYZXIYZYIIII",
        "IIIIIXZXIYZYIIII",
        "IIIIIYZYIYZYIIII",
        "IIIIIXZYIYZYIIII",
        "IIIIIYZXIXZXIIII",
        "IIIIIXZXIXZXIIII",
        "IIIIIYZYIXZXIIII",
        "IIIIIXZYIXZXIIII",
        "IIIIIYZXIXZYIIII",
        "IIIIIXZXIXZYIIII",
        "IIIIIYZYIXZYIIII",
        "IIIIIXZYIXZYIIII"
    ],
    [
        "IIIIIYZXIYZZZXII",
        "IIIIIXZXIYZZZXII",
        "IIIIIYZYIYZZZXII",
        "IIIIIXZYIYZZZXII",
        "IIIIIYZXIYZZZYII",
        "IIIIIXZXIYZZZYII",
        "IIIIIYZYIYZZZYII",
        "IIIIIXZYIYZZZYII",
        "IIIIIYZXIXZZZXII",
        "IIIIIXZXIXZZZXII",
        "IIIIIYZYIXZZZXII",
        "IIIIIXZYIXZZZXII",
        "IIIIIYZXIXZZZYII",
        "IIIIIXZXIXZZZYII",
        "IIIIIYZYIXZZZYII",
        "IIIIIXZYIXZZZYII"
    ],
    [
        "IIIIIYZXIYZZZZZX",
        "IIIIIXZXIYZZZZZX",
        "IIIIIYZYIYZZZZZX",
        "IIIIIXZYIYZZZZZX",
        "IIIIIYZXIYZZZZZY",
        "IIIIIXZXIYZZZZZY",
        "IIIIIYZYIYZZZZZY",
        "IIIIIXZYIYZZZZZY",
        "IIIIIYZXIXZZZZZX",
        "IIIIIXZXIXZZZZZX",
        "IIIIIYZYIXZZZZZX",
        "IIIIIXZYIXZZZZZX",
        "IIIIIYZXIXZZZZZY",
        "IIIIIXZXIXZZZZZY",
        "IIIIIYZYIXZZZZZY",
        "IIIIIXZYIXZZZZZY"
    ],
    [
        "IIIIIYZXIIIYZXII",
        "IIIIIXZXIIIYZXII",
        "IIIIIYZYIIIYZXII",
        "IIIIIXZYIIIYZXII",
        "IIIIIYZXIIIYZYII",
        "IIIIIXZXIIIYZYII",
        "IIIIIYZYIIIYZYII",
        "IIIIIXZYIIIYZYII",
        "IIIIIYZXIIIXZXII",
        "IIIIIXZXIIIXZXII",
        "IIIIIYZYIIIXZXII",
        "IIIIIXZYIIIXZXII",
        "IIIIIYZXIIIXZYII",
        "IIIIIXZXIIIXZYII",
        "IIIIIYZYIIIXZYII",
        "IIIIIXZYIIIXZYII"
    ],
    [
        "IIIIIYZXIIIYZZZX",
        "IIIIIXZXIIIYZZZX",
        "IIIIIYZYIIIYZZZX",
        "IIIIIXZYIIIYZZZX",
        "IIIIIYZXIIIYZZZY",
        "IIIIIXZXIIIYZZZY",
        "IIIIIYZYIIIYZZZY",
        "IIIIIXZYIIIYZZZY",
        "IIIIIYZXIIIXZZZX",
        "IIIIIXZXIIIXZZZX",
        "IIIIIYZYIIIXZZZX",
        "IIIIIXZYIIIXZZZX",
        "IIIIIYZXIIIXZZZY",
        "IIIIIXZXIIIXZZZY",
        "IIIIIYZYIIIXZZZY",
        "IIIIIXZYIIIXZZZY"
    ],
    [
        "IIIIIYZXIIIIIYZX",
        "IIIIIXZXIIIIIYZX",
        "IIIIIYZYIIIIIYZX",
        "IIIIIXZYIIIIIYZX",
        "IIIIIYZXIIIIIYZY",
        "IIIIIXZXIIIIIYZY",
        "IIIIIYZYIIIIIYZY",
        "IIIIIXZYIIIIIYZY",
        "IIIIIYZXIIIIIXZX",
        "IIIIIXZXIIIIIXZX",
        "IIIIIYZYIIIIIXZX",
        "IIIIIXZYIIIIIXZX",
        "IIIIIYZXIIIIIXZY",
        "IIIIIXZXIIIIIXZY",
        "IIIIIYZYIIIIIXZY",
        "IIIIIXZYIIIIIXZY"
    ],
    [
        "IIIIIIYXYXIIIIII",
        "IIIIIIXXYXIIIIII",
        "IIIIIIYYYXIIIIII",
        "IIIIIIXYYXIIIIII",
        "IIIIIIYXYYIIIIII",
        "IIIIIIXXYYIIIIII",
        "IIIIIIYYYYIIIIII",
        "IIIIIIXYYYIIIIII",
        "IIIIIIYXXXIIIIII",
        "IIIIIIXXXXIIIIII",
        "IIIIIIYYXXIIIIII",
        "IIIIIIXYXXIIIIII",
        "IIIIIIYXXYIIIIII",
        "IIIIIIXXXYIIIIII",
        "IIIIIIYYXYIIIIII",
        "IIIIIIXYXYIIIIII"
    ],
    [
        "IIIIIIYXYZZXIIII",
        "IIIIIIXXYZZXIIII",
        "IIIIIIYYYZZXIIII",
        "IIIIIIXYYZZXIIII",
        "IIIIIIYXYZZYIIII",
        "IIIIIIXXYZZYIIII",
        "IIIIIIYYYZZYIIII",
        "IIIIIIXYYZZYIIII",
        "IIIIIIYXXZZXIIII",
        "IIIIIIXXXZZXIIII",
        "IIIIIIYYXZZXIIII",
        "IIIIIIXYXZZXIIII",
        "IIIIIIYXXZZYIIII",
        "IIIIIIXXXZZYIIII",
        "IIIIIIYYXZZYIIII",
        "IIIIIIXYXZZYIIII"
    ],
    [
        "IIIIIIYXYZZZZXII",
        "IIIIIIXXYZZZZXII",
        "IIIIIIYYYZZZZXII",
        "IIIIIIXYYZZZZXII",
        "IIIIIIYXYZZZZYII",
        "IIIIIIXXYZZZZYII",
        "IIIIIIYYYZZZZYII",
        "IIIIIIXYYZZZZYII",
        "IIIIIIYXXZZZZXII",
        "IIIIIIXXXZZZZXII",
        "IIIIIIYYXZZZZXII",
        "IIIIIIXYXZZZZXII",
        "IIIIIIYXXZZZZYII",
        "IIIIIIXXXZZZZYII",
        "IIIIIIYYXZZZZYII",
        "IIIIIIXYXZZZZYII"
    ],
    [
        "IIIIIIYXYZZZZZZX",
        "IIIIIIXXYZZZZZZX",
        "IIIIIIYYYZZZZZZX",
        "IIIIIIXYYZZZZZZX",
        "IIIIIIYXYZZZZZZY",
        "IIIIIIXXYZZZZZZY",
        "IIIIIIYYYZZZZZZY",
        "IIIIIIXYYZZZZZZY",
        "IIIIIIYXXZZZZZZX",
        "IIIIIIXXXZZZZZZX",
        "IIIIIIYYXZZZZZZX",
        "IIIIIIXYXZZZZZZX",
        "IIIIIIYXXZZZZZZY",
        "IIIIIIXXXZZZZZZY",
        "IIIIIIYYXZZZZZZY",
        "IIIIIIXYXZZZZZZY"
    ],
    [
        "IIIIIIYXIYXIIIII",
        "IIIIIIXXIYXIIIII",
        "IIIIIIYYIYXIIIII",
        "IIIIIIXYIYXIIIII",
        "IIIIIIYXIYYIIIII",
        "IIIIIIXXIYYIIIII",
        "IIIIIIYYIYYIIIII",
        "IIIIIIXYIYYIIIII",
        "IIIIIIYXIXXIIIII",
        "IIIIIIXXIXXIIIII",
        "IIIIIIYYIXXIIIII",
        "IIIIIIXYIXXIIIII",
        "IIIIIIYXIXYIIIII",
        "IIIIIIXXIXYIIIII",
        "IIIIIIYYIXYIIIII",
        "IIIIIIXYIXYIIIII"
    ],
    [
        "IIIIIIYXIYZZXIII",
        "IIIIIIXXIYZZXIII",
        "IIIIIIYYIYZZXIII",
        "IIIIIIXYIYZZXIII",
        "IIIIIIYXIYZZYIII",
        "IIIIIIXXIYZZYIII",
        "IIIIIIYYIYZZYIII",
        "IIIIIIXYIYZZYIII",
        "IIIIIIYXIXZZXIII",
        "IIIIIIXXIXZZXIII",
        "IIIIIIYYIXZZXIII",
        "IIIIIIXYIXZZXIII",
        "IIIIIIYXIXZZYIII",
        "IIIIIIXXIXZZYIII",
        "IIIIIIYYIXZZYIII",
        "IIIIIIXYIXZZYIII"
    ],
    [
        "IIIIIIYXIYZZZZXI",
        "IIIIIIXXIYZZZZXI",
        "IIIIIIYYIYZZZZXI",
        "IIIIIIXYIYZZZZXI",
        "IIIIIIYXIYZZZZYI",
        "IIIIIIXXIYZZZZYI",
        "IIIIIIYYIYZZZZYI",
        "IIIIIIXYIYZZZZYI",
        "IIIIIIYXIXZZZZXI",
        "IIIIIIXXIXZZZZXI",
        "IIIIIIYYIXZZZZXI",
        "IIIIIIXYIXZZZZXI",
        "IIIIIIYXIXZZZZYI",
        "IIIIIIXXIXZZZZYI",
        "IIIIIIYYIXZZZZYI",
        "IIIIIIXYIXZZZZYI"
    ],
    [
        "IIIIIIYXIIYXIIII",
        "IIIIIIXXIIYXIIII",
        "IIIIIIYYIIYXIIII",
        "IIIIIIXYIIYXIIII",
        "IIIIIIYXIIYYIIII",
        "IIIIIIXXIIYYIIII",
        "IIIIIIYYIIYYIIII",
        "IIIIIIXYIIYYIIII",
        "IIIIIIYXIIXXIIII",
        "IIIIIIXXIIXXIIII",
        "IIIIIIYYIIXXIIII",
        "IIIIIIXYIIXXIIII",
        "IIIIIIYXIIXYIIII",
        "IIIIIIXXIIXYIIII",
        "IIIIIIYYIIXYIIII",
        "IIIIIIXYIIXYIIII"
    ],
    [
        "IIIIIIYXIIYZZXII",
        "IIIIIIXXIIYZZXII",
        "IIIIIIYYIIYZZXII",
        "IIIIIIXYIIYZZXII",
        "IIIIIIYXIIYZZYII",
        "IIIIIIXXIIYZZYII",
        "IIIIIIYYIIYZZYII",
        "IIIIIIXYIIYZZYII",
        "IIIIIIYXIIXZZXII",
        "IIIIIIXXIIXZZXII",
        "IIIIIIYYIIXZZXII",
        "IIIIIIXYIIXZZXII",
        "IIIIIIYXIIXZZYII",
        "IIIIIIXXIIXZZYII",
        "IIIIIIYYIIXZZYII",
        "IIIIIIXYIIXZZYII"
    ],
    [
        "IIIIIIYXIIYZZZZX",
        "IIIIIIXXIIYZZZZX",
        "IIIIIIYYIIYZZZZX",
        "IIIIIIXYIIYZZZZX",
        "IIIIIIYXIIYZZZZY",
        "IIIIIIXXIIYZZZZY",
        "IIIIIIYYIIYZZZZY",
        "IIIIIIXYIIYZZZZY",
        "IIIIIIYXIIXZZZZX",
        "IIIIIIXXIIXZZZZX",
        "IIIIIIYYIIXZZZZX",
        "IIIIIIXYIIXZZZZX",
        "IIIIIIYXIIXZZZZY",
        "IIIIIIXXIIXZZZZY",
        "IIIIIIYYIIXZZZZY",
        "IIIIIIXYIIXZZZZY"
    ],
    [
        "IIIIIIYXIIIYXIII",
        "IIIIIIXXIIIYXIII",
        "IIIIIIYYIIIYXIII",
        "IIIIIIXYIIIYXIII",
        "IIIIIIYXIIIYYIII",
        "IIIIIIXXIIIYYIII",
        "IIIIIIYYIIIYYIII",
        "IIIIIIXYIIIYYIII",
        "IIIIIIYXIIIXXIII",
        "IIIIIIXXIIIXXIII",
        "IIIIIIYYIIIXXIII",
        "IIIIIIXYIIIXXIII",
        "IIIIIIYXIIIXYIII",
        "IIIIIIXXIIIXYIII",
        "IIIIIIYYIIIXYIII",
        "IIIIIIXYIIIXYIII"
    ],
    [
        "IIIIIIYXIIIYZZXI",
        "IIIIIIXXIIIYZZXI",
        "IIIIIIYYIIIYZZXI",
        "IIIIIIXYIIIYZZXI",
        "IIIIIIYXIIIYZZYI",
        "IIIIIIXXIIIYZZYI",
        "IIIIIIYYIIIYZZYI",
        "IIIIIIXYIIIYZZYI",
        "IIIIIIYXIIIXZZXI",
        "IIIIIIXXIIIXZZXI",
        "IIIIIIYYIIIXZZXI",
        "IIIIIIXYIIIXZZXI",
        "IIIIIIYXIIIXZZYI",
        "IIIIIIXXIIIXZZYI",
        "IIIIIIYYIIIXZZYI",
        "IIIIIIXYIIIXZZYI"
    ],
    [
        "IIIIIIYXIIIIYXII",
        "IIIIIIXXIIIIYXII",
        "IIIIIIYYIIIIYXII",
        "IIIIIIXYIIIIYXII",
        "IIIIIIYXIIIIYYII",
        "IIIIIIXXIIIIYYII",
        "IIIIIIYYIIIIYYII",
        "IIIIIIXYIIIIYYII",
        "IIIIIIYXIIIIXXII",
        "IIIIIIXXIIIIXXII",
        "IIIIIIYYIIIIXXII",
        "IIIIIIXYIIIIXXII",
        "IIIIIIYXIIIIXYII",
        "IIIIIIXXIIIIXYII",
        "IIIIIIYYIIIIXYII",
        "IIIIIIXYIIIIXYII"
    ],
    [
        "IIIIIIYXIIIIYZZX",
        "IIIIIIXXIIIIYZZX",
        "IIIIIIYYIIIIYZZX",
        "IIIIIIXYIIIIYZZX",
        "IIIIIIYXIIIIYZZY",
        "IIIIIIXXIIIIYZZY",
        "IIIIIIYYIIIIYZZY",
        "IIIIIIXYIIIIYZZY",
        "IIIIIIYXIIIIXZZX",
        "IIIIIIXXIIIIXZZX",
        "IIIIIIYYIIIIXZZX",
        "IIIIIIXYIIIIXZZX",
        "IIIIIIYXIIIIXZZY",
        "IIIIIIXXIIIIXZZY",
        "IIIIIIYYIIIIXZZY",
        "IIIIIIXYIIIIXZZY"
    ],
    [
        "IIIIIIYXIIIIIYXI",
        "IIIIIIXXIIIIIYXI",
        "IIIIIIYYIIIIIYXI",
        "IIIIIIXYIIIIIYXI",
        "IIIIIIYXIIIIIYYI",
        "IIIIIIXXIIIIIYYI",
        "IIIIIIYYIIIIIYYI",
        "IIIIIIXYIIIIIYYI",
        "IIIIIIYXIIIIIXXI",
        "IIIIIIXXIIIIIXXI",
        "IIIIIIYYIIIIIXXI",
        "IIIIIIXYIIIIIXXI",
        "IIIIIIYXIIIIIXYI",
        "IIIIIIXXIIIIIXYI",
        "IIIIIIYYIIIIIXYI",
        "IIIIIIXYIIIIIXYI"
    ],
    [
        "IIIIIIYXIIIIIIYX",
        "IIIIIIXXIIIIIIYX",
        "IIIIIIYYIIIIIIYX",
        "IIIIIIXYIIIIIIYX",
        "IIIIIIYXIIIIIIYY",
        "IIIIIIXXIIIIIIYY",
        "IIIIIIYYIIIIIIYY",
        "IIIIIIXYIIIIIIYY",
        "IIIIIIYXIIIIIIXX",
        "IIIIIIXXIIIIIIXX",
        "IIIIIIYYIIIIIIXX",
        "IIIIIIXYIIIIIIXX",
        "IIIIIIYXIIIIIIXY",
        "IIIIIIXXIIIIIIXY",
        "IIIIIIYYIIIIIIXY",
        "IIIIIIXYIIIIIIXY"
    ]
]

In [73]:
path = []

for pauli_word in ucc_8_16:
    path += search_prune(Cirq_Tableau(pauli_word))
    
op_count(path)

{'CNOT': 5456, 'Single_q': 22831}